# 任务4：血管核心／转码边带／背景三分类，单组五折

**上传这一个Notebook，挂载原比赛数据、启用GPU和Internet，Run All即可。** 不需要旧训练权重或旧OOF输入；已有L1参考指标和来源记录已内嵌。只训练新方法一组5折80轮；测试推理和唯一候选CSV也在本次Run All中完成，后续不需另加推理单元。

## 方法

固定C为干净mask、P为官方转码语义的代理mask。本数据20图均有C⊆P。将像素分成三类：背景=非P、核心=C、边带=P−C。边带是标签构造差异，不等于新增的真实解剖血管。

保留ImageNet ResNet34 U-Net和全部解码器，仅把主头及两个辅头从1通道改成3通道softmax：

- `p_core = p(class=core)`
- `p_proxy = p(class=core) + p(class=band)`

天然保持代理概率≥核心概率。每个尺度固定损失为`0.5 CE + 0.25 Dice(core,C) + 0.25 Dice(proxy,P)`，各辅头仍采用原权重0.1/0.2，再除以1.3。辅头GT采用面积缩放后的三类概率分布，保持和为1及嵌套关系。不是把C/P混在同一个软标签中，也不加clDice、风格增强或大patch。

这是本任务的分解建模假设，尚无正式成绩；[DCAN的区域与轮廓联合监督](https://arxiv.org/abs/1604.02677)提供相关思路，不能据此保证这个三分类变体有效。方法对照同时改变了输出表示和对应损失，不声称只隔离某一个损失项的因果贡献。

## 控制与评价

RGB/标签均512、patch256、batch8、每轮256patch、原五折、seed42、80轮、最终EMA保持不变。正样本中心仍来自C。替换分类头之前的完整二分类模型初始化哈希、每轮RGB采样/增强计划必须与最新已完成L1一致；不重训参考模型，不用外层验证挑最佳轮次。

主比较是**新proxy输出 vs 旧L1输出**：先列D4固定0.5，分别评价C/P；再给预设13点网格的阈值敏感性。core输出单独报告，不能把core对C与proxy对P的高分拼成一个预测的成绩。

唯一测试CSV固定使用proxy输出、D4、五折概率平均。阈值按运行前固定规则在已有13点网格上最大化20图代理OOF均值，同分时更接近0.5者优先。它是开发集调参，不是嵌套或盲测，不根据公开榜选阈值；始终仅导出一个候选。生成CSV不意味着优于现有0.84276方案，先看OOF、最差折和精确率/召回，不自动提交比赛。

## 保存

每轮保留last.pt；每折保存最终EMA、OOF与测试概率，完成折校验后复用。训练量是一组完整五折，按既往T4训练/监控约半小时量级，推理和保存另计；实际以日志为准。

**跑完只需下载`submission_core_band_proxy.csv`与`core_band_analysis.zip`。不用下载权重。** 权重留在当前Kaggle会话或显式保存的版本Output；OOF小包另存`core_band_oof.zip`，可选下载。若要在新会话续训，挂载本轮完整Output目录，填写RESUME_FROM并使用新的空输出目录。分析ZIP不含权重，不能续训。

用Quick Save保留云端Output时，需在Advanced Settings显式选择保存输出，再确认版本Output确实包含文件。不要仅凭Draft saved判断已保存，也不必为了保存已经完成的文件再次Run All。


## 1. 配置


In [ ]:
from pathlib import Path
DATA_DIR = None  # 原任务4比赛数据；不需要上传旧权重或旧分析包
OUTPUT_DIR = Path('/kaggle/working/task4_core_band') if Path('/kaggle/working').exists() else Path.cwd()/'task4_core_band_output'
RESUME_FROM = None # 中断续接：本轮完整Kaggle Output目录，不能用小分析ZIP
WEIGHTS_PATH = None # 默认复用工作目录官方ImageNet缓存，否则在Kaggle下载
FOLDS_TO_RUN = [0,1,2,3,4]
RUN_EXPERIMENT = True
INSTALL_MISSING_DEPS = True


## 2. 加载内嵌实现与参考


In [ ]:
import importlib.util,sys,subprocess
if importlib.util.find_spec('skimage') is None:
    if not INSTALL_MISSING_DEPS:raise RuntimeError('Install scikit-image or enable INSTALL_MISSING_DEPS.')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','scikit-image'])
RUNTIME_DIR=(Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd())/'task4_core_band_runtime'
RUNTIME_DIR.mkdir(parents=True,exist_ok=True)
SOURCES={'task4_pipeline.py': '"""Task 4: small-data retinal vessel segmentation, audit through submission.\n\nOnly the supplied training images/labels are used. No downloaded weights.\nThe original segmentation_to_csv.py is kept unchanged and checked on export.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageDraw\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\n\n@dataclass\nclass Config:\n    seed: int = 42\n    epochs: int = 80\n    size: int = 512\n    patch: int = 256\n    patches_per_epoch: int = 128\n    batch_size: int = 8\n    base_channels: int = 24\n    lr: float = 0.001\n    weight_decay: float = 0.0001\n    folds: int = 5\n    fold: int = 0\n    threads: int = 4\n    amp: bool = True\n\n\ndef save_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    os.replace(tmp, path)\n\n\ndef file_sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef find_data(data_dir=None):\n    if data_dir:\n        candidates = [Path(data_dir)]\n    else:\n        base = Path(__file__).resolve().parent if \'__file__\' in globals() else Path.cwd()\n        candidates = [base / \'2026task-4/segmentation\', Path.cwd() / \'2026task-4/segmentation\',\n                      Path.cwd() / \'task4/2026task-4/segmentation\']\n        if Path(\'/kaggle/input\').exists():\n            candidates += [p.parent for p in Path(\'/kaggle/input\').rglob(\'sample_submission.csv\')]\n    found = {p.resolve() for p in candidates if (p / \'train/label\').is_dir()\n             and (p / \'test/image\').is_dir() and (p / \'sample_submission.csv\').is_file()}\n    if len(found) != 1:\n        raise ValueError(f\'Set data_dir to the segmentation folder. Found: {sorted(map(str, found))}\')\n    return found.pop()\n\n\ndef list_images(directory):\n    files = sorted(Path(directory).glob(\'*.jpg\'), key=lambda p: int(p.stem))\n    if not files:\n        raise ValueError(f\'No jpg images: {directory}\')\n    return files\n\n\ndef strict_label(path, size=512):\n    # JPEG labels contain ringing; >0 would thicken the foreground substantially.\n    mask = (np.asarray(Image.open(path).convert(\'L\')) > 127).astype(np.uint8)\n    return cv2.resize(mask, (size, size), interpolation=cv2.INTER_NEAREST).astype(bool)\n\n\ndef converter_label(path):\n    # Exact target IF hidden ground truth was processed by the supplied converter.\n    # This is a diagnostic proxy, not an assertion about the private scoring code.\n    return np.asarray(Image.open(path).convert(\'L\').resize((512, 512), Image.Resampling.BILINEAR)) > 0\n\n\ndef make_features(path, size=512):\n    rgb = np.asarray(Image.open(path).convert(\'RGB\'))\n    rgb = cv2.resize(rgb, (size, size), interpolation=cv2.INTER_AREA)\n    green = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(rgb[:, :, 1])\n    return np.concatenate([rgb, green[:, :, None]], axis=2)\n\n\ndef audit(root, output, cfg):\n    output.mkdir(parents=True, exist_ok=True)\n    train, test = list_images(root / \'train/image\'), list_images(root / \'test/image\')\n    labels = {p.stem: p for p in list_images(root / \'train/label\')}\n    if len(train) != 20 or len(test) != 20 or set(labels) != {p.stem for p in train}:\n        raise ValueError(\'Expected 20 train image/label pairs and 20 test images.\')\n    with (root / \'sample_submission.csv\').open(newline=\'\', encoding=\'utf-8-sig\') as f:\n        reader = csv.DictReader(f)\n        if reader.fieldnames != [\'Id\', \'Predicted\']:\n            raise ValueError(f\'Unexpected template columns: {reader.fieldnames}\')\n        ids = [r[\'Id\'] for r in reader]\n    if len(ids) != len(set(ids)) or set(ids) != {p.stem for p in test}:\n        raise ValueError(\'Template IDs do not match test images.\')\n    rows, groups, test_hashes = [], {}, set()\n    for split, paths in [(\'train\', train), (\'test\', test)]:\n        for p in paths:\n            im = Image.open(p).convert(\'RGB\')\n            digest = hashlib.sha256(np.asarray(im).tobytes()).hexdigest()\n            row = dict(split=split, id=p.stem, width=im.width, height=im.height,\n                       image_sha256=file_sha(p), pixel_sha256=digest)\n            if split == \'train\':\n                if Image.open(labels[p.stem]).size != im.size:\n                    raise ValueError(f\'Image/label size mismatch: {p.stem}\')\n                row.update(label_sha256=file_sha(labels[p.stem]),\n                           vessel_fraction=float(strict_label(labels[p.stem]).mean()),\n                           converter_fraction=float(converter_label(labels[p.stem]).mean()))\n                groups.setdefault(digest, []).append(p.stem)\n            else:\n                test_hashes.add(digest)\n            rows.append(row)\n    if set(groups) & test_hashes:\n        raise ValueError(\'Exact decoded train/test duplicate detected; inspect before training.\')\n    ordered = sorted(groups)\n    random.Random(cfg.seed).shuffle(ordered)\n    assignments = {}\n    sizes = [0] * cfg.folds\n    for digest in ordered:\n        fold = min(range(cfg.folds), key=lambda k: sizes[k])\n        for image_id in groups[digest]:\n            assignments[image_id] = fold\n        sizes[fold] += len(groups[digest])\n    for row in rows:\n        row[\'fold\'] = assignments.get(row[\'id\'], -1) if row[\'split\'] == \'train\' else -1\n    fields = sorted({key for row in rows for key in row})\n    with (output / \'manifest.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fields)\n        writer.writeheader()\n        writer.writerows(rows)\n    manifest_fingerprint = hashlib.sha256(json.dumps(rows, sort_keys=True).encode()).hexdigest()\n    summary = dict(train_images=len(train), test_images=len(test), unique_train_groups=len(groups),\n                   folds=sizes, validation_ids=[p.stem for p in train if assignments[p.stem] == cfg.fold],\n                   decoded_train_test_overlap=0, patient_ids_available=False,\n                   data_fingerprint=manifest_fingerprint,\n                   observed_label_polarity=\'white vessels / black background\',\n                   screenshot_label_polarity=\'black vessels / white background\',\n                   converter=dict(size=[512, 512], resize=\'PIL bilinear\', foreground=\'nonzero\',\n                                  flatten=\'row-major C order\', start_index=1,\n                                  source_sha256=file_sha(root / \'segmentation_to_csv.py\')),\n                   strict_label=\'native grayscale >127, then nearest resize\',\n                   mean_clean_foreground=float(np.mean([r[\'vessel_fraction\'] for r in rows if r[\'split\']==\'train\'])),\n                   mean_converter_foreground=float(np.mean([r[\'converter_fraction\'] for r in rows if r[\'split\']==\'train\'])))\n    save_json(output / \'audit_summary.json\', summary)\n    return train, test, labels, assignments, ids, summary\n\n\ndef paired_d4(image, mask, rotation, flip):\n    image, mask = np.rot90(image, rotation), np.rot90(mask, rotation)\n    if flip:\n        image, mask = np.flip(image, 1), np.flip(mask, 1)\n    return image.copy(), mask.copy()\n\n\nclass PatchDataset(Dataset):\n    def __init__(self, paths, labels, cfg):\n        self.cfg = cfg\n        self.images = [make_features(p, cfg.size) for p in paths]\n        self.masks = [strict_label(labels[p.stem], cfg.size).astype(np.float32) for p in paths]\n        self.positive = [np.argwhere(m > 0) for m in self.masks]\n\n    def __len__(self):\n        return self.cfg.patches_per_epoch\n\n    def __getitem__(self, index):\n        idx = random.randrange(len(self.images))\n        image, mask = self.images[idx], self.masks[idx]\n        size, patch = self.cfg.size, self.cfg.patch\n        crop = random.randint(patch, min(size, int(patch * 1.25)))\n        if random.random() < 0.3 and len(self.positive[idx]):\n            y, x = self.positive[idx][random.randrange(len(self.positive[idx]))]\n            top, left = int(np.clip(y - crop//2, 0, size-crop)), int(np.clip(x-crop//2, 0, size-crop))\n        else:\n            top, left = random.randint(0, size-crop), random.randint(0, size-crop)\n        image = cv2.resize(image[top:top+crop, left:left+crop], (patch, patch), interpolation=cv2.INTER_LINEAR)\n        mask = cv2.resize(mask[top:top+crop, left:left+crop], (patch, patch), interpolation=cv2.INTER_NEAREST)\n        image, mask = paired_d4(image, mask, random.randrange(4), random.random() < 0.5)\n        image = image.astype(np.float32) / 255.0\n        image = np.clip(image * random.uniform(0.9, 1.1) + random.uniform(-0.04, 0.04), 0, 1)\n        image = image ** random.uniform(0.85, 1.15)\n        return torch.from_numpy(image.transpose(2, 0, 1).copy()) * 2 - 1, torch.from_numpy(mask[None].copy())\n\n\nclass ConvBlock(nn.Sequential):\n    def __init__(self, cin, cout):\n        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU(),\n                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU())\n\n\nclass VesselUNet(nn.Module):\n    def __init__(self, base=24):\n        super().__init__()\n        ch = [base * (2**i) for i in range(5)]\n        self.enc = nn.ModuleList([ConvBlock(4 if i == 0 else ch[i-1], ch[i]) for i in range(5)])\n        self.dec = nn.ModuleList([ConvBlock(ch[i+1]+ch[i], ch[i]) for i in range(3, -1, -1)])\n        self.head = nn.Conv2d(base, 1, 1)\n\n    def forward(self, x):\n        skips = []\n        for i, block in enumerate(self.enc):\n            x = block(x if i == 0 else F.max_pool2d(x, 2))\n            skips.append(x)\n        for block, skip in zip(self.dec, reversed(skips[:-1])):\n            x = F.interpolate(x, size=skip.shape[-2:], mode=\'bilinear\', align_corners=False)\n            x = block(torch.cat([x, skip], 1))\n        return self.head(x)\n\n\ndef segmentation_loss(logits, target):\n    bce = F.binary_cross_entropy_with_logits(logits.float(), target.float())\n    prob = logits.float().sigmoid()\n    dice = (2*(prob*target).sum((1,2,3))+1) / (prob.sum((1,2,3))+target.sum((1,2,3))+1)\n    return 0.5*bce + 0.5*(1-dice.mean())\n\n\ndef dice_score(pred, target):\n    pred, target = np.asarray(pred, bool), np.asarray(target, bool)\n    return float((2*np.logical_and(pred, target).sum()+1e-7) / (pred.sum()+target.sum()+1e-7))\n\n\n@torch.inference_mode()\ndef predict(model, path, cfg, device):\n    model.eval()\n    image = make_features(path, cfg.size)\n    step = cfg.patch // 2\n    starts = list(range(0, cfg.size-cfg.patch+1, step))\n    if starts[-1] != cfg.size-cfg.patch:\n        starts.append(cfg.size-cfg.patch)\n    coords = [(y,x) for y in starts for x in starts]\n    weight = np.maximum(np.outer(np.hanning(cfg.patch), np.hanning(cfg.patch)), 0.05).astype(np.float32)\n    sums, weights = np.zeros((cfg.size, cfg.size), np.float32), np.zeros((cfg.size, cfg.size), np.float32)\n    for start in range(0, len(coords), cfg.batch_size):\n        positions = coords[start:start+cfg.batch_size]\n        patches = np.stack([image[y:y+cfg.patch, x:x+cfg.patch].transpose(2,0,1) for y,x in positions])\n        tensor = torch.from_numpy(patches.copy()).to(device).float()/127.5-1\n        with torch.autocast(device_type=device.type, enabled=cfg.amp and device.type==\'cuda\'):\n            probs = model(tensor).float().sigmoid()[:,0].cpu().numpy()\n        for (y,x), prob in zip(positions, probs):\n            sums[y:y+cfg.patch,x:x+cfg.patch] += prob * weight\n            weights[y:y+cfg.patch,x:x+cfg.patch] += weight\n    return sums/weights\n\n\ndef atomic_checkpoint(path, value):\n    tmp = path.with_suffix(\'.tmp\')\n    torch.save(value, tmp)\n    os.replace(tmp, path)\n\n\ndef rng_state():\n    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),\n                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)\n\n\ndef restore_rng(state):\n    random.setstate(state[\'python\'])\n    np.random.set_state(state[\'numpy\'])\n    torch.set_rng_state(state[\'torch\'].cpu())\n    if torch.cuda.is_available() and state[\'cuda\'] is not None:\n        torch.cuda.set_rng_state_all([s.cpu() for s in state[\'cuda\']])\n\n\ndef train_stage(paths, val_paths, labels, cfg, folder, device, epochs):\n    folder.mkdir(parents=True, exist_ok=True)\n    seed_all(cfg.seed)\n    dataset = PatchDataset(paths, labels, cfg)\n    loader = DataLoader(dataset, batch_size=cfg.batch_size, num_workers=0, shuffle=False,\n                        pin_memory=device.type==\'cuda\')\n    model = VesselUNet(cfg.base_channels).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)\n    # Full retraining retains the original 80-epoch cosine horizon.\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=cfg.lr*0.02)\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=cfg.amp and device.type==\'cuda\')\n    best, history, first = -1.0, [], 1\n    last = folder / \'last.pt\'\n    if last.exists():\n        ckpt = torch.load(last, map_location=\'cpu\', weights_only=False)\n        if ckpt[\'train_ids\'] != [p.stem for p in paths] or ckpt[\'target_epochs\'] != epochs:\n            raise ValueError(\'Stage mismatch. Use a new output directory.\')\n        model.load_state_dict(ckpt[\'model\'])\n        optimizer.load_state_dict(ckpt[\'optimizer\'])\n        scheduler.load_state_dict(ckpt[\'scheduler\'])\n        scaler.load_state_dict(ckpt[\'scaler\'])\n        first, best, history = ckpt[\'epoch\']+1, ckpt[\'best\'], ckpt[\'history\']\n        restore_rng(ckpt[\'rng\'])\n        print(f\'{folder.name}: resume at epoch {first}/{epochs}\', flush=True)\n    targets = {p.stem: strict_label(labels[p.stem], cfg.size) for p in val_paths}\n    for epoch in range(first, epochs+1):\n        begin = time.perf_counter()\n        model.train()\n        losses = []\n        lr = optimizer.param_groups[0][\'lr\']\n        for image, mask in loader:\n            image, mask = image.to(device, non_blocking=True), mask.to(device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, enabled=cfg.amp and device.type==\'cuda\'):\n                loss = segmentation_loss(model(image), mask)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Nonfinite training loss\')\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n            scaler.step(optimizer)\n            scaler.update()\n            losses.append(float(loss.detach()))\n        scheduler.step()\n        score = float(np.mean([dice_score(predict(model,p,cfg,device) >= 0.5, targets[p.stem]) for p in val_paths])) if val_paths else None\n        improved = score is not None and score > best\n        if improved:\n            best = score\n            atomic_checkpoint(folder/\'best.pt\', dict(model=model.state_dict(), epoch=epoch, dice_05=score))\n        row = dict(epoch=epoch, loss=float(np.mean(losses)), dice_05=score, lr=lr,\n                   seconds=round(time.perf_counter()-begin, 3))\n        history.append(row)\n        atomic_checkpoint(last, dict(model=model.state_dict(), optimizer=optimizer.state_dict(),\n                          scheduler=scheduler.state_dict(), scaler=scaler.state_dict(), epoch=epoch,\n                          best=best, history=history, rng=rng_state(), train_ids=[p.stem for p in paths], target_epochs=epochs))\n        with (folder/\'history.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n            writer = csv.DictWriter(f, list(row))\n            writer.writeheader()\n            writer.writerows(history)\n        print(f\'{folder.name} {epoch:03d}/{epochs} loss={row["loss"]:.4f}\' +\n              (f\' val_dice@0.5={score:.5f} best={best:.5f}\' if score is not None else \'\') +\n              f\' seconds={row["seconds"]}\', flush=True)\n    selected = torch.load(folder/(\'best.pt\' if val_paths else \'last.pt\'), map_location=\'cpu\', weights_only=False)\n    model.load_state_dict(selected[\'model\'])\n    return model, int(selected[\'epoch\'])\n\n\ndef rle_encode(mask):\n    flat = np.asarray(mask, dtype=np.uint8).ravel(order=\'C\')\n    changes = np.flatnonzero(np.diff(np.r_[np.uint8(0), flat, np.uint8(0)])) + 1\n    changes[1::2] -= changes[::2]\n    return \' \'.join(map(str, changes))\n\n\ndef rle_decode(encoded, shape=(512,512)):\n    values = np.array([int(s) for s in encoded.split()], dtype=np.int64)\n    if len(values)%2:\n        raise ValueError(\'Odd RLE length\')\n    mask = np.zeros(int(np.prod(shape)), dtype=bool)\n    previous_end = 0\n    for start, length in values.reshape(-1,2):\n        if start < 1 or length < 1 or start <= previous_end or start+length-1 > mask.size:\n            raise ValueError(\'RLE bounds/order/overlap violation\')\n        mask[start-1:start-1+length] = True\n        previous_end = start+length-1\n    return mask.reshape(shape)\n\n\ndef load_converter(root):\n    spec = importlib.util.spec_from_file_location(\'task4_official_converter\', root/\'segmentation_to_csv.py\')\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef preview(paths, probs, labels, threshold, output):\n    tile = 256\n    canvas = Image.new(\'RGB\', (tile*3, (tile+24)*min(4,len(paths))), \'white\')\n    draw = ImageDraw.Draw(canvas)\n    for row,p in enumerate(paths[:4]):\n        rgb = Image.open(p).convert(\'RGB\').resize((tile,tile))\n        target = strict_label(labels[p.stem],tile) if labels else None\n        pred = cv2.resize(probs[p.stem],(tile,tile),interpolation=cv2.INTER_LINEAR) >= threshold\n        y = row*(tile+24)\n        canvas.paste(rgb,(0,y+24))\n        draw.text((4,y+4), f\'Image {p.stem}\',fill=\'black\')\n        if target is not None:\n            canvas.paste(Image.fromarray((target*255).astype(np.uint8)).convert(\'RGB\'),(tile,y+24))\n            draw.text((tile+4,y+4),\'Label: white vessels\',fill=\'black\')\n        else:\n            overlay = np.asarray(rgb).copy()\n            overlay[pred] = (0.4*overlay[pred]+0.6*np.array([0,255,100])).astype(np.uint8)\n            canvas.paste(Image.fromarray(overlay),(tile,y+24))\n            draw.text((tile+4,y+4),\'Vessel overlay\',fill=\'black\')\n        canvas.paste(Image.fromarray((pred*255).astype(np.uint8)).convert(\'RGB\'),(tile*2,y+24))\n        draw.text((tile*2+4,y+4),f\'Prediction t={threshold:.3f}\',fill=\'black\')\n    canvas.save(output)\n\n\ndef validation_report(model, paths, labels, cfg, device, output, epoch):\n    probs = {p.stem:predict(model,p,cfg,device) for p in paths}\n    np.savez_compressed(output/\'validation_probabilities.npz\', **probs)\n    thresholds = [round(float(t),3) for t in np.arange(0.2,0.801,0.025)]\n    rows = []\n    for threshold in thresholds:\n        strict = [dice_score(probs[p.stem]>=threshold, strict_label(labels[p.stem],cfg.size)) for p in paths]\n        proxy = [dice_score(probs[p.stem]>=threshold, converter_label(labels[p.stem])) for p in paths]\n        rows.append(dict(threshold=threshold, mean_vessel_dice=float(np.mean(strict)),\n                         converter_target_proxy_dice=float(np.mean(proxy))))\n    best = max(rows,key=lambda r:(r[\'mean_vessel_dice\'],-abs(r[\'threshold\']-0.5)))\n    threshold = best[\'threshold\']\n    with (output/\'threshold_sweep.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,list(rows[0]));writer.writeheader();writer.writerows(rows)\n    per_image=[]\n    for p in paths:\n        pred=probs[p.stem]>=threshold\n        gt=strict_label(labels[p.stem],cfg.size)\n        native=np.asarray(Image.open(labels[p.stem]).convert(\'L\'))>127\n        native_pred=cv2.resize(probs[p.stem],(native.shape[1],native.shape[0]),interpolation=cv2.INTER_LINEAR)>=threshold\n        per_image.append(dict(id=p.stem,dice_05=dice_score(probs[p.stem]>=0.5,gt),dice=dice_score(pred,gt),\n                              native_dice=dice_score(native_pred,native),\n                              converter_target_proxy_dice=dice_score(pred,converter_label(labels[p.stem])),\n                              predicted_fraction=float(pred.mean()),target_fraction=float(gt.mean())))\n    with (output/\'validation_per_image.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,list(per_image[0]));writer.writeheader();writer.writerows(per_image)\n    report=dict(best_epoch=epoch,threshold=threshold,validation_ids=[p.stem for p in paths],\n                mean_vessel_dice_05=float(np.mean([r[\'dice_05\'] for r in per_image])),\n                mean_vessel_dice=best[\'mean_vessel_dice\'],\n                mean_native_vessel_dice=float(np.mean([r[\'native_dice\'] for r in per_image])),\n                converter_target_proxy_dice=best[\'converter_target_proxy_dice\'],\n                evaluation=\'single held-out fold (4 images), not full five-fold OOF\',\n                limitation=\'Epoch and threshold chosen on the same small validation set; development estimate only.\',\n                private_ground_truth_encoding_verified=False)\n    save_json(output/\'validation_metrics.json\',report)\n    preview(paths,probs,labels,threshold,output/\'validation_preview.png\')\n    return report\n\n\ndef export_submission(model, test, template_ids, cfg, device, root, output, threshold, epoch):\n    white_dir=output/\'masks_512_white_vessels\'\n    black_dir=output/\'masks_native_black_vessels\'\n    white_dir.mkdir(exist_ok=True);black_dir.mkdir(exist_ok=True)\n    probs={p.stem:predict(model,p,cfg,device) for p in test}\n    paths={p.stem:p for p in test}\n    white_paths=[]\n    encoded=[]\n    for image_id in template_ids:\n        prob=probs[image_id]\n        mask=prob>=threshold\n        white_path=white_dir/f\'{image_id}.png\'\n        Image.fromarray((mask*255).astype(np.uint8)).save(white_path)\n        white_paths.append(str(white_path))\n        width,height=Image.open(paths[image_id]).size\n        native=cv2.resize(prob,(width,height),interpolation=cv2.INTER_LINEAR)>=threshold\n        Image.fromarray(np.where(native,0,255).astype(np.uint8)).save(black_dir/f\'{image_id}.png\')\n        value=rle_encode(mask)\n        if not np.array_equal(rle_decode(value),mask):\n            raise AssertionError(\'RLE roundtrip mismatch\')\n        encoded.append(value)\n    # Execute the supplied conversion function on the actual PNG predictions.\n    official=load_converter(root).turn_to_str(white_paths)\n    if encoded != official:\n        raise AssertionError(\'Vectorized encoding differs from the official converter\')\n    target=output/\'submission.csv\'\n    with target.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.writer(f);writer.writerow([\'Id\',\'Predicted\']);writer.writerows(zip(template_ids,official))\n    with target.open(newline=\'\',encoding=\'utf-8\') as f:\n        rows=list(csv.DictReader(f))\n    if [r[\'Id\'] for r in rows]!=template_ids or len(rows)!=20:\n        raise AssertionError(\'Saved submission IDs/count mismatch\')\n    for row in rows:\n        rle_decode(row[\'Predicted\'])\n    np.savez_compressed(output/\'test_probabilities.npz\',**probs)\n    preview(test,probs,None,threshold,output/\'test_preview.png\')\n    save_json(output/\'submission_provenance.json\',dict(\n        csv_sha256=file_sha(target),model_sha256=file_sha(output/\'fulltrain/last.pt\'),\n        train_images=20,epochs=epoch,threshold=threshold,external_training_data=False,pretrained=False,\n        csv_polarity=\'white vessels, following observed training labels and supplied converter\',\n        image_output_polarity=\'masks_native_black_vessels: vessel=0, background=255\',\n        official_converter_exact_match=True,rle_roundtrip_all_20=True,template_order_preserved=True,\n        submitted_to_competition=False))\n    print(f\'Submission saved: {target}\',flush=True)\n    return target\n\n\ndef run(phase=\'full\', data_dir=None, output_dir=None, config=None):\n    cfg=config or Config()\n    if cfg.size!=512 or cfg.patch<16 or cfg.patch%16 or cfg.patch>cfg.size or cfg.base_channels<8 or cfg.base_channels%8:\n        raise ValueError(\'Require size=512, patch divisible by 16 <=512, base_channels divisible by 8.\')\n    if cfg.epochs<1 or cfg.patches_per_epoch<1 or cfg.batch_size<1 or cfg.threads<1 or not 2<=cfg.folds<=20 or not 0<=cfg.fold<cfg.folds:\n        raise ValueError(\'Invalid training configuration\')\n    torch.set_num_threads(cfg.threads)\n    cv2.setNumThreads(0)\n    root=find_data(data_dir)\n    output=Path(output_dir or \'task4_baseline_results\').resolve()\n    output.mkdir(parents=True,exist_ok=True)\n    source_sha=file_sha(Path(__file__)) if \'__file__\' in globals() else PIPELINE_SHA256\n    config_path=output/\'run_config.json\'\n    previous=json.loads(config_path.read_text(encoding=\'utf-8\')) if config_path.exists() else None\n    if previous and (previous[\'config\']!=asdict(cfg) or previous[\'pipeline_sha256\']!=source_sha):\n        raise ValueError(\'Existing output belongs to another config/source. Use a new directory.\')\n    train,test,labels,assignment,ids,summary=audit(root,output,cfg)\n    identity=dict(config=asdict(cfg),data_fingerprint=summary[\'data_fingerprint\'],\n                  pipeline_sha256=source_sha)\n    if previous and previous!=identity:\n        raise ValueError(\'Existing output belongs to another config/data/source. Use a new directory.\')\n    save_json(config_path,identity)\n    if phase==\'audit\':\n        print(json.dumps(summary,ensure_ascii=False,indent=2),flush=True)\n        return output\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    print(f\'Device: {device}; train=16, validation=4; no pretrained weights\',flush=True)\n    val=[p for p in train if assignment[p.stem]==cfg.fold]\n    development=[p for p in train if assignment[p.stem]!=cfg.fold]\n    # A single writer protects checkpoints from two simultaneous runs.\n    lock=output/\'training.lock\'\n    try:\n        descriptor=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)\n    except FileExistsError:\n        raise RuntimeError(f\'Run lock exists: {lock}. Check its PID before removing a stale lock.\')\n    os.write(descriptor,str(os.getpid()).encode());os.close(descriptor)\n    try:\n        model,epoch=train_stage(development,val,labels,cfg,output/\'holdout\',device,cfg.epochs)\n        metrics=validation_report(model,val,labels,cfg,device,output,epoch)\n        print(json.dumps(metrics,ensure_ascii=False,indent=2),flush=True)\n        if phase==\'full\':\n            del model\n            if device.type==\'cuda\':torch.cuda.empty_cache()\n            model,full_epoch=train_stage(train,[],labels,cfg,output/\'fulltrain\',device,epoch)\n            export_submission(model,test,ids,cfg,device,root,output,metrics[\'threshold\'],full_epoch)\n    finally:\n        lock.unlink(missing_ok=True)\n    return output\n\n\ndef cli():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--phase\',choices=[\'audit\',\'train\',\'full\'],default=\'full\')\n    parser.add_argument(\'--data-dir\')\n    parser.add_argument(\'--output-dir\',default=\'task4_baseline_results\')\n    for key,value in asdict(Config()).items():\n        if isinstance(value,bool):continue\n        parser.add_argument(\'--\'+key.replace(\'_\',\'-\'),type=type(value),default=value)\n    args=vars(parser.parse_args())\n    phase,data_dir,output=args.pop(\'phase\'),args.pop(\'data_dir\'),args.pop(\'output_dir\')\n    run(phase,data_dir,output,Config(**args))\n\n\nif __name__==\'__main__\':\n    cli()\n', 'task4_kaggle_v2.py': '"""Fixed-epoch five-fold ResNet34 U-Net; native pixels, EMA, paired D4 evaluation.\n\nThe notebook embeds this file AND task4_pipeline.py, so no extra source upload is needed.\n"""\nfrom __future__ import annotations\n\nimport copy\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport platform\nimport random\nimport shutil\nimport socket\nimport time\nimport zipfile\nfrom contextlib import contextmanager\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nimport torchvision\nfrom PIL import Image\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision.models import ResNet34_Weights, resnet34\n\nimport task4_pipeline as base\n\n\n@dataclass\nclass ConfigV2:\n    seed: int = 42\n    folds: int = 5\n    fold: int = 0                 # audit helper only; run_cv uses all requested folds\n    epochs: int = 80\n    patch: int = 256\n    patches_per_epoch: int = 256\n    batch_size: int = 8\n    encoder_lr: float = 0.0001\n    decoder_lr: float = 0.0007\n    weight_decay: float = 0.0001\n    warmup_epochs: int = 3\n    ema_decay: float = 0.99\n    monitor_every: int = 10\n    threads: int = 4\n    amp: bool = True\n    pretrained: bool = True\n    tta_gain_required: float = 0.001\n\n\ndef audit_v2(root, output, cfg):\n    """Freeze the original ID folds; JPEG decoder changes must not reshuffle folds."""\n    train,test,labels,_,ids,summary=base.audit(root,output,cfg)\n    frozen=[[\'2\',\'6\',\'9\',\'10\'],[\'3\',\'14\',\'17\',\'20\'],[\'1\',\'4\',\'5\',\'11\'],\n            [\'15\',\'16\',\'18\',\'19\'],[\'7\',\'8\',\'12\',\'13\']]\n    assignment={i:fold for fold,group in enumerate(frozen) for i in group}\n    if set(assignment)!={p.stem for p in train}:\n        raise ValueError(\'This frozen task-4 split expects train IDs 1 through 20.\')\n    manifest=output/\'manifest.csv\'\n    with manifest.open(newline=\'\',encoding=\'utf-8\') as f:\n        reader=csv.DictReader(f);fields=reader.fieldnames;rows=list(reader)\n    groups={}\n    for row in rows:\n        if row[\'split\']==\'train\':\n            row[\'fold\']=assignment[row[\'id\']]\n            groups.setdefault(row[\'pixel_sha256\'],set()).add(row[\'fold\'])\n    if any(len(folds)>1 for folds in groups.values()):\n        raise ValueError(\'A decoded duplicate crosses frozen folds; inspect dataset before training.\')\n    with manifest.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,fields);writer.writeheader();writer.writerows(rows)\n    # Raw file hashes are stable across JPEG decoder/library versions, unlike pixel hashes.\n    stable=[{k:row[k] for k in [\'split\',\'id\',\'width\',\'height\',\'image_sha256\',\'label_sha256\',\'fold\']}\n            for row in rows]\n    summary.update(data_fingerprint=hashlib.sha256(json.dumps(stable,sort_keys=True).encode()).hexdigest(),\n                   validation_ids=frozen[0],folds=[4]*5,\n                   split_protocol=\'frozen original seed42 ID folds; training seed may vary\')\n    base.save_json(output/\'audit_summary.json\',summary)\n    return train,test,labels,assignment,ids,summary\n\n\ndef read_rgb(path):\n    return np.asarray(Image.open(path).convert(\'RGB\')).copy()\n\n\ndef read_mask(path):\n    return (np.asarray(Image.open(path).convert(\'L\')) > 127).astype(np.float32)\n\n\ndef resize_probability(prob, shape=(512, 512)):\n    # PIL receives a float image: interpolate probabilities, then threshold once.\n    h, w = shape\n    return np.clip(np.asarray(Image.fromarray(np.asarray(prob, np.float32)).resize(\n        (w, h), Image.Resampling.BILINEAR)), 0, 1).copy()\n\n\nclass NativePatches(Dataset):\n    def __init__(self, paths, labels, cfg):\n        self.images = [read_rgb(p) for p in paths]\n        self.masks = [read_mask(labels[p.stem]) for p in paths]\n        self.positives = [np.argwhere(m > 0) for m in self.masks]\n        self.cfg = cfg\n\n    def __len__(self):\n        return self.cfg.patches_per_epoch\n\n    def __getitem__(self, index):\n        i = random.randrange(len(self.images))\n        image, mask = self.images[i], self.masks[i]\n        h, w = mask.shape\n        patch = self.cfg.patch\n        crop = random.randint(int(patch*0.9), min(int(patch*1.1), h, w))\n        if random.random() < 0.3 and len(self.positives[i]):\n            y, x = self.positives[i][random.randrange(len(self.positives[i]))]\n            top, left = int(np.clip(y-crop//2, 0, h-crop)), int(np.clip(x-crop//2, 0, w-crop))\n        else:\n            top, left = random.randint(0, h-crop), random.randint(0, w-crop)\n        image = cv2.resize(image[top:top+crop,left:left+crop], (patch,patch), interpolation=cv2.INTER_LINEAR)\n        mask = cv2.resize(mask[top:top+crop,left:left+crop], (patch,patch), interpolation=cv2.INTER_NEAREST)\n        image, mask = base.paired_d4(image,mask,random.randrange(4),random.random()<0.5)\n        image = image.astype(np.float32)/255\n        image = np.clip((image-0.5)*random.uniform(0.85,1.15)+0.5+random.uniform(-0.04,0.04),0,1)\n        image = image**random.uniform(0.85,1.15)\n        image = np.clip(image*np.array([random.uniform(0.95,1.05) for _ in range(3)],np.float32),0,1)\n        return torch.from_numpy(image.transpose(2,0,1).copy()), torch.from_numpy(mask[None].copy())\n\n\nclass ResNet34UNet(nn.Module):\n    def __init__(self, initial_state=None):\n        super().__init__()\n        self.encoder = resnet34(weights=None)\n        if initial_state is not None:\n            self.encoder.load_state_dict(initial_state,strict=True)\n        self.encoder.fc = nn.Identity()\n        self.shallow = base.ConvBlock(3,32)\n        self.up4 = base.ConvBlock(512+256,256)\n        self.up3 = base.ConvBlock(256+128,128)\n        self.up2 = base.ConvBlock(128+64,64)\n        self.up1 = base.ConvBlock(64+64,48)\n        self.up0 = base.ConvBlock(48+32,32)\n        self.head = nn.Conv2d(32,1,1)\n        self.aux2 = nn.Conv2d(48,1,1)\n        self.aux4 = nn.Conv2d(64,1,1)\n        self.register_buffer(\'mean\',torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))\n        self.register_buffer(\'std\',torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))\n\n    def train(self, mode=True):\n        super().train(mode)\n        # Small batches: keep ImageNet running statistics; affine parameters still train.\n        for module in self.encoder.modules():\n            if isinstance(module,nn.BatchNorm2d):\n                module.eval()\n        return self\n\n    @staticmethod\n    def join(x, skip, block):\n        x = F.interpolate(x,size=skip.shape[-2:],mode=\'bilinear\',align_corners=False)\n        return block(torch.cat([x,skip],1))\n\n    def forward(self, image):\n        x = (image-self.mean)/self.std\n        s0 = self.shallow(x)\n        s1 = self.encoder.relu(self.encoder.bn1(self.encoder.conv1(x)))\n        s2 = self.encoder.layer1(self.encoder.maxpool(s1))\n        s3 = self.encoder.layer2(s2)\n        s4 = self.encoder.layer3(s3)\n        z = self.encoder.layer4(s4)\n        z = self.join(z,s4,self.up4)\n        z = self.join(z,s3,self.up3)\n        z4 = self.join(z,s2,self.up2)\n        z2 = self.join(z4,s1,self.up1)\n        z = self.join(z2,s0,self.up0)\n        return self.head(z),self.aux2(z2),self.aux4(z4)\n\n\ndef objective(outputs, target):\n    losses = []\n    for logits in outputs:\n        truth = target if logits.shape[-2:]==target.shape[-2:] else F.interpolate(\n            target,size=logits.shape[-2:],mode=\'area\')\n        losses.append(base.segmentation_loss(logits,truth))\n    return (losses[0]+0.1*losses[1]+0.2*losses[2])/1.3\n\n\n@torch.no_grad()\ndef update_ema(ema, model, decay):\n    for target, source in zip(ema.parameters(),model.parameters()):\n        target.lerp_(source,1-decay)\n    for target, source in zip(ema.buffers(),model.buffers()):\n        target.copy_(source)\n\n\ndef transform_d4(x, rotation, flip):\n    x = torch.rot90(x,rotation,(-2,-1))\n    return torch.flip(x,(-1,)) if flip else x\n\n\ndef inverse_d4(x, rotation, flip):\n    if flip:\n        x = torch.flip(x,(-1,))\n    return torch.rot90(x,-rotation,(-2,-1))\n\n\ndef window_starts(length, patch, stride):\n    if length < patch:\n        raise ValueError(\'Pad before calculating window starts\')\n    starts = list(range(0,length-patch+1,stride))\n    if starts[-1]!=length-patch:\n        starts.append(length-patch)\n    return starts\n\n\n@torch.inference_mode()\ndef predict_pair(model, image, cfg, device, with_tta=True):\n    """Patchwise D4; return native probabilities for none and D4 using identical windows."""\n    model.eval()\n    if isinstance(image,(str,Path)):\n        image = read_rgb(image)\n    h,w = image.shape[:2]\n    pad_h,pad_w = max(0,cfg.patch-h),max(0,cfg.patch-w)\n    image = cv2.copyMakeBorder(image,0,pad_h,0,pad_w,cv2.BORDER_REFLECT_101)\n    height,width = image.shape[:2]\n    coords = [(y,x) for y in window_starts(height,cfg.patch,cfg.patch//2)\n              for x in window_starts(width,cfg.patch,cfg.patch//2)]\n    weight = np.maximum(np.outer(np.hanning(cfg.patch),np.hanning(cfg.patch)),0.05).astype(np.float32)\n    accum = np.zeros((2,height,width),np.float32)\n    weights = np.zeros((height,width),np.float32)\n    transforms = [(0,False)] + ([(r,f) for r in range(4) for f in [False,True] if (r,f)!=(0,False)] if with_tta else [])\n    for start in range(0,len(coords),cfg.batch_size):\n        positions = coords[start:start+cfg.batch_size]\n        batch = np.stack([image[y:y+cfg.patch,x:x+cfg.patch].transpose(2,0,1) for y,x in positions])\n        tensor = torch.from_numpy(batch.copy()).to(device).float()/255\n        plain,total = None,None\n        for rotation,flip in transforms:\n            transformed = transform_d4(tensor,rotation,flip)\n            with torch.autocast(device_type=device.type,enabled=cfg.amp and device.type==\'cuda\'):\n                prediction = model(transformed)[0].float().sigmoid()\n            prediction = inverse_d4(prediction,rotation,flip)\n            if plain is None:\n                plain = prediction\n                total = prediction.clone()\n            else:\n                total.add_(prediction)\n        pair = torch.stack([plain,total/len(transforms)])[:,:,0].cpu().numpy()\n        for i,(y,x) in enumerate(positions):\n            accum[:,y:y+cfg.patch,x:x+cfg.patch] += pair[:,i]*weight\n            weights[y:y+cfg.patch,x:x+cfg.patch] += weight\n    result = accum/weights[None]\n    if not np.isfinite(result).all():\n        raise FloatingPointError(\'Nonfinite inference output\')\n    return {\'none\':result[0,:h,:w].copy(),\'d4\':result[1,:h,:w].copy()}\n\n\ndef resolve_weights(cfg, output, weights_path=None):\n    if not cfg.pretrained:\n        return None,dict(source=\'random initialization; debug/ablation only\',sha256=None)\n    enum = ResNet34_Weights.IMAGENET1K_V1\n    target = Path(weights_path) if weights_path else output/\'pretrained_cache\'/Path(enum.url).name\n    if not target.is_file():\n        if weights_path:\n            raise FileNotFoundError(target)\n        target.parent.mkdir(parents=True,exist_ok=True)\n        print(\'Downloading official ImageNet encoder weights:\',enum.url,flush=True)\n        torch.hub.download_url_to_file(enum.url,str(target),hash_prefix=\'b627a593\',progress=True)\n    digest = base.file_sha(target)\n    if not digest.startswith(\'b627a593\'):\n        raise ValueError(\'Expected official torchvision resnet34-b627a593.pth, not task-finetuned weights.\')\n    state = torch.load(target,map_location=\'cpu\',weights_only=True)\n    probe = resnet34(weights=None)\n    probe.load_state_dict(state,strict=True)\n    del probe\n    cached=output/\'pretrained_cache\'/Path(enum.url).name\n    if target.resolve()!=cached.resolve():\n        cached.parent.mkdir(parents=True,exist_ok=True)\n        shutil.copyfile(target,cached)\n    return state,dict(source=enum.url,sha256=digest,supervision=\'generic ImageNet classification, not vessel segmentation\')\n\n\ndef set_lrs(optimizer, step, steps_per_epoch, cfg):\n    total = cfg.epochs*steps_per_epoch\n    warm = min(cfg.warmup_epochs*steps_per_epoch,max(0,total-1))\n    if warm and step<warm:\n        factor = (step+1)/warm\n    else:\n        progress = (step-warm)/max(1,total-warm-1)\n        factor = 0.02+0.98*0.5*(1+math.cos(math.pi*min(1,progress)))\n    for group,initial in zip(optimizer.param_groups,[cfg.encoder_lr,cfg.decoder_lr]):\n        group[\'lr\'] = initial*factor\n\n\ndef train_fold(paths, val_paths, labels, cfg, fold, folder, initial_state, device):\n    folder.mkdir(parents=True,exist_ok=True)\n    base.seed_all(cfg.seed+1009*fold)\n    dataset = NativePatches(paths,labels,cfg)\n    loader = DataLoader(dataset,batch_size=cfg.batch_size,num_workers=0,shuffle=False,\n                        pin_memory=device.type==\'cuda\',drop_last=False)\n    model = ResNet34UNet(initial_state).to(device)\n    ema = copy.deepcopy(model).eval().requires_grad_(False)\n    encoder = list(model.encoder.parameters())\n    decoder = [p for name,p in model.named_parameters() if not name.startswith(\'encoder.\')]\n    optimizer = torch.optim.AdamW([{\'params\':encoder,\'lr\':cfg.encoder_lr},\n                                 {\'params\':decoder,\'lr\':cfg.decoder_lr}],weight_decay=cfg.weight_decay)\n    scaler = torch.amp.GradScaler(\'cuda\',enabled=cfg.amp and device.type==\'cuda\')\n    history,first,step = [],1,0\n    train_ids,val_ids = [p.stem for p in paths],[p.stem for p in val_paths]\n    last = folder/\'last.pt\'\n    if last.exists():\n        ckpt = torch.load(last,map_location=\'cpu\',weights_only=False)\n        if ckpt[\'train_ids\']!=train_ids or ckpt[\'val_ids\']!=val_ids:\n            raise ValueError(\'Checkpoint split mismatch\')\n        model.load_state_dict(ckpt[\'model\']);ema.load_state_dict(ckpt[\'ema\'])\n        optimizer.load_state_dict(ckpt[\'optimizer\']);scaler.load_state_dict(ckpt[\'scaler\'])\n        history,first,step = ckpt[\'history\'],ckpt[\'epoch\']+1,ckpt[\'step\']\n        base.restore_rng(ckpt[\'rng\'])\n        del ckpt\n        print(f\'fold{fold}: resume at {first}/{cfg.epochs}\',flush=True)\n    for epoch in range(first,cfg.epochs+1):\n        begin = time.perf_counter()\n        model.train()\n        losses = []\n        for image,target in loader:\n            set_lrs(optimizer,step,len(loader),cfg)\n            image,target = image.to(device,non_blocking=True),target.to(device,non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type,enabled=cfg.amp and device.type==\'cuda\'):\n                loss = objective(model(image),target)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Nonfinite training loss\')\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(model.parameters(),5.0)\n            old_scale = scaler.get_scale()\n            scaler.step(optimizer);scaler.update()\n            if scaler.get_scale()>=old_scale:\n                update_ema(ema,model,cfg.ema_decay)\n            losses.append(float(loss.detach()))\n            step += 1\n        score = None\n        if epoch%cfg.monitor_every==0 or epoch==cfg.epochs:\n            score = float(np.mean([base.dice_score(\n                resize_probability(predict_pair(ema,p,cfg,device,False)[\'none\'])>=0.5,\n                base.strict_label(labels[p.stem])) for p in val_paths]))\n        row = dict(epoch=epoch,loss=float(np.mean(losses)),monitor_dice_05=score,\n                   encoder_lr=optimizer.param_groups[0][\'lr\'],decoder_lr=optimizer.param_groups[1][\'lr\'],\n                   seconds=round(time.perf_counter()-begin,3))\n        history.append(row)\n        if shutil.disk_usage(folder).free<1200*1024**2:\n            raise OSError(\'Less than 1.2 GiB free: preserve old checkpoint and free space before resuming.\')\n        base.atomic_checkpoint(last,dict(model=model.state_dict(),ema=ema.state_dict(),\n            optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),history=history,\n            epoch=epoch,step=step,rng=base.rng_state(),train_ids=train_ids,val_ids=val_ids))\n        with (folder/\'history.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n            writer=csv.DictWriter(f,list(row));writer.writeheader();writer.writerows(history)\n        print(f\'fold{fold} {epoch:03d}/{cfg.epochs} loss={row["loss"]:.4f} \'+\n              (f\'monitor_dice@0.5={score:.5f} \' if score is not None else \'\')+\n              f\'seconds={row["seconds"]}\',flush=True)\n        if epoch==1:\n            print(f\'fold{fold}: measured estimate for this fold training/monitoring: \'\n                  f\'{row["seconds"]*cfg.epochs/60:.1f} min (startup may inflate this; D4 export extra)\',flush=True)\n    # Final EMA is fixed in advance. Validation never selects epochs/checkpoints.\n    base.atomic_checkpoint(folder/\'final_ema.pt\',dict(model=ema.state_dict(),epoch=cfg.epochs,\n                          train_ids=train_ids,val_ids=val_ids,selection=\'predeclared final EMA\'))\n    del model,optimizer,scaler,dataset,loader\n    if device.type==\'cuda\':torch.cuda.empty_cache()\n    return ema\n\n\ndef infer_fold(model, val_paths, test, labels, cfg, folder, device):\n    val_probs,test_probs = {},{}\n    for p in val_paths:\n        pair = predict_pair(model,p,cfg,device)\n        for mode,native in pair.items():\n            val_probs[f\'{mode}_{p.stem}\'] = resize_probability(native)\n            val_probs[f\'native_{mode}_{p.stem}\'] = native\n    np.savez_compressed(folder/\'validation_probabilities.npz\',**val_probs)\n    for p in test:\n        pair = predict_pair(model,p,cfg,device)\n        for mode,native in pair.items():\n            test_probs[f\'{mode}_{p.stem}\'] = resize_probability(native)\n    np.savez_compressed(folder/\'test_probabilities.npz\',**test_probs)\n    clean = {p.stem:base.strict_label(labels[p.stem]) for p in val_paths}\n    summary = dict(epoch=cfg.epochs,val_ids=[p.stem for p in val_paths],\n        none_dice_05=float(np.mean([base.dice_score(val_probs[f\'none_{i}\']>=0.5,clean[i]) for i in clean])),\n        d4_dice_05=float(np.mean([base.dice_score(val_probs[f\'d4_{i}\']>=0.5,clean[i]) for i in clean])))\n    files=[\'history.csv\',\'last.pt\',\'final_ema.pt\',\'validation_probabilities.npz\',\'test_probabilities.npz\']\n    summary[\'sha256\']={name:base.file_sha(folder/name) for name in files}\n    base.save_json(folder/\'completed.json\',summary)\n    print(\'Fold completed:\',summary[\'val_ids\'],summary[\'none_dice_05\'],summary[\'d4_dice_05\'],flush=True)\n\n\ndef threshold_sweep(probs, targets):\n    thresholds = np.round(np.arange(0.1,0.901,0.025),3)\n    rows=[dict(threshold=float(t),mean_dice=float(np.mean([\n        base.dice_score(probs[i]>=t,targets[i]) for i in probs]))) for t in thresholds]\n    best=max(rows,key=lambda r:(r[\'mean_dice\'],-abs(r[\'threshold\']-0.5)))\n    return best,rows\n\n\ndef export_csv(probs, ids, threshold, name, root, output):\n    folder=output/\'masks_512_white_vessels\'/name\n    folder.mkdir(parents=True,exist_ok=True)\n    filenames,encoded=[],[]\n    for i in ids:\n        mask=probs[i]>=threshold\n        path=folder/f\'{i}.png\';Image.fromarray((mask*255).astype(np.uint8)).save(path)\n        value=base.rle_encode(mask)\n        if not np.array_equal(base.rle_decode(value),mask):\n            raise AssertionError(\'RLE mismatch\')\n        filenames.append(str(path));encoded.append(value)\n    official=base.load_converter(root).turn_to_str(filenames)\n    if official!=encoded:raise AssertionError(\'Official converter mismatch\')\n    path=output/f\'{name}.csv\'\n    with path.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.writer(f);writer.writerow([\'Id\',\'Predicted\']);writer.writerows(zip(ids,official))\n    return dict(file=path.name,threshold=threshold,sha256=base.file_sha(path),\n                rows=len(ids),official_converter_exact_match=True,rle_roundtrip=True)\n\n\ndef aggregate(root, output, cfg, train, test, labels, assignment, ids):\n    folds=[output/f\'fold{f}\' for f in range(cfg.folds)]\n    if not all((folder/\'completed.json\').exists() for folder in folds):\n        finished=[f for f,folder in enumerate(folds) if (folder/\'completed.json\').exists()]\n        base.save_json(output/\'partial_cv_summary.json\',dict(completed_folds=finished,complete_oof=False))\n        print(\'Partial CV only; no full-OOF score or ensemble submission. Completed:\',finished,flush=True)\n        return None\n    oof={mode:{} for mode in [\'none\',\'d4\']}\n    native={mode:{} for mode in [\'none\',\'d4\']}\n    test_mean={mode:{i:np.zeros((512,512),np.float64) for i in ids} for mode in oof}\n    for fold,folder in enumerate(folds):\n        metadata=json.loads((folder/\'completed.json\').read_text(encoding=\'utf-8\'))\n        for name,digest in metadata[\'sha256\'].items():\n            if base.file_sha(folder/name)!=digest:raise ValueError(f\'Artifact changed: {folder/name}\')\n        expected={p.stem for p in train if assignment[p.stem]==fold}\n        if set(metadata[\'val_ids\'])!=expected:raise ValueError(\'Validation IDs mismatch\')\n        with np.load(folder/\'validation_probabilities.npz\') as values:\n            for mode in oof:\n                for i in expected:\n                    if i in oof[mode]:raise AssertionError(\'Repeated OOF ID\')\n                    oof[mode][i]=values[f\'{mode}_{i}\'].copy()\n                    native[mode][i]=values[f\'native_{mode}_{i}\'].copy()\n        with np.load(folder/\'test_probabilities.npz\') as values:\n            for mode in oof:\n                for i in ids:test_mean[mode][i]+=values[f\'{mode}_{i}\']/cfg.folds\n    train_ids={p.stem for p in train}\n    if any(set(probs)!=train_ids for probs in oof.values()):raise AssertionError(\'Incomplete OOF coverage\')\n    clean={i:base.strict_label(labels[i]) for i in train_ids}\n    proxy={i:base.converter_label(labels[i]) for i in train_ids}\n    rows,mode_summary,exported=[],{},{}\n    for mode in oof:\n        calibrated,sweep=threshold_sweep(oof[mode],clean)\n        proxy_calibrated,proxy_sweep=threshold_sweep(oof[mode],proxy)\n        threshold=calibrated[\'threshold\']\n        cross_thresholds={}\n        for fold in range(cfg.folds):\n            other={j:oof[mode][j] for j in train_ids if assignment[j]!=fold}\n            cross_thresholds[fold]=threshold_sweep(other,{j:clean[j] for j in other})[0][\'threshold\']\n        with (output/f\'threshold_sweep_{mode}.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n            writer=csv.DictWriter(f,[\'threshold\',\'clean_dice\',\'converter_proxy_dice\']);writer.writeheader()\n            writer.writerows(dict(threshold=a[\'threshold\'],clean_dice=a[\'mean_dice\'],converter_proxy_dice=b[\'mean_dice\'])\n                             for a,b in zip(sweep,proxy_sweep))\n        per=[]\n        for i in sorted(train_ids,key=int):\n            # Threshold calibration excluding this outer fold: diagnostic, not nested CV.\n            cross_t=cross_thresholds[assignment[i]]\n            gt_native=read_mask(labels[i])>0\n            row=dict(id=i,fold=assignment[i],mode=mode,\n                dice_05=base.dice_score(oof[mode][i]>=0.5,clean[i]),\n                dice_global_threshold=base.dice_score(oof[mode][i]>=threshold,clean[i]),\n                dice_other_fold_threshold=base.dice_score(oof[mode][i]>=cross_t,clean[i]),\n                other_fold_threshold=cross_t,\n                native_dice_05=base.dice_score(native[mode][i]>=0.5,gt_native),\n                converter_proxy_dice=base.dice_score(oof[mode][i]>=threshold,proxy[i]))\n            rows.append(row);per.append(row)\n        mode_summary[mode]=dict(mean_dice_05=float(np.mean([r[\'dice_05\'] for r in per])),\n            global_threshold=threshold,mean_dice_global_threshold=calibrated[\'mean_dice\'],\n            mean_dice_other_fold_threshold=float(np.mean([r[\'dice_other_fold_threshold\'] for r in per])),\n            mean_native_dice_05=float(np.mean([r[\'native_dice_05\'] for r in per])),\n            mean_converter_proxy_dice=float(np.mean([r[\'converter_proxy_dice\'] for r in per])),\n            converter_proxy_best_threshold=proxy_calibrated[\'threshold\'],\n            converter_proxy_best_dice=proxy_calibrated[\'mean_dice\'])\n        exported[mode]=export_csv(test_mean[mode],ids,threshold,f\'submission_{mode}\',root,output)\n    chosen=\'d4\' if mode_summary[\'d4\'][\'mean_dice_05\']>=mode_summary[\'none\'][\'mean_dice_05\']+cfg.tta_gain_required else \'none\'\n    source=output/exported[chosen][\'file\']\n    shutil.copyfile(source,output/\'submission.csv\')\n    # Explicitly named diagnostic candidate; never silently substitute it for clean vessels.\n    proxy_file=export_csv(test_mean[chosen],ids,mode_summary[chosen][\'converter_proxy_best_threshold\'],\n                         \'submission_converter_proxy\',root,output)\n    threshold=mode_summary[chosen][\'global_threshold\']\n    base.preview(test,{p.stem:test_mean[chosen][p.stem] for p in test},None,threshold,output/\'test_preview.png\')\n    fold0=[p for p in train if assignment[p.stem]==0]\n    base.preview(fold0,oof[chosen],labels,threshold,output/\'oof_fold0_preview.png\')\n    black=output/\'masks_native_black_vessels\';black.mkdir(exist_ok=True)\n    for p in test:\n        width,height=Image.open(p).size\n        prob=resize_probability(test_mean[chosen][p.stem],(height,width))\n        Image.fromarray(np.where(prob>=threshold,0,255).astype(np.uint8)).save(black/f\'{p.stem}.png\')\n    with (output/\'oof_per_image.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,list(rows[0]));writer.writeheader();writer.writerows(rows)\n    np.savez_compressed(output/\'oof_probabilities.npz\',**{f\'{mode}_{i}\':p for mode,probs in oof.items() for i,p in probs.items()})\n    np.savez_compressed(output/\'test_ensemble_probabilities.npz\',**{f\'{mode}_{i}\':p.astype(np.float32) for mode,probs in test_mean.items() for i,p in probs.items()})\n    summary=dict(complete_oof=True,unique_oof_images=len(train_ids),epochs=cfg.epochs,\n        checkpoint_selection=\'fixed final EMA; no outer-validation epoch selection\',\n        modes=mode_summary,selected_mode=chosen,tta_gain_required=cfg.tta_gain_required,\n        test_ensemble=\'equal average of five fold models; each trained on 16 images\',\n        baseline_reference=dict(ids=[\'2\',\'6\',\'9\',\'10\'],baseline_dice_05=0.7839949354578135,\n            v2_same_ids_none_dice_05=float(np.mean([r[\'dice_05\'] for r in rows if r[\'id\'] in {\'2\',\'6\',\'9\',\'10\'} and r[\'mode\']==\'none\'])),\n            note=\'Matched subset only; baseline did not produce full five-fold OOF.\'),\n        selection_caveat=\'TTA mode and global threshold use OOF labels. Tuned scores are development estimates. \'\n                         \'Other-fold calibration is diagnostic, not a nested independent test.\',\n        private_target_preprocessing_verified=False,submitted_to_competition=False)\n    base.save_json(output/\'cv_summary.json\',summary)\n    base.save_json(output/\'submission_provenance.json\',dict(chosen_mode=chosen,\n        primary_csv_sha256=base.file_sha(output/\'submission.csv\'),alternatives=exported,\n        converter_proxy_candidate=proxy_file,fold_models=[base.file_sha(p/\'final_ema.pt\') for p in folds],\n        calibrated_on=\'20 OOF clean masks; proxy candidate separately calibrated on JPEG converter targets\',\n        image_polarity=\'CSV white vessels; separate native-size PNGs have vessel=0, background=255\',\n        official_converter_exact_match=True,external_vessel_training_data=False))\n    print(json.dumps(summary,ensure_ascii=False,indent=2),flush=True)\n    return summary\n\n\ndef make_backups(output, full=True):\n    """Portable analysis archive; optional full archive includes model and resume states."""\n    output=Path(output)\n    files=[p for p in output.rglob(\'*\') if p.is_file() and p.suffix not in {\'.zip\',\'.tmp\'} and p.name!=\'training.lock\']\n    light=[p for p in files if p.suffix in {\'.json\',\'.csv\',\'.png\',\'.py\',\'.md\'} and \'pretrained_cache\' not in p.parts\n           and not (p.suffix==\'.png\' and \'masks_512_white_vessels\' in p.parts)]\n    for name,paths in [(\'analysis_results.zip\',light)]+([(\'full_backup.zip\',files)] if full else []):\n        target=output/name;tmp=output/(name+\'.tmp\')\n        estimated=sum(p.stat().st_size for p in paths)\n        if shutil.disk_usage(output).free<estimated+256*1024**2:\n            raise OSError(f\'Not enough free space to create {name}; completed results remain intact.\')\n        with zipfile.ZipFile(tmp,\'w\',compression=zipfile.ZIP_STORED,allowZip64=True) as z:\n            for path in paths:z.write(path,path.relative_to(output).as_posix())\n        with zipfile.ZipFile(tmp) as z:\n            bad=z.testzip()\n            if bad:raise IOError(f\'Archive CRC failed: {bad}\')\n        os.replace(tmp,target)\n        print(\'Backup saved:\',target,f\'{target.stat().st_size/1024**2:.1f} MiB\',flush=True)\n\n\ndef restore_output(source, output):\n    source,output=Path(source).resolve(),Path(output).resolve()\n    if source==output:return\n    if output.exists() and any(output.iterdir()):\n        # Existing working output is the authoritative continuation; do not overlay backup.\n        if (output/\'run_config.json\').exists():return\n        raise ValueError(\'Restore target is not empty\')\n    output.mkdir(parents=True,exist_ok=True)\n    if source.is_dir():\n        shutil.copytree(source,output,dirs_exist_ok=True,ignore=shutil.ignore_patterns(\'*.zip\',\'*.tmp\',\'training.lock\'))\n    elif source.suffix.lower()==\'.zip\':\n        with zipfile.ZipFile(source) as z:\n            for item in z.infolist():\n                resolved=(output/item.filename).resolve()\n                if not resolved.is_relative_to(output):raise ValueError(\'Unsafe archive path\')\n            z.extractall(output)\n        (output/\'training.lock\').unlink(missing_ok=True)\n    else:raise ValueError(\'RESUME_FROM must be an existing output folder or full_backup.zip\')\n\n\n@contextmanager\ndef run_lock(output):\n    lock=output/\'training.lock\'\n    if lock.exists():\n        info=json.loads(lock.read_text())\n        alive=False\n        if info.get(\'host\')==socket.gethostname():\n            if os.name==\'nt\':\n                import ctypes\n                from ctypes import wintypes\n                kernel=ctypes.WinDLL(\'kernel32\',use_last_error=True)\n                kernel.OpenProcess.argtypes=[wintypes.DWORD,wintypes.BOOL,wintypes.DWORD]\n                kernel.OpenProcess.restype=wintypes.HANDLE\n                kernel.GetExitCodeProcess.argtypes=[wintypes.HANDLE,ctypes.POINTER(wintypes.DWORD)]\n                kernel.CloseHandle.argtypes=[wintypes.HANDLE]\n                handle=kernel.OpenProcess(0x1000,False,int(info[\'pid\']))\n                if handle:\n                    code=wintypes.DWORD()\n                    alive=bool(kernel.GetExitCodeProcess(handle,ctypes.byref(code))) and code.value==259\n                    kernel.CloseHandle(handle)\n            else:\n                try:os.kill(info[\'pid\'],0);alive=True\n                except OSError:pass\n        if alive:raise RuntimeError(f\'Output is in use by PID {info["pid"]}\')\n        lock.unlink()\n    descriptor=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)\n    os.write(descriptor,json.dumps(dict(pid=os.getpid(),host=socket.gethostname())).encode());os.close(descriptor)\n    try:yield\n    finally:lock.unlink(missing_ok=True)\n\n\ndef run_cv(data_dir=None, output_dir=\'task4_v2_results\', cfg=None, folds_to_run=(0,1,2,3,4),\n           weights_path=None, resume_from=None, backups=True, full_backup=True):\n    cfg=cfg or ConfigV2()\n    if cfg.folds!=5 or cfg.fold!=0 or cfg.patch<64 or cfg.patch%32 or cfg.patch>512:\n        raise ValueError(\'Require 5 folds, audit fold=0, patch multiple of 32 in [64,512].\')\n    if cfg.epochs<1 or cfg.batch_size<1 or cfg.patches_per_epoch<1 or cfg.monitor_every<1:\n        raise ValueError(\'Invalid training configuration\')\n    if not 0<cfg.ema_decay<1 or len(set(folds_to_run))!=len(folds_to_run) or not set(folds_to_run)<=set(range(5)):\n        raise ValueError(\'Invalid EMA decay/fold list\')\n    torch.set_num_threads(cfg.threads);cv2.setNumThreads(0)\n    output=Path(output_dir).resolve()\n    if resume_from:restore_output(resume_from,output)\n    output.mkdir(parents=True,exist_ok=True)\n    root=base.find_data(data_dir)\n    identity=dict(config=asdict(cfg),source_sha256=base.file_sha(__file__),\n                  baseline_helpers_sha256=base.file_sha(base.__file__))\n    config_path=output/\'run_config.json\'\n    previous=json.loads(config_path.read_text(encoding=\'utf-8\')) if config_path.exists() else None\n    if previous and any(previous.get(k)!=v for k,v in identity.items()):\n        raise ValueError(\'Changed config/source: use a NEW OUTPUT_DIR to preserve prior results.\')\n    with run_lock(output):\n        train,test,labels,assignment,ids,audit=audit_v2(root,output,cfg)\n        identity[\'data_fingerprint\']=audit[\'data_fingerprint\']\n        if previous and previous[\'data_fingerprint\']!=identity[\'data_fingerprint\']:\n            raise ValueError(\'Dataset changed: use a new output directory\')\n        initial_state,provenance=resolve_weights(cfg,output,weights_path)\n        identity[\'pretrained_sha256\']=provenance[\'sha256\']\n        if previous and previous!=identity:raise ValueError(\'Pretrained weights differ from saved run\')\n        base.save_json(config_path,identity)\n        base.save_json(output/\'pretrained_source.json\',provenance)\n        base.save_json(output/\'environment.json\',dict(python=platform.python_version(),torch=torch.__version__,\n            torchvision=torchvision.__version__,opencv=cv2.__version__,numpy=np.__version__,\n            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n            training_device_count=1,torch_cuda=torch.version.cuda))\n        code_dir=output/\'runtime_source\';code_dir.mkdir(exist_ok=True)\n        for p in [Path(__file__),Path(base.__file__)]:\n            if p.resolve()!=(code_dir/p.name).resolve():shutil.copyfile(p,code_dir/p.name)\n        device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        print(\'Device:\',device,\'fixed epochs:\',cfg.epochs,\'folds requested:\',folds_to_run,flush=True)\n        if device.type!=\'cuda\':print(\'CPU detected; formal five-fold training will be slow. Enable GPU.\',flush=True)\n        for fold in folds_to_run:\n            folder=output/f\'fold{fold}\'\n            if (folder/\'completed.json\').exists():\n                print(f\'fold{fold}: completed, reusing verified artifacts\',flush=True)\n                continue\n            development=[p for p in train if assignment[p.stem]!=fold]\n            validation=[p for p in train if assignment[p.stem]==fold]\n            model=train_fold(development,validation,labels,cfg,fold,folder,initial_state,device)\n            infer_fold(model,validation,test,labels,cfg,folder,device)\n            del model\n            if device.type==\'cuda\':torch.cuda.empty_cache()\n        summary=aggregate(root,output,cfg,train,test,labels,assignment,ids)\n        if backups:make_backups(output,full=full_backup)\n    return output,summary\n', 'task4_controlled_ablation.py': '"""OOF-only boundary/EMA replay and paired label-supervision experiments.\n\nNo test-set predictions, competition submissions, or public-score-based selection.\n"""\nfrom __future__ import annotations\n\nimport copy\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport shutil\nimport time\nimport zipfile\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageDraw\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nimport task4_pipeline as base\nimport task4_kaggle_v2 as v2\n\n\nVARIANTS=(\'L0_clean\',\'L1_proxy\',\'L2_soft\')\nCALIBRATION_GRID=(0.05,0.075,0.1,0.125,0.15,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9)\n\n\n@dataclass\nclass AblationConfig(v2.ConfigV2):\n    soft_boundary: float = 0.5\n\n\ndef digest_json(value):\n    return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()\n\n\ndef write_table(path, rows):\n    if not rows:return\n    with Path(path).open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,list(rows[0]));writer.writeheader();writer.writerows(rows)\n\n\ndef skeleton(mask):\n    try:\n        from skimage.morphology import skeletonize\n    except ImportError as exc:\n        raise RuntimeError(\'Install scikit-image for the preregistered centerline metrics.\') from exc\n    return skeletonize(np.asarray(mask,bool))\n\n\ndef targets(labels):\n    result={}\n    for i,path in labels.items():\n        c,p=base.strict_label(path),base.converter_label(path)\n        if np.any(c&~p):raise ValueError(\'Expected clean mask to be a subset of converter proxy\')\n        sc,sp=skeleton(c),skeleton(p)\n        radius=cv2.distanceTransform(c.astype(np.uint8),cv2.DIST_L2,5)\n        result[i]=dict(clean=c,proxy=p,clean_skeleton=sc,proxy_skeleton=sp,\n                       thin_clean_centerline=sc&(radius<=1.5))\n    return result\n\n\ndef segmentation_metrics(pred, truth, truth_skeleton, pred_skeleton):\n    pred,truth=np.asarray(pred,bool),np.asarray(truth,bool)\n    tp=int((pred&truth).sum());n_pred=int(pred.sum());n_true=int(truth.sum())\n    precision=tp/n_pred if n_pred else (1.0 if not n_true else 0.0)\n    recall=tp/n_true if n_true else 1.0\n    top_precision=float((pred_skeleton&truth).sum()/pred_skeleton.sum()) if pred_skeleton.any() else (1.0 if not n_true else 0.0)\n    top_recall=float((truth_skeleton&pred).sum()/truth_skeleton.sum()) if truth_skeleton.any() else 1.0\n    cldice=2*top_precision*top_recall/(top_precision+top_recall) if top_precision+top_recall else 0.0\n    return dict(dice=base.dice_score(pred,truth),precision=precision,recall=recall,\n                centerline_precision=top_precision,centerline_recall=top_recall,cldice=cldice,\n                foreground_fraction=n_pred/pred.size)\n\n\ndef evaluate_masks(experiment, masks, truth, assignments):\n    rows=[]\n    for i in sorted(masks,key=int):\n        mask=np.asarray(masks[i],bool);sp=skeleton(mask)\n        thin=truth[i][\'thin_clean_centerline\']\n        thin_recall=float((mask&thin).sum()/thin.sum()) if thin.any() else None\n        for target in [\'clean\',\'proxy\']:\n            values=segmentation_metrics(mask,truth[i][target],truth[i][target+\'_skeleton\'],sp)\n            rows.append(dict(experiment=experiment,id=i,fold=assignments[i],target=target,\n                             **values,thin_clean_centerline_recall=thin_recall))\n    return rows\n\n\ndef summarize_rows(rows):\n    numeric=[\'dice\',\'precision\',\'recall\',\'centerline_precision\',\'centerline_recall\',\'cldice\',\n             \'foreground_fraction\',\'thin_clean_centerline_recall\']\n    summaries,folds=[],[]\n    for experiment,target in sorted({(r[\'experiment\'],r[\'target\']) for r in rows}):\n        group=[r for r in rows if r[\'experiment\']==experiment and r[\'target\']==target]\n        assert len({r[\'id\'] for r in group})==len(group)\n        values={key:float(np.mean([r[key] for r in group if r[key] is not None]))\n                if any(r[key] is not None for r in group) else None for key in numeric}\n        fold_values=[]\n        for fold in sorted({r[\'fold\'] for r in group}):\n            subset=[r for r in group if r[\'fold\']==fold]\n            score=float(np.mean([r[\'dice\'] for r in subset]));fold_values.append(score)\n            folds.append(dict(experiment=experiment,target=target,fold=fold,n_images=len(subset),\n                              **{key:float(np.mean([r[key] for r in subset if r[key] is not None]))\n                                 if any(r[key] is not None for r in subset) else None for key in numeric}))\n        summaries.append(dict(experiment=experiment,target=target,n_images=len(group),\n            complete_20_image_oof=len(group)==20,**values,worst_fold_dice=min(fold_values),\n            fold_dice_std=float(np.std(fold_values))))\n    return summaries,folds\n\n\ndef probability_preview(images, truth, probs, output):\n    ids=sorted(probs,key=int)[:4];tile=220\n    canvas=Image.new(\'RGB\',(tile*4,(tile+24)*len(ids)),\'white\');draw=ImageDraw.Draw(canvas)\n    for row,i in enumerate(ids):\n        y=row*(tile+24)\n        p=probs[i]\n        hard=p>=0.4;low=p>=0.1\n        dil=cv2.dilate(hard.astype(np.uint8),cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3)))>0\n        panels=[Image.open(images[i]).convert(\'RGB\')]+[Image.fromarray((m*255).astype(np.uint8)).convert(\'RGB\') for m in [hard,low,dil]]\n        for col,(panel,label) in enumerate(zip(panels,[f\'OOF {i}\',\'p >= 0.4\',\'p >= 0.1\',\'dilate(p >= 0.4)\'])):\n            draw.text((col*tile+3,y+4),label,fill=\'black\')\n            canvas.paste(panel.resize((tile,tile)),(col*tile,y+24))\n    canvas.save(output)\n\n\ndef materialize_previous(source, output):\n    """Use a mounted folder read-only, or extract only the artifacts used by replay."""\n    if source is None:\n        preferred=Path(\'/kaggle/working/task4_v2_results\')\n        roots=[preferred] if (preferred/\'run_config.json\').exists() else []\n        if not roots and Path(\'/kaggle/input\').exists():\n            for p in Path(\'/kaggle/input\').rglob(\'run_config.json\'):\n                try:cfg=json.loads(p.read_text())\n                except (ValueError,OSError):continue\n                if cfg.get(\'source_sha256\')==base.file_sha(v2.__file__) and all((p.parent/f\'fold{f}/validation_probabilities.npz\').exists() for f in range(5)):\n                    roots.append(p.parent)\n        roots=sorted(set(p.resolve() for p in roots))\n        if len(roots)==1:return roots[0]\n        zips=list(Path(\'/kaggle/input\').rglob(\'full_backup.zip\')) if Path(\'/kaggle/input\').exists() else []\n        if not roots and len(zips)==1:source=zips[0]\n        else:raise ValueError(f\'Set PREVIOUS_RUN to the previous complete output folder or full_backup.zip. Folders found: {roots}\')\n    source=Path(source).resolve()\n    if source.is_dir():return source\n    if not source.is_file() or source.suffix.lower()!=\'.zip\':raise ValueError(\'PREVIOUS_RUN must be a folder or full_backup.zip\')\n    cached=output/\'previous_run_cache\';cached.mkdir(parents=True,exist_ok=True)\n    archive_sha=base.file_sha(source);marker=cached/\'archive_identity.json\'\n    if marker.exists():\n        if json.loads(marker.read_text())[\'archive_sha256\']!=archive_sha:raise ValueError(\'Different previous archive: use a new OUTPUT_DIR\')\n        return cached\n    with zipfile.ZipFile(source) as z:\n        names=z.namelist()\n        if len(names)!=len(set(names)):raise ValueError(\'Duplicate ZIP members\')\n        wanted=[]\n        for name in names:\n            dest=(cached/name).resolve()\n            if not dest.is_relative_to(cached):raise ValueError(\'Unsafe archive path\')\n            parts=Path(name).parts\n            if name in {\'run_config.json\',\'pretrained_source.json\'} or (\n                len(parts)==2 and parts[0] in {f\'fold{f}\' for f in range(5)} and parts[1] in {\n                    \'completed.json\',\'last.pt\',\'validation_probabilities.npz\',\'history.csv\'}) or (\n                len(parts)==2 and parts[0]==\'pretrained_cache\' and parts[1]==\'resnet34-b627a593.pth\'):\n                wanted.append(name)\n        required_members={\'run_config.json\'}|{f\'fold{f}/{name}\' for f in range(5) for name in [\'completed.json\',\'validation_probabilities.npz\']}\n        if not required_members<=set(wanted):raise ValueError(\'The analysis-only ZIP cannot support these ablations; use the full backup with all five OOF probability files.\')\n        required=sum(z.getinfo(n).file_size for n in wanted)\n        if shutil.disk_usage(output).free<required+512*1024**2:raise OSError(\'Not enough disk space to extract the required prior artifacts\')\n        for name in wanted:z.extract(name,cached)\n    base.save_json(marker,dict(archive_sha256=archive_sha,extracted_files=wanted))\n    return cached\n\n\ndef validate_previous(previous, assignment, fingerprint, expected=None, require_last=True, expected_epochs=80):\n    identity=json.loads((previous/\'run_config.json\').read_text())\n    if identity[\'source_sha256\']!=base.file_sha(v2.__file__) or identity[\'baseline_helpers_sha256\']!=base.file_sha(base.__file__):\n        raise ValueError(\'Previous source differs from the pinned v2 implementation\')\n    if identity[\'data_fingerprint\']!=fingerprint:raise ValueError(\'Previous run uses different training data/split\')\n    cfg=v2.ConfigV2(**identity[\'config\'])\n    if cfg.epochs!=expected_epochs:raise ValueError(f\'Expected a completed {expected_epochs}-epoch run, found {cfg.epochs}\')\n    records={}\n    for fold in range(5):\n        folder=previous/f\'fold{fold}\'\n        done=json.loads((folder/\'completed.json\').read_text())\n        if done[\'epoch\']!=cfg.epochs or set(done[\'val_ids\'])!={i for i,f in assignment.items() if f==fold}:\n            raise ValueError(f\'Incomplete/wrong split: fold{fold}\')\n        wanted=[\'validation_probabilities.npz\']+([\'last.pt\'] if require_last else [])\n        for name in wanted:\n            if not (folder/name).is_file():raise ValueError(f\'Missing fold{fold}/{name}; use the complete prior output/full backup, not the analysis-only ZIP.\')\n            actual=base.file_sha(folder/name)\n            if actual!=done[\'sha256\'][name]:raise ValueError(f\'Changed previous artifact: fold{fold}/{name}\')\n            if expected is not None and actual!=expected[str(fold)][\'sha256\'][name]:\n                raise ValueError(f\'Previous artifact does not match the returned analysis archive: fold{fold}/{name}\')\n        records[str(fold)]=done\n    return cfg,records\n\n\ndef replay_fold(previous, fold, paths, cfg, device, cache):\n    checkpoint=previous/f\'fold{fold}/last.pt\'\n    key=dict(checkpoint_sha256=base.file_sha(checkpoint),config=asdict(cfg),\n             gpu=torch.cuda.get_device_name(0) if device.type==\'cuda\' else \'cpu\',torch=torch.__version__)\n    marker=cache/f\'fold{fold}_identity.json\';npz=cache/f\'fold{fold}_replay.npz\'\n    if marker.exists() and npz.exists():\n        old=json.loads(marker.read_text())\n        if old[\'identity\']!=key or old[\'npz_sha256\']!=base.file_sha(npz):raise ValueError(\'Replay cache changed; use a new output directory\')\n        with np.load(npz) as arrays:return {k:arrays[k].copy() for k in arrays.files}\n    # Hashes were checked against the known previous run before loading its own resume file.\n    state=torch.load(checkpoint,map_location=\'cpu\',weights_only=False)\n    expected_ids={p.stem for p in paths}\n    if state[\'epoch\']!=cfg.epochs or set(state[\'val_ids\'])!=expected_ids or set(state[\'train_ids\'])&expected_ids:\n        raise ValueError(\'OOF leakage/epoch check failed\')\n    model=v2.ResNet34UNet().to(device);values={}\n    for name,key_name in [(\'replay_ema\',\'ema\'),(\'raw\',\'model\')]:\n        model.load_state_dict(state[key_name]);model.eval()\n        for p in paths:\n            pair=v2.predict_pair(model,p,cfg,device)\n            for mode,prob in pair.items():values[f\'{name}_{mode}_{p.stem}\']=v2.resize_probability(prob)\n    del model,state\n    if device.type==\'cuda\':torch.cuda.empty_cache()\n    cache.mkdir(parents=True,exist_ok=True)\n    np.savez_compressed(npz,**values)\n    base.save_json(marker,dict(identity=key,npz_sha256=base.file_sha(npz)))\n    return values\n\n\ndef run_replay(previous, output, train, labels, assignment, fingerprint, expected=None, raw_replay=True):\n    cfg,records=validate_previous(previous,assignment,fingerprint,expected,raw_replay)\n    truth=targets(labels);cached={};replayed={\'replay_ema\':{},\'raw\':{}};drift=[]\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    if raw_replay and device.type!=\'cuda\':raise RuntimeError(\'Enable GPU for paired EMA/raw replay, or turn RUN_RAW_REPLAY off\')\n    for fold in range(5):\n        val=[p for p in train if assignment[p.stem]==fold]\n        with np.load(previous/f\'fold{fold}/validation_probabilities.npz\') as arrays:\n            for p in val:\n                prob=arrays[f\'d4_{p.stem}\'].copy()\n                if prob.shape!=(512,512) or not np.isfinite(prob).all() or prob.min()<0 or prob.max()>1:raise ValueError(\'Invalid cached OOF probability\')\n                cached[p.stem]=prob\n        if raw_replay:\n            values=replay_fold(previous,fold,val,cfg,device,output/\'replay_cache\')\n            for p in val:\n                for name in replayed:replayed[name][p.stem]=values[f\'{name}_d4_{p.stem}\']\n                delta=np.abs(replayed[\'replay_ema\'][p.stem]-cached[p.stem])\n                drift.append(dict(id=p.stem,fold=fold,mean_absolute_probability_drift=float(delta.mean()),\n                                  max_absolute_probability_drift=float(delta.max())))\n        print(f\'Replay fold{fold} completed; no training\',flush=True)\n    if set(cached)!=set(assignment):raise AssertionError(\'Incomplete OOF coverage\')\n    experiments={\n        \'E1_ema_t04\':{i:p>=0.4 for i,p in cached.items()},\n        \'E1_ema_t01\':{i:p>=0.1 for i,p in cached.items()},\n        \'E1_ema_t04_dilate1\':{i:cv2.dilate((p>=0.4).astype(np.uint8),cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3)))>0 for i,p in cached.items()}}\n    if raw_replay:\n        for name,probs in replayed.items():\n            for t,label in [(0.1,\'t01\'),(0.4,\'t04\')]:experiments[f\'E2_{name}_{label}\']={i:p>=t for i,p in probs.items()}\n    rows=[row for name,masks in experiments.items() for row in evaluate_masks(name,masks,truth,assignment)]\n    summary,folds=summarize_rows(rows)\n    write_table(output/\'replay_per_image.csv\',rows);write_table(output/\'replay_summary.csv\',summary)\n    write_table(output/\'replay_per_fold.csv\',folds);write_table(output/\'ema_replay_drift.csv\',drift)\n    probability_preview({p.stem:p for p in train},truth,cached,output/\'boundary_ablation_preview.png\')\n    base.save_json(output/\'replay_audit.json\',dict(complete_20_image_oof=True,fold_records=records,\n        raw_and_ema_replayed_in_same_environment=raw_replay,previous_probability_hashes_verified=True,\n        training_started=False,competition_csv_generated=False,\n        interpretation=\'Morphology and centerline changes are paired diagnostics; no winner selected from public scores.\'))\n    return summary\n\n\nclass PairedPatches(Dataset):\n    """All label variants receive exactly the same RGB patches and augmentations."""\n    def __init__(self, paths, labels, cfg, fold):\n        self.ids=[p.stem for p in paths];self.cfg=cfg;self.fold=fold\n        self.images=[cv2.resize(v2.read_rgb(p),(512,512),interpolation=cv2.INTER_AREA) for p in paths]\n        self.clean=[base.strict_label(labels[i]).astype(np.float32) for i in self.ids]\n        self.proxy=[base.converter_label(labels[i]).astype(np.float32) for i in self.ids]\n        self.positive=[np.argwhere(c>0) for c in self.clean]\n        self.plans=[]\n\n    def set_epoch(self, epoch):\n        self.plans=[]\n        for index in range(self.cfg.patches_per_epoch):\n            rng=random.Random((self.cfg.seed+1009*self.fold)*10**10+epoch*10**6+index)\n            i=rng.randrange(len(self.ids));patch=self.cfg.patch\n            crop=rng.randint(int(patch*0.9),min(int(patch*1.1),512))\n            if rng.random()<0.3 and len(self.positive[i]):\n                y,x=self.positive[i][rng.randrange(len(self.positive[i]))]\n                top,left=int(np.clip(y-crop//2,0,512-crop)),int(np.clip(x-crop//2,0,512-crop))\n            else:top,left=rng.randint(0,512-crop),rng.randint(0,512-crop)\n            self.plans.append((i,top,left,crop,rng.randrange(4),rng.random()<0.5,\n                rng.uniform(0.85,1.15),rng.uniform(-0.04,0.04),rng.uniform(0.85,1.15),\n                [rng.uniform(0.95,1.05) for _ in range(3)]))\n        return digest_json(self.plans)\n\n    def __len__(self):return self.cfg.patches_per_epoch\n\n    def __getitem__(self,index):\n        i,top,left,crop,rotation,flip,contrast,brightness,gamma,gains=self.plans[index]\n        patch=self.cfg.patch\n        image=cv2.resize(self.images[i][top:top+crop,left:left+crop],(patch,patch),interpolation=cv2.INTER_LINEAR)\n        pair=np.stack([self.clean[i],self.proxy[i]],axis=-1)\n        pair=cv2.resize(pair[top:top+crop,left:left+crop],(patch,patch),interpolation=cv2.INTER_NEAREST)\n        image,pair=base.paired_d4(image,pair,rotation,flip)\n        image=np.clip((image.astype(np.float32)/255-0.5)*contrast+0.5+brightness,0,1)**gamma\n        image=np.clip(image*np.array(gains,np.float32),0,1)\n        return torch.from_numpy(image.transpose(2,0,1).copy()),torch.from_numpy(pair.transpose(2,0,1).copy())\n\n\ndef supervised_loss(outputs, pair, variant, alpha=0.5):\n    c,p=pair[:,0:1],pair[:,1:2]\n    if variant==\'L0_clean\':bce_target,dice_target=c,c\n    elif variant==\'L1_proxy\':bce_target,dice_target=p,p\n    elif variant==\'L2_soft\':bce_target,dice_target=c+alpha*(p-c),c\n    else:raise ValueError(variant)\n    losses=[]\n    for logits in outputs:\n        bt=F.interpolate(bce_target,size=logits.shape[-2:],mode=\'area\')\n        dt=F.interpolate(dice_target,size=logits.shape[-2:],mode=\'area\')\n        prob=logits.float().sigmoid()\n        dice=(2*(prob*dt).sum((1,2,3))+1)/(prob.sum((1,2,3))+dt.sum((1,2,3))+1)\n        losses.append(0.5*F.binary_cross_entropy_with_logits(logits.float(),bt.float())+0.5*(1-dice.mean()))\n    return (losses[0]+0.1*losses[1]+0.2*losses[2])/1.3\n\n\ndef train_label_fold(paths, validation, labels, cfg, fold, variant, folder, initial, device):\n    folder.mkdir(parents=True,exist_ok=True);base.seed_all(cfg.seed+1009*fold)\n    dataset=PairedPatches(paths,labels,cfg,fold)\n    loader=DataLoader(dataset,batch_size=cfg.batch_size,shuffle=False,num_workers=0,pin_memory=device.type==\'cuda\')\n    model=v2.ResNet34UNet(initial).to(device);ema=copy.deepcopy(model).eval().requires_grad_(False)\n    initial_model_hash=digest_json({k:hashlib.sha256(v.detach().cpu().numpy().tobytes()).hexdigest() for k,v in model.state_dict().items()})\n    optimizer=torch.optim.AdamW([{\'params\':model.encoder.parameters(),\'lr\':cfg.encoder_lr},\n        {\'params\':[p for n,p in model.named_parameters() if not n.startswith(\'encoder.\')],\'lr\':cfg.decoder_lr}],weight_decay=cfg.weight_decay)\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=cfg.amp and device.type==\'cuda\')\n    history,first,step=[],1,0;last=folder/\'last.pt\'\n    train_ids,val_ids=[p.stem for p in paths],[p.stem for p in validation]\n    if last.exists():\n        saved=torch.load(last,map_location=\'cpu\',weights_only=False)\n        if saved[\'variant\']!=variant or saved[\'train_ids\']!=train_ids or saved[\'val_ids\']!=val_ids or saved[\'initial_model_hash\']!=initial_model_hash:\n            raise ValueError(\'Label experiment checkpoint mismatch\')\n        model.load_state_dict(saved[\'model\']);ema.load_state_dict(saved[\'ema\'])\n        optimizer.load_state_dict(saved[\'optimizer\']);scaler.load_state_dict(saved[\'scaler\'])\n        first,step,history=saved[\'epoch\']+1,saved[\'step\'],saved[\'history\'];base.restore_rng(saved[\'rng\'])\n        del saved\n        print(f\'{variant}/fold{fold}: resume at {first}/{cfg.epochs}\',flush=True)\n    for epoch in range(first,cfg.epochs+1):\n        begin=time.perf_counter();model.train();plan_hash=dataset.set_epoch(epoch);losses=[]\n        for image,pair in loader:\n            v2.set_lrs(optimizer,step,len(loader),cfg)\n            image,pair=image.to(device,non_blocking=True),pair.to(device,non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type,enabled=cfg.amp and device.type==\'cuda\'):\n                loss=supervised_loss(model(image),pair,variant,cfg.soft_boundary)\n            if not torch.isfinite(loss):raise FloatingPointError(\'Nonfinite label experiment loss\')\n            scaler.scale(loss).backward();scaler.unscale_(optimizer);nn.utils.clip_grad_norm_(model.parameters(),5.0)\n            old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()\n            if scaler.get_scale()>=old_scale:v2.update_ema(ema,model,cfg.ema_decay)\n            losses.append(float(loss.detach()));step+=1\n        monitor_c,monitor_p=None,None\n        if epoch%cfg.monitor_every==0 or epoch==cfg.epochs:\n            scores=[]\n            for path in validation:\n                image=cv2.resize(v2.read_rgb(path),(512,512),interpolation=cv2.INTER_AREA)\n                prob=v2.predict_pair(ema,image,cfg,device,False)[\'none\']\n                scores.append((base.dice_score(prob>=0.5,base.strict_label(labels[path.stem])),\n                               base.dice_score(prob>=0.5,base.converter_label(labels[path.stem]))))\n            monitor_c,monitor_p=map(float,np.mean(scores,axis=0))\n        row=dict(epoch=epoch,loss=float(np.mean(losses)),monitor_clean_05=monitor_c,monitor_proxy_05=monitor_p,\n                 plan_sha256=plan_hash,seconds=round(time.perf_counter()-begin,3))\n        history.append(row)\n        if shutil.disk_usage(folder).free<1200*1024**2:raise OSError(\'Less than 1.2 GiB free; preserve checkpoint and resume after freeing space\')\n        base.atomic_checkpoint(last,dict(model=model.state_dict(),ema=ema.state_dict(),optimizer=optimizer.state_dict(),\n            scaler=scaler.state_dict(),epoch=epoch,step=step,history=history,rng=base.rng_state(),\n            train_ids=train_ids,val_ids=val_ids,variant=variant,initial_model_hash=initial_model_hash))\n        write_table(folder/\'history.csv\',history)\n        print(f\'{variant}/fold{fold} {epoch:03d}/{cfg.epochs} loss={row["loss"]:.4f} seconds={row["seconds"]}\',flush=True)\n    base.atomic_checkpoint(folder/\'final_ema.pt\',dict(model=ema.state_dict(),epoch=cfg.epochs,\n        variant=variant,train_ids=train_ids,val_ids=val_ids,initial_model_hash=initial_model_hash))\n    values={}\n    for path in validation:\n        image=cv2.resize(v2.read_rgb(path),(512,512),interpolation=cv2.INTER_AREA)\n        for mode,prob in v2.predict_pair(ema,image,cfg,device).items():values[f\'{mode}_{path.stem}\']=prob\n    np.savez_compressed(folder/\'validation_probabilities.npz\',**values)\n    files=[\'last.pt\',\'final_ema.pt\',\'history.csv\',\'validation_probabilities.npz\']\n    base.save_json(folder/\'completed.json\',dict(epoch=cfg.epochs,variant=variant,train_ids=train_ids,val_ids=val_ids,\n        initial_model_hash=initial_model_hash,plan_hashes=[r[\'plan_sha256\'] for r in history],\n        sha256={name:base.file_sha(folder/name) for name in files}))\n    del model,ema,optimizer,scaler,dataset,loader\n    if device.type==\'cuda\':torch.cuda.empty_cache()\n\n\ndef aggregate_labels(output, variants, cfg, labels, assignment):\n    complete=[variant for variant in variants if all((output/variant/f\'fold{fold}/completed.json\').exists() for fold in range(5))]\n    truth=targets(labels);all_rows=[];calibrated=[];sweeps=[]\n    for variant in complete:\n        probs={mode:{} for mode in [\'none\',\'d4\']}\n        for fold in range(5):\n            folder=output/variant/f\'fold{fold}\';done=json.loads((folder/\'completed.json\').read_text())\n            if done[\'epoch\']!=cfg.epochs or set(done[\'val_ids\'])!={i for i,f in assignment.items() if f==fold}:raise ValueError(\'Wrong label OOF fold\')\n            for name,digest in done[\'sha256\'].items():\n                if base.file_sha(folder/name)!=digest:raise ValueError(f\'Changed label experiment artifact: {folder/name}\')\n            with np.load(folder/\'validation_probabilities.npz\') as arrays:\n                for mode in probs:\n                    for i in done[\'val_ids\']:probs[mode][i]=arrays[f\'{mode}_{i}\'].copy()\n        if any(set(p)!=set(assignment) for p in probs.values()):raise AssertionError(\'Incomplete label OOF\')\n        for mode,pred in probs.items():\n            for t in [0.1,0.4,0.5]:\n                all_rows+=evaluate_masks(f\'{variant}_{mode}_t{t:g}\',{i:p>=t for i,p in pred.items()},truth,assignment)\n            for target in [\'clean\',\'proxy\']:\n                grid=[]\n                for t in CALIBRATION_GRID:\n                    fold_scores={f:float(np.mean([base.dice_score(pred[i]>=t,truth[i][target]) for i in pred if assignment[i]==f])) for f in range(5)}\n                    row=dict(variant=variant,mode=mode,target=target,threshold=t,mean_dice=float(np.mean(list(fold_scores.values()))),\n                             worst_fold_dice=min(fold_scores.values()),**{f\'fold{f}\':v for f,v in fold_scores.items()})\n                    grid.append(row);sweeps.append(row)\n                best=max(grid,key=lambda r:(r[\'mean_dice\'],-abs(r[\'threshold\']-0.5)))\n                calibrated.append(dict(**best,selection_caveat=\'OOF-tuned development estimate; not nested or independent validation\'))\n    # Pairing is verified even when only some folds/variants are ready.\n    pairing=[]\n    for fold in range(5):\n        records=[]\n        for variant in variants:\n            path=output/variant/f\'fold{fold}/completed.json\'\n            if path.exists():records.append((variant,json.loads(path.read_text())))\n        if len(records)>1:\n            first=records[0][1]\n            for variant,done in records[1:]:\n                if done[\'initial_model_hash\']!=first[\'initial_model_hash\'] or done[\'plan_hashes\']!=first[\'plan_hashes\']:\n                    raise AssertionError(\'Ablation pairing failed: initial weights or sampled RGB patches differ\')\n            pairing.append(dict(fold=fold,variants=[v for v,_ in records],same_initial_weights=True,same_rgb_plan=True))\n    summary,fold_rows=summarize_rows(all_rows)\n    write_table(output/\'label_per_image.csv\',all_rows);write_table(output/\'label_summary.csv\',summary)\n    write_table(output/\'label_per_fold.csv\',fold_rows);write_table(output/\'label_threshold_grid.csv\',sweeps)\n    write_table(output/\'label_calibration_development.csv\',calibrated)\n    base.save_json(output/\'label_audit.json\',dict(complete_variants=complete,requested_variants=list(variants),\n        pairing=pairing,complete_all_requested=len(complete)==len(variants),\n        no_automatic_winner=True,competition_csv_generated=False,public_scores_used=False,\n        calibration_grid=list(CALIBRATION_GRID),geometry=\'RGB/targets both 512; matched across L0/L1/L2\'))\n    return summary\n\n\ndef analysis_archive(output):\n    path=output/\'ablation_analysis_results.zip\';tmp=output/\'ablation_analysis_results.zip.tmp\'\n    files=[p for p in output.rglob(\'*\') if p.is_file() and p.suffix in {\'.json\',\'.csv\',\'.png\',\'.md\',\'.py\'}\n           and \'previous_run_cache\' not in p.parts and p.name!=\'training.lock\']\n    with zipfile.ZipFile(tmp,\'w\',compression=zipfile.ZIP_DEFLATED) as z:\n        for p in files:z.write(p,p.relative_to(output).as_posix())\n    with zipfile.ZipFile(tmp) as z:\n        if z.testzip():raise IOError(\'Analysis ZIP CRC failed\')\n    os.replace(tmp,path)\n    return path\n\n\ndef run_ablation(data_dir=None,output_dir=\'task4_ablation_results\',cfg=None,previous_run=None,\n                 run_inference=True,raw_replay=True,run_training=False,variants=VARIANTS,\n                 folds_to_run=(0,1,2,3,4),weights_path=None,expected_previous=None,resume_from=None):\n    cfg=cfg or AblationConfig()\n    if cfg.patch<64 or cfg.patch%32 or cfg.patch>512 or cfg.soft_boundary!=0.5 or cfg.folds!=5:\n        raise ValueError(\'Require patch multiple of 32 in [64,512], five folds, fixed soft band=0.5\')\n    if cfg.epochs<1 or cfg.batch_size<1 or cfg.patches_per_epoch<1 or cfg.monitor_every<1 or cfg.threads<1:\n        raise ValueError(\'Invalid experiment settings\')\n    if not set(variants)<=set(VARIANTS) or len(set(variants))!=len(variants):raise ValueError(\'Invalid label variants\')\n    if not set(folds_to_run)<=set(range(5)) or len(set(folds_to_run))!=len(folds_to_run):raise ValueError(\'Invalid fold list\')\n    output=Path(output_dir).resolve()\n    if resume_from and not (output/\'ablation_config.json\').exists():\n        source=Path(resume_from)\n        if source.is_dir() and not (source/\'ablation_config.json\').exists():\n            raise ValueError(\'RESUME_ABLATION_FROM must point to this ablation output, not the original v2 run\')\n        if source.is_file():\n            if source.name==\'ablation_analysis_results.zip\':raise ValueError(\'The lightweight analysis ZIP cannot resume training\')\n            with zipfile.ZipFile(source) as z:\n                if \'ablation_config.json\' not in z.namelist():raise ValueError(\'Not an ablation resume archive\')\n        v2.restore_output(source,output)\n    output.mkdir(parents=True,exist_ok=True)\n    torch.set_num_threads(cfg.threads);cv2.setNumThreads(0)\n    root=base.find_data(data_dir)\n    identity=dict(config=asdict(cfg),source_sha256=base.file_sha(__file__),\n        v2_source_sha256=base.file_sha(v2.__file__),base_source_sha256=base.file_sha(base.__file__))\n    path=output/\'ablation_config.json\'\n    previous_identity=json.loads(path.read_text()) if path.exists() else None\n    if previous_identity and any(previous_identity.get(k)!=v for k,v in identity.items()):raise ValueError(\'Changed recipe/source: use a new OUTPUT_DIR\')\n    with v2.run_lock(output):\n        train,_,labels,assignment,_,audit=v2.audit_v2(root,output,cfg)\n        identity[\'data_fingerprint\']=audit[\'data_fingerprint\']\n        if previous_identity and previous_identity!=identity:raise ValueError(\'Dataset changed\')\n        base.save_json(path,identity)\n        sources=output/\'runtime_source\';sources.mkdir(exist_ok=True)\n        for source in [Path(__file__),Path(v2.__file__),Path(base.__file__)]:shutil.copyfile(source,sources/source.name)\n        previous=None;inference_summary=[];label_summary=[]\n        if run_inference:\n            previous=materialize_previous(previous_run,output)\n            inference_summary=run_replay(previous,output,train,labels,assignment,audit[\'data_fingerprint\'],expected_previous,raw_replay)\n        if run_training:\n            device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n            if device.type!=\'cuda\':raise RuntimeError(\'Enable GPU for label-supervision training\')\n            if weights_path is None and previous is not None:\n                candidate=previous/\'pretrained_cache/resnet34-b627a593.pth\'\n                if candidate.exists():weights_path=candidate\n            initial,weight_info=v2.resolve_weights(cfg,output,weights_path)\n            base.save_json(output/\'pretrained_source.json\',weight_info)\n            for variant in variants:\n                for fold in folds_to_run:\n                    folder=output/variant/f\'fold{fold}\'\n                    if (folder/\'completed.json\').exists():\n                        print(f\'{variant}/fold{fold}: completed, reusing\',flush=True);continue\n                    development=[p for p in train if assignment[p.stem]!=fold]\n                    validation=[p for p in train if assignment[p.stem]==fold]\n                    train_label_fold(development,validation,labels,cfg,fold,variant,folder,initial,device)\n            label_summary=aggregate_labels(output,variants,cfg,labels,assignment)\n        base.save_json(output/\'experiment_status.json\',dict(inference_done=run_inference,raw_replayed=run_inference and raw_replay,\n            label_training_requested=run_training,requested_folds=list(folds_to_run),requested_variants=list(variants),\n            competition_csv_generated=False,public_scores_used=False,automatic_model_selection=False,\n            inference_summary=inference_summary,label_summary=label_summary))\n        archive=analysis_archive(output)\n        print(\'OOF-only analysis saved:\',archive,flush=True)\n    return output\n', 'task4_core_band.py': '"""One five-fold experiment: background/core/converter-band softmax segmentation."""\nfrom __future__ import annotations\nimport copy\nimport hashlib\nimport json\nimport os\nimport platform\nimport shutil\nimport time\nimport zipfile\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader\n\nimport task4_controlled_ablation as abl\nbase,v2=abl.base,abl.v2\nMETHOD=\'core_band_3class\'\nLOSS=dict(ce=.5,core_dice=.25,proxy_dice=.25,aux2=.1,aux4=.2,\n          class_order=[\'background\',\'core\',\'band\'],smooth=1.)\n\n\ndef state_hash(model):\n    return abl.digest_json({k:hashlib.sha256(v.detach().cpu().numpy().tobytes()).hexdigest()\n                            for k,v in model.state_dict().items()})\n\n\nclass CoreBandUNet(v2.ResNet34UNet):\n    def __init__(self,initial=None):\n        super().__init__(initial)\n        # Hash before replacing heads binds shared initialization to the old L1.\n        self.reference_binary_initial_hash=state_hash(self)\n        self.head=nn.Conv2d(32,3,1)\n        self.aux2=nn.Conv2d(48,3,1)\n        self.aux4=nn.Conv2d(64,3,1)\n\n\ndef categorical_target(pair):\n    c,p=pair[:,:1].float(),pair[:,1:2].float()\n    if torch.any(c>p+1e-6):raise ValueError(\'Core must be a subset of converter proxy\')\n    return torch.cat([1-p,c,p-c],dim=1)\n\n\ndef probabilities(logits):\n    p=logits.float().softmax(dim=1)\n    return torch.cat([p[:,1:2],p[:,1:2]+p[:,2:3]],dim=1)\n\n\ndef objective(outputs,pair):\n    target=categorical_target(pair)\n    terms=[]\n    for logits in outputs:\n        truth=F.interpolate(target,size=logits.shape[-2:],mode=\'area\')\n        log_p=logits.float().log_softmax(dim=1)\n        ce=-(truth*log_p).sum(1).mean()\n        pp=probabilities(logits)\n        c,p=truth[:,1:2],truth[:,1:2]+truth[:,2:3]\n        def dice_loss(pred,gt):\n            score=(2*(pred*gt).sum((1,2,3))+1)/(pred.sum((1,2,3))+gt.sum((1,2,3))+1)\n            return 1-score.mean()\n        terms.append((ce,dice_loss(pp[:,:1],c),dice_loss(pp[:,1:2],p)))\n    combined=[(terms[0][j]+.1*terms[1][j]+.2*terms[2][j])/1.3 for j in range(3)]\n    loss=.5*combined[0]+.25*combined[1]+.25*combined[2]\n    return loss,combined\n\n\n@torch.inference_mode()\ndef predict(model,image,cfg,device,with_tta=True):\n    model.eval()\n    if isinstance(image,(str,Path)):image=cv2.resize(v2.read_rgb(image),(512,512),interpolation=cv2.INTER_AREA)\n    h,w=image.shape[:2]\n    image=cv2.copyMakeBorder(image,0,max(0,cfg.patch-h),0,max(0,cfg.patch-w),cv2.BORDER_REFLECT_101)\n    height,width=image.shape[:2]\n    coords=[(y,x) for y in v2.window_starts(height,cfg.patch,cfg.patch//2)\n            for x in v2.window_starts(width,cfg.patch,cfg.patch//2)]\n    weight=np.maximum(np.outer(np.hanning(cfg.patch),np.hanning(cfg.patch)),.05).astype(np.float32)\n    accum=np.zeros((2,2,height,width),np.float32);weights=np.zeros((height,width),np.float32)\n    transforms=[(0,False)]+([(r,f) for r in range(4) for f in [False,True] if (r,f)!=(0,False)] if with_tta else [])\n    for start in range(0,len(coords),cfg.batch_size):\n        positions=coords[start:start+cfg.batch_size]\n        batch=np.stack([image[y:y+cfg.patch,x:x+cfg.patch].transpose(2,0,1) for y,x in positions])\n        tensor=torch.from_numpy(batch.copy()).to(device).float()/255\n        plain,total=None,None\n        for rotation,flip in transforms:\n            with torch.autocast(device_type=device.type,enabled=cfg.amp and device.type==\'cuda\'):\n                prob=probabilities(model(v2.transform_d4(tensor,rotation,flip))[0])\n            prob=v2.inverse_d4(prob,rotation,flip)\n            if plain is None:plain=prob;total=prob.clone()\n            else:total.add_(prob)\n        pair=torch.stack([plain,total/len(transforms)]).cpu().numpy()\n        for k,(y,x) in enumerate(positions):\n            accum[:,:,y:y+cfg.patch,x:x+cfg.patch]+=pair[:,k]*weight[None,None]\n            weights[y:y+cfg.patch,x:x+cfg.patch]+=weight\n    result=np.clip(accum/weights[None,None],0,1)\n    if not np.isfinite(result).all():raise FloatingPointError(\'Nonfinite inference\')\n    if np.any(result[:,0]>result[:,1]+1e-6):raise AssertionError(\'Proxy probability smaller than core\')\n    result[:,1]=np.maximum(result[:,0],result[:,1])\n    return {f\'{mode}_{head}\':result[m,b,:h,:w].copy() for m,mode in enumerate([\'none\',\'d4\'])\n            for b,head in enumerate([\'core\',\'proxy\'])}\n\n\ndef train_fold(paths,validation,test,labels,cfg,fold,folder,initial,device,reference):\n    folder.mkdir(parents=True,exist_ok=True)\n    base.seed_all(cfg.seed+1009*fold)\n    dataset=abl.PairedPatches(paths,labels,cfg,fold)\n    loader=DataLoader(dataset,batch_size=cfg.batch_size,shuffle=False,num_workers=0,pin_memory=device.type==\'cuda\')\n    model=CoreBandUNet(initial).to(device)\n    train_ids,val_ids=[p.stem for p in paths],[p.stem for p in validation]\n    if (model.reference_binary_initial_hash!=reference[\'initial_model_hash\']\n            or train_ids!=reference[\'train_ids\'] or val_ids!=reference[\'val_ids\']):\n        raise ValueError(\'Shared initialization/split differs from the paired L1 reference\')\n    init_hash=state_hash(model)\n    ema=copy.deepcopy(model).eval().requires_grad_(False)\n    optimizer=torch.optim.AdamW([{\'params\':model.encoder.parameters(),\'lr\':cfg.encoder_lr},\n        {\'params\':[p for n,p in model.named_parameters() if not n.startswith(\'encoder.\')],\'lr\':cfg.decoder_lr}],weight_decay=cfg.weight_decay)\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=cfg.amp and device.type==\'cuda\')\n    history,first,step=[],1,0;last=folder/\'last.pt\'\n    if last.exists():\n        saved=torch.load(last,map_location=\'cpu\',weights_only=False)\n        if (saved[\'method\']!=METHOD or saved[\'config\']!=asdict(cfg) or saved[\'initial_model_hash\']!=init_hash\n                or saved[\'train_ids\']!=train_ids or saved[\'val_ids\']!=val_ids):raise ValueError(\'Checkpoint mismatch\')\n        model.load_state_dict(saved[\'model\']);ema.load_state_dict(saved[\'ema\'])\n        optimizer.load_state_dict(saved[\'optimizer\']);scaler.load_state_dict(saved[\'scaler\'])\n        history,first,step=saved[\'history\'],saved[\'epoch\']+1,saved[\'step\'];base.restore_rng(saved[\'rng\'])\n        del saved\n        print(f\'{METHOD}/fold{fold}: resume {first}/{cfg.epochs}\',flush=True)\n    for epoch in range(first,cfg.epochs+1):\n        begin=time.perf_counter();model.train();plan=dataset.set_epoch(epoch);values=[]\n        if plan!=reference[\'plan_hashes\'][epoch-1]:raise ValueError(\'RGB sampling/augmentation plan changed\')\n        for image,pair in loader:\n            v2.set_lrs(optimizer,step,len(loader),cfg)\n            image,pair=image.to(device,non_blocking=True),pair.to(device,non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type,enabled=cfg.amp and device.type==\'cuda\'):\n                loss,terms=objective(model(image),pair)\n            if not torch.isfinite(loss):raise FloatingPointError(\'Nonfinite three-class loss\')\n            scaler.scale(loss).backward();scaler.unscale_(optimizer);nn.utils.clip_grad_norm_(model.parameters(),5.)\n            old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()\n            if scaler.get_scale()>=old_scale:v2.update_ema(ema,model,cfg.ema_decay)\n            values.append([float(loss.detach())]+[float(t.detach()) for t in terms]);step+=1\n        monitor_c,monitor_p=None,None\n        if epoch%cfg.monitor_every==0 or epoch==cfg.epochs:\n            scores=[]\n            for path in validation:\n                pred=predict(ema,path,cfg,device,False)\n                scores.append([base.dice_score(pred[\'none_core\']>=.5,base.strict_label(labels[path.stem])),\n                               base.dice_score(pred[\'none_proxy\']>=.5,base.converter_label(labels[path.stem]))])\n            monitor_c,monitor_p=map(float,np.mean(scores,axis=0))\n        means=np.mean(values,axis=0)\n        row=dict(epoch=epoch,loss=float(means[0]),ce=float(means[1]),core_dice_loss=float(means[2]),\n                 proxy_dice_loss=float(means[3]),monitor_core_clean_05=monitor_c,monitor_proxy_proxy_05=monitor_p,\n                 plan_sha256=plan,seconds=round(time.perf_counter()-begin,3))\n        history.append(row)\n        if shutil.disk_usage(folder).free<1200*1024**2:raise OSError(\'Insufficient disk space for checkpoint\')\n        base.atomic_checkpoint(last,dict(model=model.state_dict(),ema=ema.state_dict(),optimizer=optimizer.state_dict(),\n            scaler=scaler.state_dict(),epoch=epoch,step=step,history=history,rng=base.rng_state(),method=METHOD,\n            config=asdict(cfg),train_ids=train_ids,val_ids=val_ids,initial_model_hash=init_hash))\n        abl.write_table(folder/\'history.csv\',history)\n        print(f\'{METHOD}/fold{fold} {epoch:03d}/{cfg.epochs} loss={row["loss"]:.4f} seconds={row["seconds"]}\',flush=True)\n    base.atomic_checkpoint(folder/\'final_ema.pt\',dict(model=ema.state_dict(),epoch=cfg.epochs,method=METHOD,\n        config=asdict(cfg),train_ids=train_ids,val_ids=val_ids,initial_model_hash=init_hash))\n    for name,paths_to_predict in [(\'validation\',validation),(\'test\',test)]:\n        probabilities_={}\n        for path in paths_to_predict:\n            for key,p in predict(ema,path,cfg,device).items():probabilities_[f\'{key}_{path.stem}\']=p\n        np.savez_compressed(folder/f\'{name}_probabilities.npz\',**probabilities_)\n    names=[\'last.pt\',\'final_ema.pt\',\'history.csv\',\'validation_probabilities.npz\',\'test_probabilities.npz\']\n    base.save_json(folder/\'completed.json\',dict(epoch=cfg.epochs,method=METHOD,train_ids=train_ids,val_ids=val_ids,\n        test_ids=[p.stem for p in test],config=asdict(cfg),initial_model_hash=init_hash,\n        reference_binary_initial_hash=model.reference_binary_initial_hash,plan_hashes=[r[\'plan_sha256\'] for r in history],\n        sha256={n:base.file_sha(folder/n) for n in names}))\n    del model,ema,optimizer,scaler,dataset,loader\n    if device.type==\'cuda\':torch.cuda.empty_cache()\n\n\ndef verify_fold(folder,cfg,fold,assignment,reference,test_ids,check_weights=True):\n    done=json.loads((folder/\'completed.json\').read_text())\n    if done[\'method\']!=METHOD or done[\'epoch\']!=cfg.epochs or done[\'config\']!=asdict(cfg):\n        raise ValueError(\'Wrong completed experiment\')\n    for key in [\'train_ids\',\'val_ids\',\'plan_hashes\']:\n        if done[key]!=reference[key]:raise ValueError(f\'Pairing mismatch: {key}\')\n    if done[\'reference_binary_initial_hash\']!=reference[\'initial_model_hash\']:raise ValueError(\'Shared initialization mismatch\')\n    if set(done[\'val_ids\'])!={i for i,f in assignment.items() if f==fold} or done[\'test_ids\']!=test_ids:\n        raise ValueError(\'Wrong OOF/test IDs\')\n    for name,digest in done[\'sha256\'].items():\n        if not check_weights and name.endswith(\'.pt\'):continue\n        if base.file_sha(folder/name)!=digest:raise ValueError(f\'Artifact changed: {folder/name}\')\n    for split,ids in [(\'validation\',done[\'val_ids\']),(\'test\',test_ids)]:\n        with np.load(folder/f\'{split}_probabilities.npz\',allow_pickle=False) as arrays:\n            expected={f\'{mode}_{head}_{i}\' for mode in [\'none\',\'d4\'] for head in [\'core\',\'proxy\'] for i in ids}\n            if set(arrays.files)!=expected:raise ValueError(\'Unexpected probability keys\')\n            for mode in [\'none\',\'d4\']:\n                for i in ids:\n                    c,p=arrays[f\'{mode}_core_{i}\'],arrays[f\'{mode}_proxy_{i}\']\n                    if c.shape!=(512,512) or p.shape!=(512,512) or not np.isfinite(c).all() or not np.isfinite(p).all():\n                        raise ValueError(\'Invalid probability geometry\')\n                    if c.min()<0 or p.max()>1 or np.any(c>p+1e-6):raise ValueError(\'Probability hierarchy violated\')\n    return done\n\n\ndef compare_reference(output,rows,reference_rows):\n    old={(r[\'experiment\'],r[\'target\'],r[\'id\']):r for r in reference_rows}\n    paired=[];summary=[]\n    metrics=[\'dice\',\'precision\',\'recall\',\'centerline_precision\',\'centerline_recall\',\'cldice\',\n             \'foreground_fraction\',\'thin_clean_centerline_recall\']\n    for r in rows:\n        if not r[\'experiment\'].startswith(\'proxy_\'):continue\n        key=(\'L1_proxy_\'+r[\'experiment\'].removeprefix(\'proxy_\'),r[\'target\'],r[\'id\'])\n        before=old[key]\n        for metric in metrics:\n            paired.append(dict(experiment=r[\'experiment\'],target=r[\'target\'],id=r[\'id\'],fold=r[\'fold\'],\n                               metric=metric,delta=float(r[metric])-float(before[metric])))\n    for e,t,k in sorted({(r[\'experiment\'],r[\'target\'],r[\'metric\']) for r in paired}):\n        rs=[r for r in paired if (r[\'experiment\'],r[\'target\'],r[\'metric\'])==(e,t,k)]\n        ds=[r[\'delta\'] for r in rs]\n        fs=[float(np.mean([r[\'delta\'] for r in rs if int(r[\'fold\'])==f])) for f in range(5)]\n        summary.append(dict(experiment=e,target=t,metric=k,mean_delta=float(np.mean(ds)),\n            positive_images=sum(d>0 for d in ds),positive_folds=sum(d>0 for d in fs),\n            **{f\'fold{f}_delta\':d for f,d in enumerate(fs)}))\n    abl.write_table(output/\'paired_proxy_vs_L1_per_image.csv\',paired)\n    abl.write_table(output/\'paired_proxy_vs_L1_summary.csv\',summary)\n\n\ndef aggregate(output,cfg,labels,assignment,reference_records,reference_rows,reference_grid,test_ids,root,export=True):\n    complete=[]\n    for fold in range(5):\n        folder=output/f\'fold{fold}\'\n        if (folder/\'completed.json\').exists():\n            verify_fold(folder,cfg,fold,assignment,reference_records[str(fold)],test_ids,False);complete.append(fold)\n    if complete!=list(range(5)):\n        return dict(complete_20_image_oof=False,complete_folds=complete,summary=[],candidate_csv_generated=False)\n    truth=abl.targets(labels)\n    probs={f\'{mode}_{head}\':{} for mode in [\'none\',\'d4\'] for head in [\'core\',\'proxy\']}\n    test_mean={head:{i:np.zeros((512,512),np.float64) for i in test_ids} for head in [\'core\',\'proxy\']}\n    for fold in range(5):\n        folder=output/f\'fold{fold}\';done=json.loads((folder/\'completed.json\').read_text())\n        with np.load(folder/\'validation_probabilities.npz\',allow_pickle=False) as arrays:\n            for key in probs:\n                for i in done[\'val_ids\']:probs[key][i]=arrays[f\'{key}_{i}\'].copy()\n        with np.load(folder/\'test_probabilities.npz\',allow_pickle=False) as arrays:\n            for head in test_mean:\n                for i in test_ids:test_mean[head][i]+=arrays[f\'d4_{head}_{i}\'].astype(np.float64)/5\n    if any(set(p)!=set(assignment) for p in probs.values()):raise ValueError(\'Incomplete OOF\')\n    rows=[];grid=[];calibration=[]\n    for mode in [\'none\',\'d4\']:\n        for head in [\'core\',\'proxy\']:\n            values=probs[f\'{mode}_{head}\']\n            for t in [.1,.4,.5]:\n                rows+=abl.evaluate_masks(f\'{head}_{mode}_t{t:g}\',{i:p>=t for i,p in values.items()},truth,assignment)\n            for target in [\'clean\',\'proxy\']:\n                group=[]\n                for t in abl.CALIBRATION_GRID:\n                    fs=[float(np.mean([base.dice_score(p>=t,truth[i][target]) for i,p in values.items() if assignment[i]==f])) for f in range(5)]\n                    group.append(dict(head=head,mode=mode,target=target,threshold=t,mean_dice=float(np.mean(fs)),\n                        worst_fold_dice=min(fs),**{f\'fold{f}\':s for f,s in enumerate(fs)}))\n                grid+=group\n                best=max(group,key=lambda r:(r[\'mean_dice\'],-abs(r[\'threshold\']-.5)))\n                calibration.append(dict(**best,caveat=\'OOF-tuned development estimate; not independent or nested validation\'))\n    summary,folds=abl.summarize_rows(rows)\n    for name,rs in [(\'core_band_per_image.csv\',rows),(\'core_band_per_fold.csv\',folds),(\'core_band_summary.csv\',summary),\n                    (\'core_band_threshold_grid.csv\',grid),(\'core_band_calibration_development.csv\',calibration)]:\n        abl.write_table(output/name,rs)\n    compare_reference(output,rows,reference_rows)\n    selected=next(r for r in calibration if r[\'head\']==\'proxy\' and r[\'mode\']==\'d4\' and r[\'target\']==\'proxy\')\n    threshold=selected[\'threshold\']\n    selected_masks={i:p>=threshold for i,p in probs[\'d4_proxy\'].items()}\n    candidate_rows=abl.evaluate_masks(\'proxy_candidate_selected_threshold\',selected_masks,truth,assignment)\n    candidate_summary,_=abl.summarize_rows(candidate_rows)\n    abl.write_table(output/\'candidate_oof_per_image.csv\',candidate_rows)\n    abl.write_table(output/\'candidate_oof_summary.csv\',candidate_summary)\n    mask_root=output/\'oof_binary_masks\'\n    for tag,masks in [(\'core_t05\',{i:p>=.5 for i,p in probs[\'d4_core\'].items()}),\n                      (\'proxy_t05\',{i:p>=.5 for i,p in probs[\'d4_proxy\'].items()}),(\'proxy_candidate\',selected_masks)]:\n        folder=mask_root/tag;folder.mkdir(parents=True,exist_ok=True)\n        for i,m in masks.items():Image.fromarray((m*255).astype(np.uint8)).save(folder/f\'{i}.png\')\n    ref_best=max([r for r in reference_grid if r[\'mode\']==\'d4\' and r[\'target\']==\'proxy\'],\n                 key=lambda r:(float(r[\'mean_dice\']),-abs(float(r[\'threshold\'])-.5)))\n    deltas=[float(selected[f\'fold{f}\'])-float(ref_best[f\'fold{f}\']) for f in range(5)]\n    calibrated_comparison=dict(new_head=\'proxy\',new_threshold=threshold,reference_threshold=float(ref_best[\'threshold\']),\n        mean_delta=selected[\'mean_dice\']-float(ref_best[\'mean_dice\']),positive_folds=sum(d>0 for d in deltas),\n        **{f\'fold{f}_delta\':d for f,d in enumerate(deltas)},\n        caveat=\'Both thresholds selected on reused OOF; descriptive development comparison\')\n    exported=None\n    if export:\n        if any(np.any(test_mean[\'core\'][i]>test_mean[\'proxy\'][i]+1e-6) for i in test_ids):raise AssertionError(\'Ensemble hierarchy violated\')\n        exported=v2.export_csv(test_mean[\'proxy\'],test_ids,threshold,\'submission_core_band_proxy\',root,output)\n        np.savez_compressed(output/\'test_ensemble_probabilities.npz\',**{f\'{h}_{i}\':p.astype(np.float32) for h,vs in test_mean.items() for i,p in vs.items()})\n    status=dict(complete_20_image_oof=True,complete_folds=complete,summary=summary,candidate_oof_summary=candidate_summary,\n        selected_proxy_threshold=threshold,automatic_threshold_selection=True,automatic_final_winner_selection=False,\n        threshold_selection_rule=\'proxy head D4; maximum mean proxy-GT Dice on the predeclared grid; tie nearer 0.5\',\n        calibrated_proxy_comparison=calibrated_comparison,output_head_comparison_warning=\'Core-head C score and proxy-head P score are from different masks.\',\n        shared_initialization_and_rgb_plans_verified=True,probability_hierarchy_verified=True,\n        candidate_csv_generated=exported is not None,exported=exported,test_inference_complete=export,\n        submitted_to_competition=False,public_scores_used=False,additional_model_or_loss_search=False)\n    return status\n\n\ndef package(output):\n    files=[p for p in output.rglob(\'*\') if p.is_file() and p.name!=\'training.lock\']\n    paths={}\n    for kind in [\'analysis\',\'oof\']:\n        path=output/f\'core_band_{kind}.zip\';temporary=path.with_suffix(\'.zip.tmp\')\n        with zipfile.ZipFile(temporary,\'w\',compression=zipfile.ZIP_DEFLATED) as z:\n            for p in files:\n                include=p.suffix in {\'.json\',\'.csv\',\'.png\',\'.md\',\'.py\'}\n                if kind==\'oof\' and p.name==\'validation_probabilities.npz\':include=True\n                if include:z.write(p,p.relative_to(output).as_posix())\n        with zipfile.ZipFile(temporary) as z:\n            if z.testzip() is not None:raise IOError(\'ZIP CRC failed\')\n        os.replace(temporary,path);paths[kind]=str(path)\n    if (output/\'submission_core_band_proxy.csv\').exists():\n        print(\'COMPLETE: download CSV + core_band_analysis.zip only. Model checkpoints stay on Kaggle.\',flush=True)\n    else:\n        print(\'PARTIAL checkpoint package only; wait for all five folds. No candidate CSV yet.\',flush=True)\n    return paths\n\n\ndef restore(source,output):\n    source=Path(source).resolve()\n    if source==output:return\n    if not source.is_dir() or not (source/\'core_band_config.json\').exists():\n        raise ValueError(\'RESUME_FROM must be this experiment\\\'s full mounted Kaggle Output folder, not a lightweight ZIP\')\n    if output.is_relative_to(source):raise ValueError(\'Resume target cannot be inside source\')\n    if output.exists() and any(output.iterdir()):raise ValueError(\'Use an empty resume destination\')\n    if not any(source.glob(\'fold*/last.pt\')):raise ValueError(\'No resumable checkpoints in source\')\n    shutil.copytree(source,output,ignore=shutil.ignore_patterns(\'*.zip\',\'*.tmp\',\'training.lock\'))\n\n\ndef run(data_dir,output_dir,cfg,reference_records,reference_rows,reference_grid,reference_info,\n        weights_path=None,folds_to_run=(0,1,2,3,4),resume_from=None,on_package=None,engineering=False):\n    if not engineering and asdict(cfg)!=reference_info[\'config\']:\n        raise ValueError(\'Keep the paired seed42/final80 L1 recipe\')\n    if len(set(folds_to_run))!=len(folds_to_run) or not set(folds_to_run)<=set(range(5)):raise ValueError(\'Invalid folds\')\n    output=Path(output_dir).resolve()\n    if resume_from and not (output/\'core_band_config.json\').exists():restore(resume_from,output)\n    output.mkdir(parents=True,exist_ok=True)\n    identity=dict(config=asdict(cfg),loss=LOSS,source_sha256=base.file_sha(__file__),\n        helper_source_sha256={Path(p).name:base.file_sha(p) for p in [abl.__file__,v2.__file__,base.__file__]},\n        reference_info=reference_info,reference_records_hash=abl.digest_json(reference_records),\n        reference_rows_hash=abl.digest_json(reference_rows),reference_grid_hash=abl.digest_json(reference_grid),engineering_only=engineering)\n    marker=output/\'core_band_config.json\';previous=json.loads(marker.read_text()) if marker.exists() else None\n    if previous and any(previous.get(k)!=v for k,v in identity.items()):raise ValueError(\'Changed config/source/reference: use a new output directory\')\n    torch.set_num_threads(cfg.threads);cv2.setNumThreads(0)\n    with v2.run_lock(output):\n        root=base.find_data(data_dir)\n        train,test,labels,assignment,ids,audit=v2.audit_v2(root,output,cfg)\n        identity[\'data_fingerprint\']=audit[\'data_fingerprint\']\n        if audit[\'data_fingerprint\']!=reference_info[\'data_fingerprint\']:raise ValueError(\'Dataset differs from paired reference\')\n        if previous and previous!=identity:raise ValueError(\'Dataset changed\')\n        base.save_json(marker,identity)\n        import PIL,skimage\n        current_env=dict(python=platform.python_version(),torch=str(torch.__version__),\n            torchvision=str(v2.torchvision.__version__),numpy=np.__version__,opencv=cv2.__version__,\n            pillow=PIL.__version__,skimage=skimage.__version__,gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)\n        base.save_json(output/\'execution_environment.json\',current_env)\n        base.save_json(output/\'reference_info.json\',reference_info)\n        abl.write_table(output/\'reference_L1_per_image.csv\',reference_rows)\n        abl.write_table(output/\'reference_L1_threshold_grid.csv\',reference_grid)\n        sources=output/\'runtime_source\';sources.mkdir(exist_ok=True)\n        for path in [__file__,abl.__file__,v2.__file__,base.__file__]:shutil.copyfile(path,sources/Path(path).name)\n        device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        if device.type!=\'cuda\':raise RuntimeError(\'Enable GPU for this one five-fold experiment\')\n        if weights_path is None:\n            candidate=Path(\'/kaggle/working/task4_recovery/pretrained_cache/resnet34-b627a593.pth\')\n            if candidate.exists():weights_path=candidate\n        initial,provenance=v2.resolve_weights(cfg,output,weights_path)\n        if provenance[\'sha256\']!=\'b627a593bcbe140c234610266fe4f8ae95ea42fc881d091c9b6052e6b1d0590f\':\n            raise ValueError(\'Wrong generic ImageNet initializer\')\n        base.save_json(output/\'pretrained_source.json\',provenance)\n        for fold in folds_to_run:\n            folder=output/f\'fold{fold}\'\n            if (folder/\'completed.json\').exists():\n                verify_fold(folder,cfg,fold,assignment,reference_records[str(fold)],ids)\n                print(f\'{METHOD}/fold{fold}: verified complete, reusing\',flush=True)\n                continue\n            dev=[p for p in train if assignment[p.stem]!=fold];val=[p for p in train if assignment[p.stem]==fold]\n            train_fold(dev,val,test,labels,cfg,fold,folder,initial,device,reference_records[str(fold)])\n            verify_fold(folder,cfg,fold,assignment,reference_records[str(fold)],ids)\n            base.save_json(output/\'core_band_status.json\',dict(complete_20_image_oof=False,\n                completed_requested_folds=[f for f in range(5) if (output/f\'fold{f}/completed.json\').exists()],\n                summary=[],engineering_only=engineering,candidate_csv_generated=False,public_scores_used=False))\n            paths=package(output)\n            if on_package is not None:on_package(output,paths)\n        status=aggregate(output,cfg,labels,assignment,reference_records,reference_rows,reference_grid,ids,root)\n        status.update(engineering_only=engineering,model_weights_local_download_required=False,\n            environment_differences={k:dict(reference=v,current=current_env.get(k)) for k,v in reference_info.get(\'environment\',{}).items() if current_env.get(k)!=v})\n        base.save_json(output/\'core_band_status.json\',status)\n        paths=package(output)\n        if on_package is not None:on_package(output,paths)\n    return output\n'}
REFERENCE_RECORDS={'0': {'epoch': 80, 'variant': 'L1_proxy', 'train_ids': ['1', '3', '4', '5', '7', '8', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20'], 'val_ids': ['2', '6', '9', '10'], 'initial_model_hash': '554e9ecba699a48f180c59f5dd7dbb998e1a9a24e21b13684d40192494495172', 'plan_hashes': ['242132163dc7dc6439a5f4ac1728a54454b7fc46202b4403f18319a46196eb1d', 'b7e9fc189550613fe3c976ed4b8a2328ec95e4d362ccac11569b0bfc86108134', '5ee347a5cca68958928e62d92883861ae2150e0b03057bff56e314a3760e6757', '90ecf03fbfae9c4a12ddfcf8d7e8ef1c3771706f0eb082eaa5b8e28759e8c4b7', '42b0a72173d6171ce5bb573c960e27df39156937e4da70b8e6b11d2680867cc0', '2aeb8a559966023a7c59784c87eb6d3d7385d4181e90383fa580e79b08acf5c3', '4fb00eff521d20f113edaf159fdb2c67b4c273ffaed15b1e1ce1d2400103bc66', 'f962c6b2ddb9c01ba1121f50343cc28ed8f473f97b3c8f8bb42bfe662c908ed8', 'a7ece28bdce15d69b7440f8ba80bdd28c3088bce53fd3618b578ec5ab0815272', 'e40d633cffe41509c03296db73ba2ce9a217b80206fbda45a8c86718415b1d83', '340940a3b3cf4fdb7790a997fcb7dc426f95da47ce46398415da543e0d0db1ef', '18ef8ba96305100fbf5e933431b1ea286ff00acbc7e50ec769170be050b5d0fe', 'e87dbf1b20e01be162457ec8accae7e706c199fb6af6b4693aefd6d6b5fe7484', '9fbae9a9e0df5ba948357bbb0990da7f74c2b0fee6ec5b8aa92ad1756b73c69d', 'ae3dc00ce1420c373428ff9e3357eee55ddcb648a8e2eb685ee5e207acacd381', 'c2c8e09f664ed532fcfeabfed957a76586b780523fe0ef53912091134b0c1abe', '51c7e9a4eab1dab4335c3af50dddc142639ad34672c19bd60027dde9eb2daacd', '3f7e1bf10a3f17d084d7efc66584c05788b9f6f99bb306f8e17db67a5e9df8a1', 'd5c048a299397d6f22557ba851e3a5ca5959bb14acbd499eac55c1f62889bacc', '920b03ff794873f2a745d84388f9337a8e51f2901dd18c79faf7bf0290c76586', '34e3ea1d6e9f57a7122c9e43168367d5b3a54f474337127413f4803ef01eb962', '94ffaee6dcdf1794bf118ed31d52c962158a41a54677ee0a95f6b02172a0cba2', '86bf84c34952ad8ab2510f7d5e103efa97642c88b83f60520256e9b33b7419ba', '4131b964e0080fbccaf4d345b6e2bd52206057044e2fb54c3272a94cf0fba923', 'efd66094c31bf162efbf6a85fff421c5ea5507f077f4a2ec11a890457ddd8c40', 'fbb119259a476094f998d424f256c6b151de856427cd4e632cec4cc69f4eeca1', 'c10e3efdca0e96421345c137502eb970965b18c9aa8d38216c3e23dad8c88917', '8fbeb1269c77afdf971bd54609f323d529b4d84edefaf0b6e654e0951dd8f817', '2c3d25529f682fda51a5e0b34747767b6b5f839342dfbbc402e6461b445c4aae', 'c9fe0c417660f49fa0d719adea78b29e377f77539ca3d742e15c24e2f7da14cd', '4a3ec9d6e65c822c2b7b8baa2b6fda4a29f8f45f0f0ff45211e134e4ae8c0902', '60ffa348398aa6a2888cd796e10c398c39bf0554717969280bdbbc06dedb208c', '34ff2734ab5f1c336f89d53ddddccfb1f22bac63bfdec843c0c083fbf43331e7', '874eb170e66ab11968e057357664284af2252d3699baa2ea8668644d3cebc122', 'b30d2ed94aa6fde7cf751f75d91cd8bb132fb4f15d3495ac54276e4413672a0b', '2fb3b866729f98120bb0f254143cfb991fc31cc8a49f54aebfa56a65599e7e91', '0860fec0c1b2fa6bb8edc191841ba577bb4ca3f5bbac3e2ae41ad765f0e8a55a', '03a936d3d6c27d73d1bea945b5a807e7338387e9e436ec5feae1f6c550f00dc5', '40dc73333098f7a9b8f1b2bf95284c6dea62490cd8056d74e687f7414995c1ce', 'a59259a07526cb4ee7f8ef841649b49635cb4db0feea80b55ec1f76839566d31', '194836dc10e2fb7de79ef42652e03956e4b8316d43de1de71585ff01c6f628a1', 'd7db719738a8581c97672c72f66e19a9e743c04529e6d0dfade7302e46195a7d', '33c7679bc09370e5af7a477d45073729dfa34850b5f97b65ccd95f1ae1243049', '04f278bf55b97ff603477dfc463c0bcb8899f82e1144dceb053b2ce98330aa0a', 'e234e7457a9be1f140302820024f33520b3af2191413ecf53d9e34bcb92c3883', '3da5fd9fd2bd41c192d4a6e07a7a27cab202133b6fd322b5f07e68759722358e', 'bfbc208b7f05c828c37f266956d52fc2ffcbc2991144e278f0f5677d975be49b', '423967fbd9fde6e276afb6178b30318b59e079eb981c53eea01249fd42c075ab', '809a5b5e1ea936fd7688253ec57a82cec5370357c6d6ce3f3f8927d16805df67', 'c1da6f70b2c4e1b494f2cf58b711913d617e8b65f39bf2ab9f147d35ee9c1194', 'd3e022f90ed51962c8affeb742ed6a70e11de3666cce772c3e0d8bf092954827', '37de2378bfb5ce5db9ddd3f9253c8e42bda5e61a8f0658660045fbe7a75cb213', '43253c142e3858c2ec81ea58957ba67e595e4c6a32286f9758ac1d7415a21247', '78a0b40a78ba6c1c4c8776966bb1ae0757c803a4abc5ba903321490acf2363b9', 'e8bdde6a690ece792dabd2cc5cbae8612a70bb29a13ba632188487af103dae19', '172a6bff04b0f806dc7035397b12e9d9203f59bfca6425b2850307f755370a67', '7c2840869466e70bf334380a90c3c8dc8b0dfe2f862ee288e5166edd665cd589', '8c9fca6d32b7a09c3701f05e4a955cfd4429bcec8a0ebab65244f6387fe445aa', 'f665c155cbe1579054f4c51ccd1f15d38cd78cdbce9b883a976593cf3020f964', 'f7f4c0bbd888c6f828e1bdb1daf35c5f4f2a3231d25958c973cc73eb1f1b10c7', '6adef9920b8a2c87f2f7fef3d1a5b3a6b3064e62846978fb488e880c2ae56c29', '0a2a02ebe1a5eb50c47dc7a2dd0fbb3fc4fb8b1e66b095ad87c0c35fb8d4e09f', 'def5b202586579806f84212ea09a536d3aad0aa2e0a9d6c5147f02ea4b26aa8d', '58b2b41d79bca0ef3937ca0239fa8f9a3f4e6f9b01dd28e38cfe5506efd031a3', '5b43418ea5f0d9dcae655bf149db2f6f638f67e3ac8379f6bed0edea31fdb456', '1e8b61a43396869a42d72c0dd0f7b9e0856fb32f1fc1ac5b9d931255de12b7c3', '0932a7b36c82e0f075c08e43dbdc51a70ffdc866a911563c99728265d744ebdb', 'bea97340820a0d1989cbecfc19f402a4a42f361c0a7079ffd0e8b74843155e87', '45d8e1f16530bf82eac6c4727b9ce0111feb1a3cfb5889622cd45d56d4d90064', '2cf2af5e42a47a75a48c119751fd932f3cbe6dd109f405842503799352642307', 'c4537a7dcdd21413c19e467b6e2d7d50589e8f332b58e8cb27ac2d3241504c0b', '35ec3145f111cdceb0f6a4ff26ba597b499ce8b1b7201897310d7cef257a1031', '8b43eef27448d5216750440ce377a064bf94cba4d50847d6141742fd3d534bf8', '6719fb66607dacbcc8bc6d555345054a45eeacf9f24e831c0fe1b5e65fe728ae', '5d0ff59383857dab0d8c63b550ef652a8896adfb2d9d2556af1ffafedecbec38', '57a1a1baac01dafd7e2c9003158366b8cbf293c080a93d2ebbb9b31c51f89c09', 'b181619045fc9dd5646b515802442c81bb7896d64a2a40eca163ff81c44551be', 'c029f8c7f5cbe900b2107505eee4e37a3f58541e3a6525b6488446f3b16a8ae5', '0880587a0f435d163cb4910835ad44cf32cf2f40b2fd770bbed3e24379c20d02', '81bae5930b366991e5b91e48c575969e6165c170e3b7fe5a85c83bafe0bf29b0'], 'sha256': {'last.pt': '623d4f528d568af00f432f4c07ed1f5a85b22993c3de844cb0cd8314db2713ee', 'final_ema.pt': '1c56fd9e4d7d25f59d6d5bf9dd014d2eab64d4617a4809d4bd898aa2ec42fb02', 'history.csv': '1d372d97b50eeba962f78b1e3bba996ee386d31831a1a1bb0e565342464d524c', 'validation_probabilities.npz': '1b74d8254632f851471e5cd5865afb67b4d1414fb4bab7bb17619adb721ad504'}}, '1': {'epoch': 80, 'variant': 'L1_proxy', 'train_ids': ['1', '2', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '15', '16', '18', '19'], 'val_ids': ['3', '14', '17', '20'], 'initial_model_hash': 'de11a410772039ba0b124d230bab99f122be7de14c073e5f90f98b59dd59b695', 'plan_hashes': ['c2fe080e2fcdab031905af5dc0fd6059d93ca74a8af01fa54002eeeab1bcef8c', 'e5f970fb2d8f41a826a7f102fcd9bd03cc5bf597156b22f1c972ce6de6d0a57a', '9750469041100b41b9a530581587c8359c9a31a7c568701c6353270925091780', '33a0f60d0f0e0a3836627c307e038693eda3399c5be0e5e7c33bdbc5ea2f79ee', 'b5ce08802371411cb8da7357d80aefeb9ab5c6a4ec80e6b278fb98e51f3457aa', '5ca360cf09ee38e2cf59a9e3a15b84b15156dcd4e5e5647fe2e052f3f7c88a4f', '9554e0f5c7390a7676aaf3f9619830e95d934b6eafa6ce9622486582d5e84da3', 'c87a21a82776c40195c3075133a7cf3ee231450b40e67e40718276fbec49693c', 'ccc9e0155ada69edff16d0a1d935fb94087c518d50c83af6c78313cd0e5b1887', '13ce5f5a75a639c792e021bbdc1206bdd838cb382f61bf04998b9a9e0c47c6eb', 'dbc21728c975d76f84f408eb2edffc125b810a3adb40b105a0e53a1c5849b6f1', '04a01f500d0a9747501b087f6cb7a90efda720943c42c551d10200d17ecc490e', 'ca1726c891370a6d763414131063de9927ccea456fe08a3fea35d5b2c2ce7a73', '6c4cf9dc4cfc166db48f496309309910a01e9cd41ad3c9b1cf00019f8ddbe485', '45b9fd007d47c5d6166fb2790c894b81c94ccd2c9f91900b0c84622a63e3959c', '54a1f9aeaeb56dcced4c7577231806684d59ffb7dbd2aa095b5440ed45d21065', '7bdd33b2150876bb3929ea6772c0a226be5e3b076f854597b8244671a552ecfd', 'eea186cca6de68e34550b1e5753acf55f6fbe6f00ad0c83339483f07948dc6de', 'fbbf92e6e22c98b5dd01553a622861009963c224b111f781cb227cc555e34b85', 'f1626fddf7db898d7b9bcf38dafd3316a3eabd0724b3cd66e7046b4c8afd4589', 'becd9d1960eaab45e694221cae1d6f21f4ca503673ec83d82946e18851e61182', '62d7cd6437e562895d151d74d9ff27e48bd7dbafca6657035bc5c9c4f43f467d', 'b2bbb490f178a6d4096fcda4e74de13b6699e450b61d258f234eeaae0d6b51f4', 'f32b871ae3e91eece1c0a49d422038c32ccbf99d7b9af722cb12e3f5400040e2', 'e34190e88a1b2cde8528ab72af509c3d701bb511bf2fcf536e3fa29ae280088e', '8dd4c22a304583ab460b6934054c11dfcd87e1bf7c14d2710cfc55fb6c7fb8e7', 'bb9f4eb9cc43efd7056be575594267027a8a1f9b14978239079aaf831541376d', '9f8f5300d65aa4f8852cc50a28ac66264596cecd28a510d8d51ac6f029c6ebf2', '423feb5db827f43539930e1e3b2e96895de2cba88f46512b5d4be67cc1f7d8ef', '004acf252284cf62968f098892fb2f6ddfb53874b3b163d3639baba1a0b9d57a', 'd679cbafc17a858df0a94b31d5b32729820dc381ab4472a52a5f2c1f68c26caf', 'd832de421d9e87fd9cb8ec4ce9edf0a77d2d600014c54f561c483127e51b1c50', '2105eb2e732b156bf893d9da4abacfefe128a7382d5d9b9cb569afa8b989810b', 'dea53cc49113e12dfaec52aac82f4265851a7d6a75cf15ef3423549cf3e059b3', '90252974060ad9a7ee8a3a2888c1536cca17fc1149002d69227e416e45b48a06', '08d9bf9a76ff00ab0190944320961cb7d513787082d53d41db2788c415e4005b', '8662037bb4f695cd06b1b0f9f2bec18340d071b70a642a66f7dfb94b550c14f9', 'bfa0040f0024a98e0a9e4e3ed92d79f48d75063e5ca0fb68860ab69845cee279', 'cf1c98bc1324f49d34fb14241fdb45525eb289d88ba9d6680c146fd3fdd9ea42', '5277cd3d9b9904d24fe75421e68fe7db683828bb12a70f5dd6a8018fe8dabecf', '0793510783ccbcb15964532989835e6dc2e9d588140d6455180f84ba55fa1165', '49b67b75c15614c87fea5620e7b11294ff8710c91e4e0b51e5b120a1c1303a5f', '5221eb046bc31a9ee67dbc2a040bbf73a3e4f01b8f2eca071d0827c6bf75d523', '6325c3fdb4587bd3a6b5166f5e6e538654318afa4177afc550aad9903b7ee9dd', 'cfaa05d6234b70cf7978f919cc3f14a77bcc5206b536323900b2d6e6ea581274', '6bcacbfa5bd20531f456bd3915436e2ea75de6b24353180c69db2afc8e31e223', '8abb18ad57a6940147990bb0e200af810053821d38e38602e5460bf189dc0947', '89c140e63d89e7c824771029668af1affc43547e2fc97fcac258f4dfdf1ba612', '70b1ff2c69f99198bb41c72e263b8afb936bbbe3552378c9fb9363900ae04ac4', 'a66c046476db6e98b8bd0426db0a8d9049d485e5b0c3fa92820e836c677221ec', '15d81d90bbdae536e1bbecc6d32e183ab7125d12042b823a75836f155ca58e2c', 'a2dc4d08bb2faaf585d2dbfb9a023007f1f1cd51db062c5dcf9eb6613ef91115', 'a209f0558038fb79656f1b89e2afdaa30ed14d34bc403c4f3a3ccc47844e6d77', 'e107d7d56abdf4224d9302913a81f10b922b48db99d3825d867952b8c9a076b3', '0e711ba5a47417ea46f815d0ad964618eb70de1222ba54ba7262a6b44415e8d0', '787d72ced217d3ab3d70d30503cc270bbc1345593153d3a15f7a8c5b7434114b', '53605ad078384fba433f6d8f4c9876d2451d691027d7149b40944ba9282c3e1a', '9394f08929ca64bef43ac34776f818be47852b02404dbd2b1f184f0f7e873c71', '93b0f5ed8895e85fbc9ab36032a38ca151ddcbff733effe6a130c55e1ac272e4', '1003d4197cf4ac97eb0ff6bd29b335f54e1d31d7fad1958983e2b3782d263110', '5ccde161dd05fe0ec845e7460a5920cc3c4ce5cfeaf892e0f9d72d65afed1618', '8489206b09ab80a9092667010c3c9576177ba99695d3c4187770012be17a8ea2', '1ae60d84dce8076a2f83818aeab39506c25f98f0a0c6a0b8114279c0bf8585c9', 'c75b507f9607ea92265303487be09e399aba9aa53f348ba38b9587739057efaf', 'f6990f6ad8a6e999edb8e8d361bd193f7c11365c9037c2ae4e4bd7240b4334fc', 'a17b503a4aadf9190d47f7cc21e89e139114dece1e3149fe32a70c4c85b5d024', 'b0bbe6947e159db6d0a3f186ad96232f6d9d572f438835c7b4f0433530a80073', '3574395c1def98963945a0daa37fa29bd43d9c3cba977f24f59bb111ba03fa4c', 'b256da49a287f36c82af1a72cc0113cd61bf6c470a71cdbc096dea87f0b897b3', 'b03c1983df524a12a41f1a0b4693c0d8ba1b55fb6ea52f59e266b96392310ea4', 'ed6eb49f7060851c199f80e0e8c81bdf94ac472f5ff36eb77617a2e77dca00ff', '314e9cdb25c516d5416a428290a43609f09c0fa58352e4e160a48dab2a817f0a', 'e9f7cc7c3a0389bd9dfd83611834fd449af15bdaca08d6d80bf2f07615af55fb', 'b55050c86a34653fda593d756cab60b695a0dc123f2214f0d163d292e6ec4687', '28a21c3172478f7f614594684b7c4091b857f2cbf17794fe73226cdb3635b1d9', 'e5ac474a91308144187360f612dd5b9de1c321e958b13455a23ac60e76752756', '25820fef80e376995f39bb0d99fb63fe7637a6fb81f86b73d0a1ec1fa54d2a0b', '0a3a292dcda9303e9a6fc50806be2e9aa31427899f601e35481cc718922e773f', 'fa33ca5b31553887cbb6e3b45197c9c3e02c78846c561e1164fd539e6cdb6446', 'f40a52cd29fa4de3b080e0ab4afa3d93b4a442892dc123c30015529130cf7d88'], 'sha256': {'last.pt': '4d55cef2037398a1d2709b639d74a1d2c1a1593278a0902b6b9a14875c51fe0b', 'final_ema.pt': '43728e1e8200cbcda6feb1698287d85203e5dcd588872730d6873dd89f1ba9d4', 'history.csv': '6c2ba71cc822ba235be69c6563344aea81d4c02396f64bfdd45a00a38f7b1dfa', 'validation_probabilities.npz': 'a0d89ed42b0a073db0b600dd691559c6cef83b26d06664cc823f077c37c84afc'}}, '2': {'epoch': 80, 'variant': 'L1_proxy', 'train_ids': ['2', '3', '6', '7', '8', '9', '10', '12', '13', '14', '15', '16', '17', '18', '19', '20'], 'val_ids': ['1', '4', '5', '11'], 'initial_model_hash': '930a0e666a716697967ef01373c993cee807b0ccaaad3ac7d4fbb40115397000', 'plan_hashes': ['1244c68ba10fb206c9998ec3d215203733484d5544b235b33bbfdb0b7058de6c', 'dee06c40ffcf73af9ce876f0f937a2c4a890fd01ddade24ae74cca73ee48a69f', '0c0602df343625ff527f94c710bb1d9f3c386a753580405635f15aa142d866a8', '0f8f3dec7ba9681183a3611df435805cf7d66324986e10cb7ddf8c305d6eb611', 'f80877e7dedde111d6a05a3d327efd543d908c5ad1181b40162e35911dbeb833', '250c9aaa1e47d63db22ef63f9326f59ac71a42ec3b1ef71d90d0178ed3e72590', '1c04aaf2f1e06242bdf72681560ecd9edd4d031f33ffbbbde5fdab203a0ebc3b', '7c3783104de2cdca06289dcd45c6422756b407f673e13bb562fa6fa13f171e7d', 'cd69be5cba742c86844622f6bf3aa69d415e49fb90521be7eede082072002c5b', 'ba18fb4b42433c59fad02211720cf4326f95b705fdb4c2f929099a034e9e53e0', 'c3e5712693f958f6b7778d8d5ce50b307f158b2769a70c50232e0e4b9581e998', '9329bda092650a7012dae0e04cec4f961238c4a30665ecd80669aafaee66f24c', '52d9afa4aad8d7fcbfccdc79105559b820033c87a641f99407e22f5382d2d8f0', '20f2650e67a3c4003759ade9c6966f5a1dd8c72b3ce61680ddf458a755decafc', 'de0a6202106ad28ab73e2bb563220f561a4eb67412c02e9c88ad9d2f37d45842', '2cbfe6a284f2b19d99b2af75106f32fe65127db388869c41441cc367b5f4a849', 'f43c989f8d119bd3626fa57e1ee7621918336971f8d34d5db86729f56a48d527', '4e82d334dac1e868dcacc2a7c9b7b785822957cb4b2b1ba489108c893e7d8448', 'b74c7485a2975984bdb720b56d129094ec832cd22d52ccc3a3e1033fc3852589', '9bb0a01fc960b93ed73a6e06657f0beb357a8251a39b2a396c27ba4a7ad06300', 'd22e1fc2898428fe32abee0c228a381dad8528f1e544a78f6c60f748a4c54f75', '9505e4c75b146e9042a75fb3d26237b34c1cd6571b1f24d62740fabcd6ccd0f5', 'cde8ab5b81253abe08c41992758241823679e611bdc38556b8cd0c4595c4243b', 'c75d7a2aaf5bbfb6ab709d3249cddab682ba75fe5177389ebaca585b9eda77ea', 'd2c7821f0ece5c70494389800d086636d71a37b58ea75cfc7a360a9b38329011', '7cc000325bf23aff366c1327796229ffd1ae666181665c4ae35acfe7c3dbbb83', 'eff7f762f94d1c2f12886abcb462251d022b2b0ef843510f28ce9099ac4fca3a', 'e9f26a144c4b7ea32d4a43ca9927b95facd6de9777c24f8408fb0b8381e63c50', 'eb0157c243c0f74518833e6393e60023ea4ec42f32c57fcda83b965a8a728dbe', '51eca664f273ac664bfdddb5df4c6ac1337d5f5f6bee06c010ba211c61d9bd5c', '84ed65d82a4e0d6621c909ccc593da7d3e512c1b9d136558e79594e5dbc29c31', '2cdcda523f7840c7d5a7dd4d0ee717ca2bffed6caadd521f504fc0a1f16d83fb', '9dac9634bb23588f79c819783d5e5e757803a1431d927f746512ed52a184ee00', 'df62e98679f0e29a81d3d93790fdbd2223b3f4444b1360c1da83f104e2ada19f', 'd3a874aa38faea354a90979f346b0d64d5506b68c677cb13b0c147cd4a302fc7', 'e8c3c7932561e3ac85c6fcc768a8a1255b6e8dfd091b27111f16092a759b71c0', 'd739b90e5a5b3650cb36982fa46559f7e3061816bb873a5e4c83ac7eff73b73a', '59ac84b4a0a66dae4ee7d3647d15673f93be59c69028ab7064fabadabca9cde7', '9b766626ff9f34ed023abf626c4196ebd2ea781ff46dbe78d6507e0b4344b104', 'edc3ffb1b563394b12a1390eabf17fae63486c6592739d60fc95690bd2a1cd32', 'c66409df7208be6a1c375bbb82d6d4619e9ae2ea068aac802464ae66e338a094', '63675031934d117073d0828dbe5fe85638e65a03887047777209bf3d4e83c4f3', '3f75e4d55e66bff043f6dd1ec28fc8951e37aeb60c38f1cfc5f90eb9925ed33e', '0678322d25e45436fbdd0278cb6706517c5a00e26bba28fa97552fc05ae1880c', '7ba342ed9025fa80235f6ee1f347654660c1c7dbe55c6a3818fb798456a75154', 'bb967e09dfdf1098a184d96543cac6565df220337a68469cd985aa51b4a9345f', 'bf7ee1eaf5ab4491e95347f7ac92ea017c51c101a524f7bc96505d219c1cad5d', '2061c865594e5da2c3886821d19d7af304d5bec03bda29175d37a6b3a2d6160d', '61b07f5a04577981bf138576a109e948e514637deb0c176109999acbff6410bb', 'b92833dc28a61db3302bbdde08c28dd6bdf3bd25d838666ae76e13fc20d94937', 'df73e137ef6f8aae81bc2e41e090b29232123d45ed06391287904f66e8348710', '19fb045acc7492d472e9b11e982c01449b038a651c6eb1d7b003a60bea7df2c9', 'ae090a8f93968acee5af30b8cabf0de53b5310593e1c6c70904f89a82248c604', '083a3c53346d0ea1ee2628e2e90fcadfb69b410982542f73a7316dc0dd799455', 'b5c695eb83eb8e66edf20fceb633cfa76adfbc158b659fe01f6da24651ed8396', '14665367c0484b3d9030943dc6062a0685b78beadbc11a4e1a2a1f810789476c', '8789aa9099d6e5babd3c2ded2a00ad068009d188485b10ea548a0070add4ec68', 'd7c9b1bb8e25e486f51f6733e423edb500bdb7549ee6eac52e799db50e51987f', 'f2bfcb9d4ae808ef0937b090e41b4e55884e13a80cdc0bd6dc40e81b2a5f30d6', '9a61c5649f4970f6b73bd58dfd8d313a5c7c6d9fa72e99f5cc243f1b238744d3', 'eadb362c4ecefda0593ff213da06975610ae8b7e55fcfbebdbc3ca6c8df1ba2d', '7109ab3e8ec43b73ae3bc185471a5715b13ac053e17e1a31e77e77052008e6e4', '1e7a8bc427790cba2adaad6cea2b20453c27dfc9178524c9c267f3a5be0d82b4', '5ff5cb48f13181a91fb6e803db87c41e9138d2297b03ca500fdf9c0f61149a15', 'f31bcdf53b8e10968e306236770922835f6602dc9944dafa29ae65fa8b3da5f7', '5bc125241e1cb4f42c1cced0823cb066ee4c07e7cd11c5cf49333320efce0c3c', '1bf33806aad119b0307e372cc8e0c2087ad7d86d7da87b414e162d7774b6cc63', '125ec2929f73112301fe0b4ce79aad38e31955e8b860422e5db6a25035390041', '76eab83f137de96c2b12c4afe76d685cf4ab41faa246468df52425a80029961c', '4f3a214c17ba433ed46035836574b8a6e4a55f9e9b5d5cd17d549cd9a049350c', '6c8a91833c4831d75070ddd084f49034f72af3b35a42eed85ea84afa2564f598', '2a25befe1b12a5404fb0a4e8fd75a42319a676081367520022d3df09958b790c', '28ec5cd0f695f9969a0fef9cf618c56aacf25c54b4eafd4d19afdd2c6eedfd91', 'cbf8cab5cd14321a99913baf678483413619b1314fdf66273487c9f3f8b3ea0c', '0fd94afad6eea27ef02138019bdd1412a90842d5f27f5639c800f0680e5a5b8f', 'ee210975871490982da1aec55da64e30acc9c5139f7ce4a4fa85dae5a6475f91', '1acd5f3385d8de7ba797fd6ddb8859adcac41ffd917c43e9af40c44f1f7245e2', '0e22acf7c902e06f7353a6b8ef0a851df2b66c76a74c012034bc3f9b8d6b1402', '15a6e3ecdb9977493b59bad49f07e4d126940029163cd246ac911afe666fecac', 'df2d1ed2ba1cadd27cff5b7a62b24c9987f4d9718a5e5eb7b66a6b8c6b8dfacb'], 'sha256': {'last.pt': '1f36b600fe048c85c90013cef18b516cc54547becc676eb9c770e3b016846167', 'final_ema.pt': '44fe8cf5616320075f9032077f67cd7ce3d6709b449866c31fc2626fe5478d8c', 'history.csv': '6cbdf116eebbc77ab44a53fe1e73a88288739be48e61c0572bbafdb093893e43', 'validation_probabilities.npz': '41eebbeda25255d7cbeb1459a0b3878cd50e3a28edd99b2c051cccce9964adb3'}}, '3': {'epoch': 80, 'variant': 'L1_proxy', 'train_ids': ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '17', '20'], 'val_ids': ['15', '16', '18', '19'], 'initial_model_hash': '30be10d91a902aaa0811e387e3d878b94fc609d5e94e86b691ac8b1179ee2327', 'plan_hashes': ['81b4d90d1f56fc0625202ad8a0ef2c352e006bf93c7b9bbc47f8c95576f05362', 'd17d26122c0ed914242428f1564632de804d3c83eb62c46c33600c566005b400', '98bdf0da80b045d98d8e86fb2548a8c5c20e11f554099dcafec0f123f6413a5c', '449b1946f2d4cffe3175cbc3ceb529e5fc7baf2aec0acdeb6cbc53b98725ac22', 'f28636b43951f848b1df1a659804d0780b499c505b155efe6c3abf2b91f6fc6c', '063b64652af11af66155bc68b12f1af852a100b05f3dca60f608f61d5294c9ec', '4fc056c423d2d2943edf4ba4c861c87296685defdcb707684c57f68d2336c45b', 'a5df0b13567114ee944481a574713f23e297a57a0070665e79e4a97a7884330a', '8898b7f3e38db7a5e1dbe29f86207b0e21c573a1e40c3761bdcf8911452eb408', 'fbc38ca65e818b31fe6d6d95e38be321965d1e56decfde78920d834eb59209a6', '002fa527f409675a543141e6f60c4515641bdc4cecd3dd40adbe55d346eb3cce', 'd2ffca0f9129a8736dd9bd743374d4a0bd32bc8d54a4c0a6f43642381593ac05', '9b54b0ad17260bf96f48607cf87912aed251a1a945dc1623280ac2db9dfa5c0c', '2caf3808c90969fdd0bfa238ca3283d82e04d0e5765152e3d2cdc42eecdfceb8', '87d8179785eccc0c9924aa0c1f02aa630d0be76c196a373e6086eb0519132624', '6f97e1967c4bc62ba9c54e8b6af2dc6c6fad7e89fb04b810adb43b4d3dfcea80', 'e726f1482397d40ba99b7e76110fff84f6d6254c09ff5076649260b2cb967784', 'a54eaa62f2d75b8e2688231b97c0215093a399e91405071bfad1d96413772fad', '8b9c9f00bcb45641a8c58f90852560ab8fd09b708bfb2cd59518aa2d84c48cd0', '1948b6dc9a6d10fc4b4e64a8565147f4bcd518045182943d1336ad668fbc3cee', '276f7cec4a296e50eacad2639f0f6735adf65be387da8d56a402d15a36634057', 'a9fc3fb63567f50a45aac6c97a6955ea3ac24c452fbae04653e860591184a58b', '8e79f0f8b1241d6d5235f6355cf20dbc6b3e8b5461c471963600dd1d277c9bc2', '3b9451e851a1012f0d0742dd83b8dff07c6ae64c18f109f4207f844ad24a65cd', 'df31d200fdea8f907817225fb67342d3945a375373c27355273b67564ffe565c', 'a6cfdb25bac0d5c0c1e49982257e78b022ddf14ac979dec4cfa4043ff013fcfa', '419fb3f15ff2bdd57aa043fbfcdbd66da136511f5cd37924fd1f8e0eac913046', '5b32acc8fa068dc601132e4c4ba26587de06f2433e8e9e5f2487e541ab0bfa7b', 'b05dd0cd56a34f03a8437cc1f44dd59e822e81bae163cff823e6aa46111dbc1e', 'ee584a58fc10e9f855a41f3809a112b3db9542e4124903352ac7a2d462f6df60', 'cbfee3c96d92c87432df7437ebb1f8562707a70605797d8e9931543e73bdbcbb', '135831995d1838378fab49526d7eeaeb43945e36fbcac2cd20352fc8e54b679a', 'b7739eaedcede36b61c2fced09988590552eb911f4448075f88a58aac3d28973', 'e139d2fdab6c00c6babfa932d9b798616cd5fd4b9cb684e030aa82addb642a6c', '89c847285842660bdf78a38e97b99ed98781fe6e0bba8a2ae1102b2b4d01869f', '5024f2152ef2ff8e580fdfd88428c2278de99e376efb860386a42f84f5feb870', 'b50cc8671f5eeaf93ad07c29cf14ffc98279776008d9e3ea28d61d168c69993d', 'f62a016fb0ca1ad34f33a2e4ca2364f023d402ec5ae12658aaed22d46a5b0633', 'b7b7b802a313de0fbf0753e03ed2ed663e764a7c09aede800d5c6213f696c76d', '761014389e4f4337356700264862c548f0065b7ae0fbc08927424c6ac22a34ef', '733333dec00441275f08948f17c095d355f97d30287628cd7024a1cef5446e05', '0f265a111d54ccbb5d2efeba9c1aeadbc587935a19fddc193dae463b3ca9e28c', 'a61d72b819465ee2c701e87f62af83e46abd8816deefb5ce269d2e4d4b6193ab', 'fcbed2281e2343f609b082cb5c65a6cf1e1f3795ff9d130bf5c9b0f7cbaae578', 'a4596c5c336b6ded8db5f0144cc70adaf38c4c9c2acd4b006f161074ab0b9adb', '2f634da0dc2afb4c124866846ccb93316ccf822e522a4e77a2700e6424e97007', '1ad21afb4bf197f88f85d37e0c30fd2c05dec2da51a477e5b12d8039f99f047c', '78e00704e1c3337399a511351b0e7603f54d032708233fb3d28ebae0b469a3c1', 'c5958700ce74efbc1eb62169bbbb9c086a1acf7a481d948f0089e344c52ba74d', '9dcf2ab18b0771e1359a3a2e66f4eadfe8f09515659f9229c7231dddeb0c055e', '838887e8db55e0352277564ab5c37e8daa7c692daca5a70c6350af05629cb534', '8e2cf6ef7ca0963aa4c8144c0b28ca95137319d47d3b9af3fdca44b42204e115', '7450247525c4261dddc1b9f8b6f576c8a76f2401a5cbae140491fcce8d477a01', '87961dd7798ddd38fb9a4f6750431caa306af7194831ad169091ac2b9cc6ca3a', '6e16e6e177ad37cd97d13078c994081d169529f9a25fd900a27b850e96e3e2f0', 'a0ddcd3fdc18597ad8979454ea81ece261eae500231dfb56b7eddf8d6ef863ba', '7ce38219a7ba45c334a50a1ac1a87ae588e765accc493604229c60f0ed21c79b', '1306ac9788685633477c81d7bc65254ac206a8a7e0bfa669879d66f7ba4a03c8', '1ebd0d8c4cd3aa70cddb5957893b7bf48bcfce0d8fa5224b39af037e96a063df', '05a2a00999d82ece1f826a0531d83c8997be5f9cda7874da14fc7ac9e309d1ea', '7c79f26870c4ee0ee685fd43a5684e85dd6db9b602e2fb8baa7c58c06f5f1d6d', '2d76ee34a51a153bc835e817d09b2270330835d13c7de054876e6fb9c0c8d381', '356a51d148cc837e03bdc07c8b966507fbb8af9ce08a972673f9b9062b7cd074', '2366a237e3121c2ed4a040911dc3713d5aed65ac21b5ef16dc5eecce35994457', '96bdb41c2583df99b26fcddabcf513e5d9d6ff3d86149b472e0387eee5463739', 'de9eb043108373db2ac00707ab2b69c61e341db04393520f63820e86e2fd31de', '621f095855f3b88973b1f21d8ca139d11f8ad5f2928613e541d20044ad7df7be', '2b4283369eff2299a04ab8db4746750c4f8889f206be423a052343524ceea10e', '7953d72c15b9cb7fcf2721f2f68e104d9858c57e26c72723cf636d37e53f7059', '2a50d9bbbec34a62239c35a39d06f2693f29f9d04bcbbb6f179e75d1db705146', '81075a8666bbdc0e7c8b866b6d41278d9e90fdc7925e070de251282861112947', 'f4d4a47ccd5198ba46ac98871ae1395e3979dc4638eef9096dea3a848122d980', 'c22fc8312c41aa297db92c7c884361411c06cffea48e735777d6d96a0ca5d62e', 'e9cf5652c1c168f2c25249e006148240d591b3f8b47ec0869c49e03130192d7a', '9d812bcba7381de6f64e0a775de31c6f6fb7afe706951bdf6a445507cd635920', 'c35cd6e9f4cb64f83496235e5e8c8d3de8f9c7ff314d49b7b09e2b2dd23b9a4b', '031011f010b91daac8f1afe3c93c3d10b9ca64488650322a3e2790d2ac216cc4', 'c5ef2b62e01d5af71fd54fa706d5ffd46fc9ddd28d8292d21f134a5b1795593a', '88947c60ff6a5cf9dd32691289d895c1339250eaba5d4bd6677b8ffefa3a5fb9', '067a846bfb50e1f18cff7156f5a8add4b50f2414a176821ddc69556bc3a49444'], 'sha256': {'last.pt': 'd280fc477a150da31aa3f6f7eeff8119e92fe18a6371cf61aa4cce780bdd9c5b', 'final_ema.pt': 'fe3afdf49d6bb4259344e74e250ae67b9b4f62c13b68d018a65aadde43a24417', 'history.csv': '9d87fd3c6350f48029d745b65d2465e8cff1246ba226e8094a753a882d5cb1ce', 'validation_probabilities.npz': '162000aaa7dc086e456906d84bb76009b2b7f15adda3f8ade4b337d5cf25b673'}}, '4': {'epoch': 80, 'variant': 'L1_proxy', 'train_ids': ['1', '2', '3', '4', '5', '6', '9', '10', '11', '14', '15', '16', '17', '18', '19', '20'], 'val_ids': ['7', '8', '12', '13'], 'initial_model_hash': '673ef3e38716ac52bd423a8a6275ee34c5b9e67ed9c415f2cf29517a284164b3', 'plan_hashes': ['e589f0aa26da48ed4d2926ec468bda6c4f54200dcdd3fa5852ff86783f1d3cf2', '22cc0c24a26d134f4da358d448e9b7a73add501581cc29c6dfec2b7c795f574a', '7d6225367cc063f12e0c461ccff09461e2233df2f7605f126815933d9bbca8d3', '28fedd890f66176a1a29695ed7ef274a5190d66dea38854e916fbe052db2529a', 'ae824b0d9c9e3743c967d4933d48557d7fac05441bf23495cd99f01e6d05f859', 'aa1b1a4423d137563855f871e4ca12dcb72da7c83ce652a40bc0422a82370bf6', '03afec12422556368f21bc86176c8d55906f3e0af0e0d11f78b5ea8bab88ae98', 'ef8df46c3af125962914035d725eb32c9a2e454f2811ff44420f258a1bd04e58', 'a1b0b7809aa4e4a292fdf18471148fcb885c8978ea9cd4bf2a6ae2a241616958', '941347e13b7a53bb55430473261572f6fcad9a6a2bfbe4f9c76d2295e0ead783', 'aee66cff5fd71efe20bc54b119083468a29408a698d72316014d8fdd2e07208c', '98ced5de7c192861ee511cee3bd08e648f67c4d47ae87c04ae102d286342db04', 'dbe7ddcc867a87d1fe9f165011cf3515f3ad5daf13b69790248268d834c889d2', '209a141c6590b166019071976f83c3203f751447452d255373fcb833a1a54d4e', '58ac602e910e28368c4786aac5ac99ff3c89a4b99039a48fbb70f43bc8642145', 'a4c0d14358915a44f9fa789390a859eff0bfcb10acb102ff20dd039f62f4c6b0', 'ff61bf72c5c575dbd5d5e771fe21ae9de1c2388947fcbd6f60800bb092f08f85', '5495626b251fbfaea3947301a59b4df7184b14d2285a9a407ef6f1daeaa140ba', 'fcbc462ef187965efe648a248b96631afd7c0c9d6b175498698dfc90b2158849', 'e720069a8439e847523d8fbd95bbc5c4a08b4de3bc44a1d1f2e6066a98f1cef1', '97ecfaec725998e0d2a120641931b793ed6b3948dc97132eef09931d222c3631', '3175713e7c12fadb5b2736b0240b27916576c064cf5e3bd80090fc11169f28fb', '5dec6603327987aa92a3226f868e38a32408e16723e4c057e44594e20a18b3e2', 'b723c10ff24d32afb831a383f5c21cce677b749dfa9c5f48c3806b94ce8399f5', '7fbbbac8609257604f0a2b6cb235f7df129c51c0eb8caedd980b06d5518ea51e', '6ce507c85cfddfbbd1cf709eafcd8eee78b2dce0c36cc3c18fe167a2ca64a3e6', 'c92d3873cfc23684337ef047a8704411f2aaceae91d0a39846be0f22437a32b1', 'f8ea80ae277551e3e5f835a614a66c59d661cb0abbd9372dacc032068283bd2d', 'a6065632bcb566512a21aa23ce8b53ee108eaa39c43b74731b155a715802452c', '90c6e3f747e1add7ca09c684ada100d4963d1878420805026286ce050c7009d7', '8c4fdbbff1a5ad92aaae47a9db48009cbb4ada2cd4816347210dc6e749c11acf', 'bb31a567b1892eb2807dc24d4f11528faab6e41cae1f8cb02e7d3618d494ca5b', '076258eb13512bdab8eb3eb6b0fb4db677681fbe5199a7d6f5bca84a86b74bc5', '36b030ba78751582100876570848f7c2d37025146848a4162d344ed1c976e06d', '3ec827ef40aeb044ec0efd8f786fd535cd66db3c43a94c572a7cda0a0dde476e', 'd12ee7f6941ce4720ecf42e437775e9fb410dd7d76ebb9af9d4d3736455f1e4b', 'd0b3542b3b6c575da73c3c7cc67e0731a593e7ab1f825429cf1d35277baa4245', 'f79f5e5cb818cd8577d3d6f83bbbcc63d81cbde85593c8aa9a157446f4c8ce28', 'acf19d2e4daa21c5539efdfd4b49aa345798abca4bbd411d525ca469617cdff2', 'be0dafbbbdcde1baae5e4bcba3868835afefdbf49d6fbbca5559836bb6b581c2', '2cd5dc1069c68de3e13655550c783856ef272f37f1d32dc6413d90a2223017df', '1fa5f7727d1be306e8ca04f7c637f78485fa35a81fdea0f597646bdba8328ba6', 'a280296e60b642612d90deb72b2149b60827dcb923466167f98256e3a7ccbb87', 'c659af1b6df39d5723407ea69f18753b6581f13209a442a5767fec76120b2cde', '171cf713387dbe16747dc3d2746921707cf357404fc7965f77a5fab23e722946', '96eaed0aa9c37a4c95af7c28e4da4b76643c4a444a3da5c17287d717a2be2ae0', 'b27a3a2fe284d32971b0ab1ee6cff429ea3c64ae79c951acf17c9079e9b53cdc', '68380678ce3c3cba4754f5e4167414b9e80d922ebcd1af141059979a3519c086', '1b664d243df3c25e9b2808c84e4bc389d0a6dc18dae330af8f83570a17095f2d', '194046435220224359482d3a11e302afda7b00aa138ad4f549ee9024b60e1267', '0c2c5bb68683f53afb905dc14c27adb9af981115ee970459897f35a2f8e5527e', '806a945e75fb0e9eb94e9e59eea8f49053ac81f43e15177d1e4e0652f0cd6303', 'b94fb4867abb6a0bbf733cbe86bf92e3e980615b182294d54836bf37422f6a30', 'a1d50f65c38606aa2234660dcd225a9c0fcffd25dffe94b29627e98a829f0461', 'e4c935aade0438ec220151babf2536fe648a0261bb83aecd15e1f8807415bea4', 'dc6421a51223c75bc1e7ed09f9759a15fe886c6a028c595ca79f5f3af1fb0969', 'b82e13fe1f70555d0ab62b529fb538c7751fb653ab0efba4238204b386edb8be', '2b55709b2c63d9d4b178bce8368d416d3e7d27cc822dc5fc5778a7c6a787f330', '2d0bcd14543ff5a845c4c09951526b5f355aa536cd24e6f906af8c29d3ed15e7', 'aa897406a1b0ee07d2a454fd6b6105d32f93d61e0e89bc1ea34c3cb13f51a8c4', 'b2cf5eb48561867280d27c9d20055e5c255a9d17c24e1454c3f422a312de91b0', '7dffb9d2fc545f44405d38179182a3325bbaa5d483358a074023f82755e0f14a', '26a7b626967e41bf5fa5e435186da101057da6eabe2d70a75787ab1767012ce6', '323fa480a3bcedaaf9ff4aab0179eb1b4c96a344d578a83ef3d863e74d4fa851', '6d3fff3e495ffc5c459f1b4383bd589d7cebc8ddbc16efbaed063869467649b8', 'e60f71b1e2f4e63233c7832651c955e81870cf378607b91b25fa77f784239602', 'fe1bcae1857869e890528942b6f1bb0a979749b34bdfb4aa5279bc22aa816785', '29c92bc6a28467a3edb9cb6298d1abbe21b3e98e0db1aa7e33051f39f692109f', '3a5d68f263c0a249b8bcffc98fa58c8244b3975c14d1a422730294d79d67e850', '49aadaa4fb6c2b2689002cdb2c4fe682134f382dc17d63748959a9d14d129798', '57f7806d927c29e2b1789df5ed80681b06e51d3c83e7669549cfac0b6673b20b', '0b75ff61a40282e8d16501158aeb887a8f805b6923ff68349b46447d4425bf26', 'a5dd52b3579ce9d2977b028a14fd410cd38609e7a93cf8062c10c7dbce0b8394', 'dbdd1446c580bbd16a8e9121eb7f48f23e2579f562bb9b050ea3ff7a1e569aad', 'c1b441cc2cd9f758fb59cdb06f81e65d6951dd9f5ec2cac745c9dedb04a79677', '570032d1b9d43ce269e0c18a8edbaddda643ec6288ea98d0ff0f8ec445584018', 'e0b52d2662dd3d7fe82ff1ad23d971a4eff50b473d2e42b02b91cc2db7f49611', '0e3c36eb1a09dcbde1434473eeecea0402eaf4ec455701aeadae11a8401ae8f5', 'ae825a3dc9ed3cb3187970f069349675dd78c2d4ca8da27f047d51366fff7169', '3a724859c08f0ff60594da247fb9ac77c1fbcb9915d55ba897bae53b09e61f11'], 'sha256': {'last.pt': 'cca8fc8686c47b633814a83b5989797e3450e291fc494b9d099b060d897499a9', 'final_ema.pt': '1abc45c723be4a9acad281887390dcdbf787096c78aa9224e2a6e7544f7d4f99', 'history.csv': '4c7c6802e7144610428220e33a6ce37666f3abe6be0c079c75c76cd225008e17', 'validation_probabilities.npz': 'ca864c98a2f23aa6d466ee615a25c2b6bb1d07aa2c037e0f32e37957dce403df'}}}
REFERENCE_ROWS=[{'experiment': 'L1_proxy_none_t0.1', 'id': '1', 'fold': 2, 'target': 'clean', 'dice': 0.674631061138937, 'precision': 0.5184315567034515, 'recall': 0.9655413678502711, 'centerline_precision': 0.737788881948782, 'centerline_recall': 0.9409731821951893, 'cldice': 0.8270851085613262, 'foreground_fraction': 0.16246795654296875, 'thin_clean_centerline_recall': 0.8975728155339806}, {'experiment': 'L1_proxy_none_t0.1', 'id': '1', 'fold': 2, 'target': 'proxy', 'dice': 0.8119063195082927, 'precision': 0.7285043437426626, 'recall': 0.9168735224586289, 'centerline_precision': 0.8405996252342286, 'centerline_recall': 0.9322681960944967, 'cldice': 0.8840639858503215, 'foreground_fraction': 0.16246795654296875, 'thin_clean_centerline_recall': 0.8975728155339806}, {'experiment': 'L1_proxy_none_t0.1', 'id': '2', 'fold': 0, 'target': 'clean', 'dice': 0.6815906331603413, 'precision': 0.5350689367035721, 'recall': 0.9386193944391004, 'centerline_precision': 0.6565686419388207, 'centerline_recall': 0.8832720588235294, 'cldice': 0.7532321178900616, 'foreground_fraction': 0.146087646484375, 'thin_clean_centerline_recall': 0.782520325203252}, {'experiment': 'L1_proxy_none_t0.1', 'id': '2', 'fold': 0, 'target': 'proxy', 'dice': 0.8035543637948346, 'precision': 0.7355598495926468, 'recall': 0.8853999685682854, 'centerline_precision': 0.8279887690261564, 'centerline_recall': 0.8749446331020228, 'cldice': 0.8508193318926482, 'foreground_fraction': 0.146087646484375, 'thin_clean_centerline_recall': 0.782520325203252}, {'experiment': 'L1_proxy_none_t0.1', 'id': '3', 'fold': 1, 'target': 'clean', 'dice': 0.7320505259696733, 'precision': 0.6008916107533661, 'recall': 0.936453910663532, 'centerline_precision': 0.8289042591918456, 'centerline_recall': 0.9049442896935933, 'cldice': 0.8652568605724528, 'foreground_fraction': 0.16942596435546875, 'thin_clean_centerline_recall': 0.8343133137337253}, {'experiment': 'L1_proxy_none_t0.1', 'id': '3', 'fold': 1, 'target': 'proxy', 'dice': 0.8463002533834889, 'precision': 0.815936416445265, 'recall': 0.8790113275280763, 'centerline_precision': 0.9252517898313312, 'centerline_recall': 0.8960770818995183, 'cldice': 0.9104307703270045, 'foreground_fraction': 0.16942596435546875, 'thin_clean_centerline_recall': 0.8343133137337253}, {'experiment': 'L1_proxy_none_t0.1', 'id': '4', 'fold': 2, 'target': 'clean', 'dice': 0.6279578961032357, 'precision': 0.4623265588804241, 'recall': 0.9785179943388597, 'centerline_precision': 0.7128791910590739, 'centerline_recall': 0.9646831874165307, 'cldice': 0.8198831579648305, 'foreground_fraction': 0.15973281860351562, 'thin_clean_centerline_recall': 0.9435041716328963}, {'experiment': 'L1_proxy_none_t0.1', 'id': '4', 'fold': 2, 'target': 'proxy', 'dice': 0.7871296077425545, 'precision': 0.6759487020275595, 'recall': 0.9420849420849421, 'centerline_precision': 0.8384779137839277, 'centerline_recall': 0.953016241299304, 'cldice': 0.8920856007701746, 'foreground_fraction': 0.15973281860351562, 'thin_clean_centerline_recall': 0.9435041716328963}, {'experiment': 'L1_proxy_none_t0.1', 'id': '5', 'fold': 2, 'target': 'clean', 'dice': 0.5874590905932953, 'precision': 0.4212668214615634, 'recall': 0.9702144787533782, 'centerline_precision': 0.5220491153947716, 'centerline_recall': 0.9564144736842105, 'cldice': 0.6754245875593557, 'foreground_fraction': 0.15279006958007812, 'thin_clean_centerline_recall': 0.903476669716377}, {'experiment': 'L1_proxy_none_t0.1', 'id': '5', 'fold': 2, 'target': 'proxy', 'dice': 0.7002753457295111, 'precision': 0.5651262077746986, 'recall': 0.9203838490627414, 'centerline_precision': 0.5991550039609189, 'centerline_recall': 0.9435665914221218, 'cldice': 0.7329159668378712, 'foreground_fraction': 0.15279006958007812, 'thin_clean_centerline_recall': 0.903476669716377}, {'experiment': 'L1_proxy_none_t0.1', 'id': '6', 'fold': 0, 'target': 'clean', 'dice': 0.7217220246655548, 'precision': 0.587147869262808, 'recall': 0.936328240028793, 'centerline_precision': 0.7428537819080108, 'centerline_recall': 0.8558539205155746, 'cldice': 0.7953603033900203, 'foreground_fraction': 0.18592453002929688, 'thin_clean_centerline_recall': 0.7272170004086637}, {'experiment': 'L1_proxy_none_t0.1', 'id': '6', 'fold': 0, 'target': 'proxy', 'dice': 0.8265187669291172, 'precision': 0.7888549211104045, 'recall': 0.8679594554936, 'centerline_precision': 0.8864839430655217, 'centerline_recall': 0.8317912464913193, 'cldice': 0.858267160555562, 'foreground_fraction': 0.18592453002929688, 'thin_clean_centerline_recall': 0.7272170004086637}, {'experiment': 'L1_proxy_none_t0.1', 'id': '7', 'fold': 4, 'target': 'clean', 'dice': 0.6317057195384823, 'precision': 0.47506912003050816, 'recall': 0.9424410540915396, 'centerline_precision': 0.7057375497064949, 'centerline_recall': 0.874042401567789, 'cldice': 0.7809246374148144, 'foreground_fraction': 0.12003707885742188, 'thin_clean_centerline_recall': 0.798460661345496}, {'experiment': 'L1_proxy_none_t0.1', 'id': '7', 'fold': 4, 'target': 'proxy', 'dice': 0.7797004872460238, 'precision': 0.6916134362983443, 'recall': 0.893500841647165, 'centerline_precision': 0.8731300889982958, 'centerline_recall': 0.8597134870719776, 'cldice': 0.8663698487805683, 'foreground_fraction': 0.12003707885742188, 'thin_clean_centerline_recall': 0.798460661345496}, {'experiment': 'L1_proxy_none_t0.1', 'id': '8', 'fold': 4, 'target': 'clean', 'dice': 0.6351481511719286, 'precision': 0.47789023731145286, 'recall': 0.9466640712443428, 'centerline_precision': 0.6416088765603328, 'centerline_recall': 0.8944985594731788, 'cldice': 0.7472370777793719, 'foreground_fraction': 0.15528106689453125, 'thin_clean_centerline_recall': 0.8312829525483304}, {'experiment': 'L1_proxy_none_t0.1', 'id': '8', 'fold': 4, 'target': 'proxy', 'dice': 0.7799826007708411, 'precision': 0.6938043531666094, 'recall': 0.8906057834820724, 'centerline_precision': 0.8381414701803052, 'centerline_recall': 0.8745030479724357, 'cldice': 0.8559362582672361, 'foreground_fraction': 0.15528106689453125, 'thin_clean_centerline_recall': 0.8312829525483304}, {'experiment': 'L1_proxy_none_t0.1', 'id': '9', 'fold': 0, 'target': 'clean', 'dice': 0.6668860659519732, 'precision': 0.5126752011334311, 'recall': 0.9537795349712888, 'centerline_precision': 0.6910702113156101, 'centerline_recall': 0.9041564123598578, 'cldice': 0.7833815630687555, 'foreground_fraction': 0.15077972412109375, 'thin_clean_centerline_recall': 0.8451111111111111}, {'experiment': 'L1_proxy_none_t0.1', 'id': '9', 'fold': 0, 'target': 'proxy', 'dice': 0.8110446053607202, 'precision': 0.735338764357638, 'recall': 0.9041279124024014, 'centerline_precision': 0.8688479890933879, 'centerline_recall': 0.8954768536941239, 'cldice': 0.8819614673481267, 'foreground_fraction': 0.15077972412109375, 'thin_clean_centerline_recall': 0.8451111111111111}, {'experiment': 'L1_proxy_none_t0.1', 'id': '10', 'fold': 0, 'target': 'clean', 'dice': 0.645648676781936, 'precision': 0.4863150392017106, 'recall': 0.9602645899793583, 'centerline_precision': 0.6398744541484717, 'centerline_recall': 0.9091796875, 'cldice': 0.7511175247145881, 'foreground_fraction': 0.16056060791015625, 'thin_clean_centerline_recall': 0.8416058394160584}, {'experiment': 'L1_proxy_none_t0.1', 'id': '10', 'fold': 0, 'target': 'proxy', 'dice': 0.7927295231947682, 'precision': 0.699952482775006, 'recall': 0.913859420559588, 'centerline_precision': 0.835971615720524, 'centerline_recall': 0.8919497191762503, 'cldice': 0.8630539282342761, 'foreground_fraction': 0.16056060791015625, 'thin_clean_centerline_recall': 0.8416058394160584}, {'experiment': 'L1_proxy_none_t0.1', 'id': '11', 'fold': 2, 'target': 'clean', 'dice': 0.6418954040147523, 'precision': 0.4782261086820737, 'recall': 0.9758845722443152, 'centerline_precision': 0.6326449033977348, 'centerline_recall': 0.9572560155432671, 'cldice': 0.761812427762668, 'foreground_fraction': 0.15268325805664062, 'thin_clean_centerline_recall': 0.9294697903822441}, {'experiment': 'L1_proxy_none_t0.1', 'id': '11', 'fold': 2, 'target': 'proxy', 'dice': 0.8027982496548209, 'precision': 0.6967145534041225, 'recall': 0.9469895065711278, 'centerline_precision': 0.8310459693537642, 'centerline_recall': 0.9532206353832702, 'cldice': 0.8879504495987034, 'foreground_fraction': 0.15268325805664062, 'thin_clean_centerline_recall': 0.9294697903822441}, {'experiment': 'L1_proxy_none_t0.1', 'id': '12', 'fold': 4, 'target': 'clean', 'dice': 0.6168261020694612, 'precision': 0.504648862512364, 'recall': 0.7931289106525203, 'centerline_precision': 0.7091219581613776, 'centerline_recall': 0.7996657115568291, 'cldice': 0.7516770273707218, 'foreground_fraction': 0.154266357421875, 'thin_clean_centerline_recall': 0.7677358490566037}, {'experiment': 'L1_proxy_none_t0.1', 'id': '12', 'fold': 4, 'target': 'proxy', 'dice': 0.7263252240720534, 'precision': 0.7013600395647873, 'recall': 0.753133297928837, 'centerline_precision': 0.8359185996869218, 'centerline_recall': 0.7929135743356476, 'cldice': 0.8138483697734504, 'foreground_fraction': 0.154266357421875, 'thin_clean_centerline_recall': 0.7677358490566037}, {'experiment': 'L1_proxy_none_t0.1', 'id': '13', 'fold': 4, 'target': 'clean', 'dice': 0.6763823003348994, 'precision': 0.5289224033081975, 'recall': 0.9378477463877507, 'centerline_precision': 0.6893006726741973, 'centerline_recall': 0.8849040867389492, 'cldice': 0.7749499912180355, 'foreground_fraction': 0.15682220458984375, 'thin_clean_centerline_recall': 0.8233876945885842}, {'experiment': 'L1_proxy_none_t0.1', 'id': '13', 'fold': 4, 'target': 'proxy', 'dice': 0.8193853243256148, 'precision': 0.7655801508148868, 'recall': 0.8813250819075354, 'centerline_precision': 0.8926259677624064, 'centerline_recall': 0.8779976717112922, 'cldice': 0.8852513927097306, 'foreground_fraction': 0.15682220458984375, 'thin_clean_centerline_recall': 0.8233876945885842}, {'experiment': 'L1_proxy_none_t0.1', 'id': '14', 'fold': 1, 'target': 'clean', 'dice': 0.6600809078655474, 'precision': 0.5022248864500957, 'recall': 0.9626568852748807, 'centerline_precision': 0.7364814353351608, 'centerline_recall': 0.9393489126029988, 'cldice': 0.8256361227561536, 'foreground_fraction': 0.16545486450195312, 'thin_clean_centerline_recall': 0.8987570621468927}, {'experiment': 'L1_proxy_none_t0.1', 'id': '14', 'fold': 1, 'target': 'proxy', 'dice': 0.8064323693427488, 'precision': 0.7168514974753879, 'recall': 0.9215994308918991, 'centerline_precision': 0.8458759033142288, 'centerline_recall': 0.9325391513853567, 'cldice': 0.8870959509361792, 'foreground_fraction': 0.16545486450195312, 'thin_clean_centerline_recall': 0.8987570621468927}, {'experiment': 'L1_proxy_none_t0.1', 'id': '15', 'fold': 3, 'target': 'clean', 'dice': 0.6301455984688155, 'precision': 0.46559405417959926, 'recall': 0.9745867585963052, 'centerline_precision': 0.7151415482203688, 'centerline_recall': 0.9593694795488518, 'cldice': 0.8194451556809822, 'foreground_fraction': 0.18066787719726562, 'thin_clean_centerline_recall': 0.9352385342244757}, {'experiment': 'L1_proxy_none_t0.1', 'id': '15', 'fold': 3, 'target': 'proxy', 'dice': 0.7786382106292951, 'precision': 0.6655687168767551, 'recall': 0.9379872641790157, 'centerline_precision': 0.811347351110067, 'centerline_recall': 0.9523809523809523, 'cldice': 0.8762253930296504, 'foreground_fraction': 0.18066787719726562, 'thin_clean_centerline_recall': 0.9352385342244757}, {'experiment': 'L1_proxy_none_t0.1', 'id': '16', 'fold': 3, 'target': 'clean', 'dice': 0.662614568259105, 'precision': 0.5134057609887371, 'recall': 0.9340828724272373, 'centerline_precision': 0.6644597830680378, 'centerline_recall': 0.851455423573946, 'cldice': 0.7464241846261318, 'foreground_fraction': 0.1530914306640625, 'thin_clean_centerline_recall': 0.7635078969243557}, {'experiment': 'L1_proxy_none_t0.1', 'id': '16', 'fold': 3, 'target': 'proxy', 'dice': 0.7967437898070387, 'precision': 0.7340775441044552, 'recall': 0.8711079570655549, 'centerline_precision': 0.8737850401464995, 'centerline_recall': 0.8390127787521924, 'cldice': 0.8560459459678618, 'foreground_fraction': 0.1530914306640625, 'thin_clean_centerline_recall': 0.7635078969243557}, {'experiment': 'L1_proxy_none_t0.1', 'id': '17', 'fold': 1, 'target': 'clean', 'dice': 0.6799227077234933, 'precision': 0.5386134702745408, 'recall': 0.9217508872739564, 'centerline_precision': 0.7098678301039394, 'centerline_recall': 0.8538053949903661, 'cldice': 0.775211819638716, 'foreground_fraction': 0.154510498046875, 'thin_clean_centerline_recall': 0.7684973302822273}, {'experiment': 'L1_proxy_none_t0.1', 'id': '17', 'fold': 1, 'target': 'proxy', 'dice': 0.8088730959164309, 'precision': 0.7643195733754691, 'recall': 0.8589423450418956, 'centerline_precision': 0.8958039266007956, 'centerline_recall': 0.845383630224706, 'cldice': 0.8698637576069257, 'foreground_fraction': 0.154510498046875, 'thin_clean_centerline_recall': 0.7684973302822273}, {'experiment': 'L1_proxy_none_t0.1', 'id': '18', 'fold': 3, 'target': 'clean', 'dice': 0.651816383887267, 'precision': 0.4893935334047874, 'recall': 0.975606498998442, 'centerline_precision': 0.7274975272007913, 'centerline_recall': 0.9645791245791245, 'cldice': 0.8294292426795913, 'foreground_fraction': 0.17083740234375, 'thin_clean_centerline_recall': 0.9405380963147185}, {'experiment': 'L1_proxy_none_t0.1', 'id': '18', 'fold': 3, 'target': 'proxy', 'dice': 0.8001274859767974, 'precision': 0.7007190067881386, 'recall': 0.9324043261231281, 'centerline_precision': 0.8597922848664689, 'centerline_recall': 0.9544016868740116, 'cldice': 0.9046300669278193, 'foreground_fraction': 0.17083740234375, 'thin_clean_centerline_recall': 0.9405380963147185}, {'experiment': 'L1_proxy_none_t0.1', 'id': '19', 'fold': 3, 'target': 'clean', 'dice': 0.7336141089764846, 'precision': 0.6248003362757462, 'recall': 0.8883222439141002, 'centerline_precision': 0.7646139418568303, 'centerline_recall': 0.8027595269382392, 'cldice': 0.7832225547715005, 'foreground_fraction': 0.13612747192382812, 'thin_clean_centerline_recall': 0.6329720191580539}, {'experiment': 'L1_proxy_none_t0.1', 'id': '19', 'fold': 3, 'target': 'proxy', 'dice': 0.8073005446308744, 'precision': 0.816225304749895, 'recall': 0.7985688435597961, 'centerline_precision': 0.8927789934354485, 'centerline_recall': 0.7888476787312956, 'cldice': 0.837601648746665, 'foreground_fraction': 0.13612747192382812, 'thin_clean_centerline_recall': 0.6329720191580539}, {'experiment': 'L1_proxy_none_t0.1', 'id': '20', 'fold': 1, 'target': 'clean', 'dice': 0.7135767720913805, 'precision': 0.5763754277646158, 'recall': 0.9365036161443347, 'centerline_precision': 0.7344408945686901, 'centerline_recall': 0.8857074683846338, 'cldice': 0.8030126132656175, 'foreground_fraction': 0.15940475463867188, 'thin_clean_centerline_recall': 0.8099700897308075}, {'experiment': 'L1_proxy_none_t0.1', 'id': '20', 'fold': 1, 'target': 'proxy', 'dice': 0.833893897148209, 'precision': 0.7980233086845191, 'recall': 0.8731409719312945, 'centerline_precision': 0.9035143769968051, 'centerline_recall': 0.8734693877551021, 'cldice': 0.8882378841694308, 'foreground_fraction': 0.15940475463867188, 'thin_clean_centerline_recall': 0.8099700897308075}, {'experiment': 'L1_proxy_none_t0.4', 'id': '1', 'fold': 2, 'target': 'clean', 'dice': 0.7431878466365648, 'precision': 0.6130718954248366, 'recall': 0.9434143781703691, 'centerline_precision': 0.8024220982446592, 'centerline_recall': 0.9173348078518109, 'cldice': 0.8560392677591843, 'foreground_fraction': 0.13423919677734375, 'thin_clean_centerline_recall': 0.8570388349514563}, {'experiment': 'L1_proxy_none_t0.4', 'id': '1', 'fold': 2, 'target': 'proxy', 'dice': 0.8378675938000321, 'precision': 0.8217959647627167, 'recall': 0.854580378250591, 'centerline_precision': 0.8934548918220167, 'centerline_recall': 0.9019527516045337, 'cldice': 0.8976837110656606, 'foreground_fraction': 0.13423919677734375, 'thin_clean_centerline_recall': 0.8570388349514563}, {'experiment': 'L1_proxy_none_t0.4', 'id': '2', 'fold': 0, 'target': 'clean', 'dice': 0.7389288329051814, 'precision': 0.6260258286150518, 'recall': 0.9015161925701983, 'centerline_precision': 0.7155047496377395, 'centerline_recall': 0.8491115196078431, 'cldice': 0.7766036148204849, 'foreground_fraction': 0.11992645263671875, 'thin_clean_centerline_recall': 0.7215447154471545}, {'experiment': 'L1_proxy_none_t0.4', 'id': '2', 'fold': 0, 'target': 'proxy', 'dice': 0.8239925379036188, 'precision': 0.8289331382403461, 'recall': 0.8191104824768191, 'centerline_precision': 0.8927708903558204, 'centerline_recall': 0.8415768492543925, 'cldice': 0.8664183033796318, 'foreground_fraction': 0.11992645263671875, 'thin_clean_centerline_recall': 0.7215447154471545}, {'experiment': 'L1_proxy_none_t0.4', 'id': '3', 'fold': 1, 'target': 'clean', 'dice': 0.78187591597493, 'precision': 0.6920145926226186, 'recall': 0.8985578441348819, 'centerline_precision': 0.8747904577691812, 'centerline_recall': 0.8698932219127206, 'cldice': 0.8723349666984687, 'foreground_fraction': 0.14116287231445312, 'thin_clean_centerline_recall': 0.7771944561108778}, {'experiment': 'L1_proxy_none_t0.4', 'id': '3', 'fold': 1, 'target': 'proxy', 'dice': 0.8443603640456023, 'precision': 0.8925280367517903, 'recall': 0.8011254760229947, 'centerline_precision': 0.9548678272082527, 'centerline_recall': 0.8589125946317963, 'cldice': 0.9043520297411084, 'foreground_fraction': 0.14116287231445312, 'thin_clean_centerline_recall': 0.7771944561108778}, {'experiment': 'L1_proxy_none_t0.4', 'id': '4', 'fold': 2, 'target': 'clean', 'dice': 0.6969093268515766, 'precision': 0.5458215236622863, 'recall': 0.963657501010918, 'centerline_precision': 0.768893291819983, 'centerline_recall': 0.9465796112182816, 'cldice': 0.848534199462164, 'foreground_fraction': 0.13324356079101562, 'thin_clean_centerline_recall': 0.9144219308700834}, {'experiment': 'L1_proxy_none_t0.4', 'id': '4', 'fold': 2, 'target': 'proxy', 'dice': 0.8242500731075613, 'precision': 0.7666122706060866, 'recall': 0.8912594860870723, 'centerline_precision': 0.8887630908576281, 'centerline_recall': 0.923723897911833, 'cldice': 0.9059063173353352, 'foreground_fraction': 0.13324356079101562, 'thin_clean_centerline_recall': 0.9144219308700834}, {'experiment': 'L1_proxy_none_t0.4', 'id': '5', 'fold': 2, 'target': 'clean', 'dice': 0.6664627182255131, 'precision': 0.5163869662779306, 'recall': 0.9395089414064746, 'centerline_precision': 0.581978021978022, 'centerline_recall': 0.9352384868421053, 'cldice': 0.7174826288613955, 'foreground_fraction': 0.12070083618164062, 'thin_clean_centerline_recall': 0.8581884720951509}, {'experiment': 'L1_proxy_none_t0.4', 'id': '5', 'fold': 2, 'target': 'proxy', 'dice': 0.7441049898642386, 'precision': 0.661230681710439, 'recall': 0.8507298824868865, 'centerline_precision': 0.6545054945054946, 'centerline_recall': 0.9187358916478555, 'cldice': 0.7644315670504658, 'foreground_fraction': 0.12070083618164062, 'thin_clean_centerline_recall': 0.8581884720951509}, {'experiment': 'L1_proxy_none_t0.4', 'id': '6', 'fold': 0, 'target': 'clean', 'dice': 0.7779965717041211, 'precision': 0.6860290810972868, 'recall': 0.8984392893367797, 'centerline_precision': 0.8088142442227554, 'centerline_recall': 0.8163265306122449, 'cldice': 0.8125530244764948, 'foreground_fraction': 0.15268707275390625, 'thin_clean_centerline_recall': 0.6538618716796076}, {'experiment': 'L1_proxy_none_t0.4', 'id': '6', 'fold': 0, 'target': 'proxy', 'dice': 0.8362605694770866, 'precision': 0.880877429670714, 'recall': 0.7959455493600018, 'centerline_precision': 0.9363556004546029, 'centerline_recall': 0.7877118203555463, 'cldice': 0.855625906076937, 'foreground_fraction': 0.15268707275390625, 'thin_clean_centerline_recall': 0.6538618716796076}, {'experiment': 'L1_proxy_none_t0.4', 'id': '7', 'fold': 4, 'target': 'clean', 'dice': 0.6970154620647321, 'precision': 0.5623331915058214, 'recall': 0.9165300718698777, 'centerline_precision': 0.7614432989690721, 'centerline_recall': 0.8393016212364155, 'cldice': 0.7984789921711052, 'foreground_fraction': 0.09862136840820312, 'thin_clean_centerline_recall': 0.7428734321550741}, {'experiment': 'L1_proxy_none_t0.4', 'id': '7', 'fold': 4, 'target': 'proxy', 'dice': 0.8159729137625653, 'precision': 0.7923645224925541, 'recall': 0.841031325696925, 'centerline_precision': 0.9303092783505155, 'centerline_recall': 0.8212788259958071, 'cldice': 0.8724006631934192, 'foreground_fraction': 0.09862136840820312, 'thin_clean_centerline_recall': 0.7428734321550741}, {'experiment': 'L1_proxy_none_t0.4', 'id': '8', 'fold': 4, 'target': 'clean', 'dice': 0.7030606899653837, 'precision': 0.5710725465763, 'recall': 0.914399727480656, 'centerline_precision': 0.7112062375783519, 'centerline_recall': 0.8575936342433804, 'cldice': 0.7775701068525936, 'foreground_fraction': 0.12551498413085938, 'thin_clean_centerline_recall': 0.773286467486819}, {'experiment': 'L1_proxy_none_t0.4', 'id': '8', 'fold': 4, 'target': 'proxy', 'dice': 0.8106602284337595, 'precision': 0.7959760508160351, 'recall': 0.8258963766516351, 'centerline_precision': 0.907047852010396, 'centerline_recall': 0.8315663927908826, 'cldice': 0.8676686190054947, 'foreground_fraction': 0.12551498413085938, 'thin_clean_centerline_recall': 0.773286467486819}, {'experiment': 'L1_proxy_none_t0.4', 'id': '9', 'fold': 0, 'target': 'clean', 'dice': 0.7327104192600585, 'precision': 0.6052106668303311, 'recall': 0.9282688506071731, 'centerline_precision': 0.73914949583516, 'centerline_recall': 0.8743505605687723, 'cldice': 0.8010855326128422, 'foreground_fraction': 0.12430953979492188, 'thin_clean_centerline_recall': 0.7971111111111111}, {'experiment': 'L1_proxy_none_t0.4', 'id': '9', 'fold': 0, 'target': 'proxy', 'dice': 0.8381376092936662, 'precision': 0.8324792095007212, 'recall': 0.8438734563100756, 'centerline_precision': 0.9126114277363729, 'centerline_recall': 0.862315957023478, 'cldice': 0.8867510901642088, 'foreground_fraction': 0.12430953979492188, 'thin_clean_centerline_recall': 0.7971111111111111}, {'experiment': 'L1_proxy_none_t0.4', 'id': '10', 'fold': 0, 'target': 'clean', 'dice': 0.711665207995228, 'precision': 0.5746012517666061, 'recall': 0.9346031150309626, 'centerline_precision': 0.6979441997063143, 'centerline_recall': 0.8706752232142857, 'cldice': 0.7747994357215998, 'foreground_fraction': 0.13225936889648438, 'thin_clean_centerline_recall': 0.7754257907542579}, {'experiment': 'L1_proxy_none_t0.4', 'id': '10', 'fold': 0, 'target': 'proxy', 'dice': 0.8290663438403966, 'precision': 0.799976925961178, 'recall': 0.8603511384080899, 'centerline_precision': 0.8904552129221732, 'centerline_recall': 0.8539716501738432, 'cldice': 0.8718319164558681, 'foreground_fraction': 0.13225936889648438, 'thin_clean_centerline_recall': 0.7754257907542579}, {'experiment': 'L1_proxy_none_t0.4', 'id': '11', 'fold': 2, 'target': 'clean', 'dice': 0.7175083852425156, 'precision': 0.57485949448727, 'recall': 0.9543183440399714, 'centerline_precision': 0.691157440690151, 'centerline_recall': 0.9385742041548348, 'cldice': 0.7960851062730061, 'foreground_fraction': 0.12421035766601562, 'thin_clean_centerline_recall': 0.899383477188656}, {'experiment': 'L1_proxy_none_t0.4', 'id': '11', 'fold': 2, 'target': 'proxy', 'dice': 0.8493742742874316, 'precision': 0.8087589447498541, 'recall': 0.8942846469929026, 'centerline_precision': 0.8908698777857657, 'centerline_recall': 0.9284465170504226, 'cldice': 0.9092701385234362, 'foreground_fraction': 0.12421035766601562, 'thin_clean_centerline_recall': 0.899383477188656}, {'experiment': 'L1_proxy_none_t0.4', 'id': '12', 'fold': 4, 'target': 'clean', 'dice': 0.6475983586778946, 'precision': 0.5819776269715844, 'recall': 0.7298977886595935, 'centerline_precision': 0.7636250389286826, 'centerline_recall': 0.7412846227316141, 'cldice': 0.7522890088514033, 'foreground_fraction': 0.12310409545898438, 'thin_clean_centerline_recall': 0.7045283018867925}, {'experiment': 'L1_proxy_none_t0.4', 'id': '12', 'fold': 4, 'target': 'proxy', 'dice': 0.7198524259631353, 'precision': 0.7799572371479037, 'recall': 0.668348380244291, 'centerline_precision': 0.8869511055745873, 'centerline_recall': 0.727316255685899, 'cldice': 0.7992405379233605, 'foreground_fraction': 0.12310409545898438, 'thin_clean_centerline_recall': 0.7045283018867925}, {'experiment': 'L1_proxy_none_t0.4', 'id': '13', 'fold': 4, 'target': 'clean', 'dice': 0.7354282193473543, 'precision': 0.6212968471148126, 'recall': 0.9009273237006685, 'centerline_precision': 0.7366992399565689, 'centerline_recall': 0.8443941379721196, 'cldice': 0.7868789134804656, 'foreground_fraction': 0.1282501220703125, 'thin_clean_centerline_recall': 0.7614899925871016}, {'experiment': 'L1_proxy_none_t0.4', 'id': '13', 'fold': 4, 'target': 'proxy', 'dice': 0.8318068396535001, 'precision': 0.857674003569304, 'recall': 0.8074542857942931, 'centerline_precision': 0.9314603691639523, 'centerline_recall': 0.830034924330617, 'cldice': 0.8778276500553742, 'foreground_fraction': 0.1282501220703125, 'thin_clean_centerline_recall': 0.7614899925871016}, {'experiment': 'L1_proxy_none_t0.4', 'id': '14', 'fold': 1, 'target': 'clean', 'dice': 0.7309958598894866, 'precision': 0.597925975887362, 'recall': 0.9402510164398091, 'centerline_precision': 0.7954606863009998, 'centerline_recall': 0.9135485613940294, 'cldice': 0.8504248488952156, 'foreground_fraction': 0.13573837280273438, 'thin_clean_centerline_recall': 0.855819209039548}, {'experiment': 'L1_proxy_none_t0.4', 'id': '14', 'fold': 1, 'target': 'proxy', 'dice': 0.8378822850550521, 'precision': 0.8161481606385071, 'recall': 0.8608056436553339, 'centerline_precision': 0.8990813293704404, 'centerline_recall': 0.9056351224735645, 'cldice': 0.9023463259351666, 'foreground_fraction': 0.13573837280273438, 'thin_clean_centerline_recall': 0.855819209039548}, {'experiment': 'L1_proxy_none_t0.4', 'id': '15', 'fold': 3, 'target': 'clean', 'dice': 0.7047110879754934, 'precision': 0.5575863666786799, 'recall': 0.9573057544417927, 'centerline_precision': 0.7778909831889964, 'centerline_recall': 0.9410245957331159, 'cldice': 0.8517166950559478, 'foreground_fraction': 0.14818572998046875, 'thin_clean_centerline_recall': 0.9048167780594607}, {'experiment': 'L1_proxy_none_t0.4', 'id': '15', 'fold': 3, 'target': 'proxy', 'dice': 0.8200739800146026, 'precision': 0.7647634248056427, 'recall': 0.8840088079509612, 'centerline_precision': 0.8753183902190524, 'centerline_recall': 0.9213075060532687, 'cldice': 0.8977243451388418, 'foreground_fraction': 0.14818572998046875, 'thin_clean_centerline_recall': 0.9048167780594607}, {'experiment': 'L1_proxy_none_t0.4', 'id': '16', 'fold': 3, 'target': 'clean', 'dice': 0.7212173724688234, 'precision': 0.5991901619676064, 'recall': 0.9056578112249524, 'centerline_precision': 0.708143421452446, 'centerline_recall': 0.8154287951964495, 'cldice': 0.7580087516315347, 'foreground_fraction': 0.1271820068359375, 'thin_clean_centerline_recall': 0.707190357439734}, {'experiment': 'L1_proxy_none_t0.4', 'id': '16', 'fold': 3, 'target': 'proxy', 'dice': 0.8185946783010191, 'precision': 0.8244751049790042, 'recall': 0.8127975398444661, 'centerline_precision': 0.9187177149802491, 'centerline_recall': 0.7945377098471561, 'cldice': 0.8521273111742294, 'foreground_fraction': 0.1271820068359375, 'thin_clean_centerline_recall': 0.707190357439734}, {'experiment': 'L1_proxy_none_t0.4', 'id': '17', 'fold': 1, 'target': 'clean', 'dice': 0.7376546473516308, 'precision': 0.6327628953256335, 'recall': 0.884231874260605, 'centerline_precision': 0.7514132083275886, 'centerline_recall': 0.8167148362235067, 'cldice': 0.7827043429366459, 'foreground_fraction': 0.12616729736328125, 'thin_clean_centerline_recall': 0.7105263157894737}, {'experiment': 'L1_proxy_none_t0.4', 'id': '17', 'fold': 1, 'target': 'proxy', 'dice': 0.8219225649635078, 'precision': 0.8588014754792284, 'recall': 0.7880805726652239, 'centerline_precision': 0.9303736384944161, 'centerline_recall': 0.8046338339736873, 'cldice': 0.8629474162493257, 'foreground_fraction': 0.12616729736328125, 'thin_clean_centerline_recall': 0.7105263157894737}, {'experiment': 'L1_proxy_none_t0.4', 'id': '18', 'fold': 3, 'target': 'clean', 'dice': 0.7194193602840419, 'precision': 0.5794181620445894, 'recall': 0.9486312040952593, 'centerline_precision': 0.7783049953868459, 'centerline_recall': 0.9336026936026937, 'cldice': 0.8489098387851581, 'foreground_fraction': 0.1403045654296875, 'thin_clean_centerline_recall': 0.8898937372823875}, {'experiment': 'L1_proxy_none_t0.4', 'id': '18', 'fold': 3, 'target': 'proxy', 'dice': 0.8266795388724137, 'precision': 0.7915715062533986, 'recall': 0.8650463513192298, 'centerline_precision': 0.9035191775405299, 'centerline_recall': 0.9111755403268318, 'cldice': 0.9073312074866803, 'foreground_fraction': 0.1403045654296875, 'thin_clean_centerline_recall': 0.8898937372823875}, {'experiment': 'L1_proxy_none_t0.4', 'id': '19', 'fold': 3, 'target': 'clean', 'dice': 0.7733043430197097, 'precision': 0.719319895790484, 'recall': 0.8360492449898402, 'centerline_precision': 0.8108735289242499, 'centerline_recall': 0.7703022339027595, 'cldice': 0.7900673731884171, 'foreground_fraction': 0.1112823486328125, 'thin_clean_centerline_recall': 0.5775144945802874}, {'experiment': 'L1_proxy_none_t0.4', 'id': '19', 'fold': 3, 'target': 'proxy', 'dice': 0.7961185753895194, 'precision': 0.895756204579734, 'recall': 0.7164281405932993, 'centerline_precision': 0.94165423504061, 'centerline_recall': 0.7525258984524875, 'cldice': 0.836533477457888, 'foreground_fraction': 0.1112823486328125, 'thin_clean_centerline_recall': 0.5775144945802874}, {'experiment': 'L1_proxy_none_t0.4', 'id': '20', 'fold': 1, 'target': 'clean', 'dice': 0.7668247703078596, 'precision': 0.6685656448814343, 'recall': 0.8989423749902792, 'centerline_precision': 0.7853183723130999, 'centerline_recall': 0.8530183727034121, 'cldice': 0.8177696093825465, 'foreground_fraction': 0.1319122314453125, 'thin_clean_centerline_recall': 0.7563310069790629}, {'experiment': 'L1_proxy_none_t0.4', 'id': '20', 'fold': 1, 'target': 'proxy', 'dice': 0.8407354476998835, 'precision': 0.8846443030653557, 'recall': 0.800979262672811, 'centerline_precision': 0.9470055427876166, 'centerline_recall': 0.8377842565597667, 'cldice': 0.8890529684923202, 'foreground_fraction': 0.1319122314453125, 'thin_clean_centerline_recall': 0.7563310069790629}, {'experiment': 'L1_proxy_none_t0.5', 'id': '1', 'fold': 2, 'target': 'clean', 'dice': 0.7569030477778343, 'precision': 0.6355265112134264, 'recall': 0.9355868462480322, 'centerline_precision': 0.8113285556780595, 'centerline_recall': 0.9124965440973182, 'cldice': 0.8589438722990712, 'foreground_fraction': 0.12842178344726562, 'thin_clean_centerline_recall': 0.8492718446601941}, {'experiment': 'L1_proxy_none_t0.5', 'id': '1', 'fold': 2, 'target': 'proxy', 'dice': 0.8386638026815219, 'precision': 0.8408436061191148, 'recall': 0.8364952718676123, 'centerline_precision': 0.9031146637265711, 'centerline_recall': 0.8930766079475625, 'cldice': 0.8980675868855404, 'foreground_fraction': 0.12842178344726562, 'thin_clean_centerline_recall': 0.8492718446601941}, {'experiment': 'L1_proxy_none_t0.5', 'id': '2', 'fold': 0, 'target': 'clean', 'dice': 0.7489200863935728, 'precision': 0.6467277268942548, 'recall': 0.8894691035683203, 'centerline_precision': 0.7241322855271775, 'centerline_recall': 0.8390012254901961, 'cldice': 0.7773461072802321, 'foreground_fraction': 0.11453628540039062, 'thin_clean_centerline_recall': 0.7044134727061556}, {'experiment': 'L1_proxy_none_t0.5', 'id': '2', 'fold': 0, 'target': 'proxy', 'dice': 0.822703751617363, 'precision': 0.8472273105745213, 'recall': 0.7995599559955996, 'centerline_precision': 0.9034053700065487, 'centerline_recall': 0.8303558245976672, 'cldice': 0.8653416783030407, 'foreground_fraction': 0.11453628540039062, 'thin_clean_centerline_recall': 0.7044134727061556}, {'experiment': 'L1_proxy_none_t0.5', 'id': '3', 'fold': 1, 'target': 'clean', 'dice': 0.7899609069588899, 'precision': 0.7125045837917126, 'recall': 0.8863118004140497, 'centerline_precision': 0.8763707571801567, 'centerline_recall': 0.8617688022284122, 'cldice': 0.8690084448456252, 'foreground_fraction': 0.13523483276367188, 'thin_clean_centerline_recall': 0.7637547249055019}, {'experiment': 'L1_proxy_none_t0.5', 'id': '3', 'fold': 1, 'target': 'proxy', 'dice': 0.837919611883672, 'precision': 0.9061803616259062, 'recall': 0.7792223542823877, 'centerline_precision': 0.9587467362924282, 'centerline_recall': 0.8492773571920165, 'cldice': 0.9006980574530822, 'foreground_fraction': 0.13523483276367188, 'thin_clean_centerline_recall': 0.7637547249055019}, {'experiment': 'L1_proxy_none_t0.5', 'id': '4', 'fold': 2, 'target': 'clean', 'dice': 0.712240757439675, 'precision': 0.5667603444152117, 'recall': 0.9581985442782046, 'centerline_precision': 0.7765850865893803, 'centerline_recall': 0.9404956224959192, 'cldice': 0.8507169995777335, 'foreground_fraction': 0.127593994140625, 'thin_clean_centerline_recall': 0.9048867699642431}, {'experiment': 'L1_proxy_none_t0.5', 'id': '4', 'fold': 2, 'target': 'proxy', 'dice': 0.8281673281675989, 'precision': 0.7860260703181057, 'recall': 0.8750832112901078, 'centerline_precision': 0.8972377272076714, 'centerline_recall': 0.9135730858468677, 'cldice': 0.9053317257374413, 'foreground_fraction': 0.127593994140625, 'thin_clean_centerline_recall': 0.9048867699642431}, {'experiment': 'L1_proxy_none_t0.5', 'id': '5', 'fold': 2, 'target': 'clean', 'dice': 0.6830619581313532, 'precision': 0.5401852904779424, 'recall': 0.928698752228164, 'centerline_precision': 0.5930476475876536, 'centerline_recall': 0.9286595394736842, 'cldice': 0.7238440614298062, 'foreground_fraction': 0.11405563354492188, 'thin_clean_centerline_recall': 0.8440073193046661}, {'experiment': 'L1_proxy_none_t0.5', 'id': '5', 'fold': 2, 'target': 'proxy', 'dice': 0.7482933274613736, 'precision': 0.6818957155757718, 'recall': 0.8290163867767251, 'centerline_precision': 0.66841474378184, 'centerline_recall': 0.9107326082495383, 'cldice': 0.7709820140771587, 'foreground_fraction': 0.11405563354492188, 'thin_clean_centerline_recall': 0.8440073193046661}, {'experiment': 'L1_proxy_none_t0.5', 'id': '6', 'fold': 0, 'target': 'clean', 'dice': 0.7880948915735874, 'precision': 0.7097543712480667, 'recall': 0.8858750777083402, 'centerline_precision': 0.8168056446440025, 'centerline_recall': 0.8033297529538131, 'cldice': 0.8100116542062413, 'foreground_fraction': 0.14551925659179688, 'thin_clean_centerline_recall': 0.6326113608500205}, {'experiment': 'L1_proxy_none_t0.5', 'id': '6', 'fold': 0, 'target': 'proxy', 'dice': 0.8318616272866044, 'precision': 0.8989173460560463, 'recall': 0.7741156285978734, 'centerline_precision': 0.9447081462475946, 'centerline_recall': 0.7755483938039297, 'cldice': 0.8518111902238319, 'foreground_fraction': 0.14551925659179688, 'thin_clean_centerline_recall': 0.6326113608500205}, {'experiment': 'L1_proxy_none_t0.5', 'id': '7', 'fold': 4, 'target': 'clean', 'dice': 0.7096058566893204, 'precision': 0.5825879305459991, 'recall': 0.9074517715294415, 'centerline_precision': 0.7739769150052466, 'centerline_recall': 0.8286121503652236, 'cldice': 0.8003632244018636, 'foreground_fraction': 0.09424972534179688, 'thin_clean_centerline_recall': 0.725769669327252}, {'experiment': 'L1_proxy_none_t0.5', 'id': '7', 'fold': 4, 'target': 'proxy', 'dice': 0.8185635088867224, 'precision': 0.8127656129841745, 'recall': 0.8244447181508396, 'centerline_precision': 0.9374606505771249, 'centerline_recall': 0.8099231306778477, 'cldice': 0.8690375556277714, 'foreground_fraction': 0.09424972534179688, 'thin_clean_centerline_recall': 0.725769669327252}, {'experiment': 'L1_proxy_none_t0.5', 'id': '8', 'fold': 4, 'target': 'clean', 'dice': 0.7152026831671482, 'precision': 0.5921481008617938, 'recall': 0.9028176553603582, 'centerline_precision': 0.712513567995038, 'centerline_recall': 0.8456578405816985, 'cldice': 0.7733971782298573, 'foreground_fraction': 0.11951446533203125, 'thin_clean_centerline_recall': 0.7543936731107206}, {'experiment': 'L1_proxy_none_t0.5', 'id': '8', 'fold': 4, 'target': 'proxy', 'dice': 0.810805666154073, 'precision': 0.8157357165655921, 'recall': 0.8059348491059884, 'centerline_precision': 0.9139401457590324, 'centerline_recall': 0.8199045852107076, 'cldice': 0.8643723428416396, 'foreground_fraction': 0.11951446533203125, 'thin_clean_centerline_recall': 0.7543936731107206}, {'experiment': 'L1_proxy_none_t0.5', 'id': '9', 'fold': 0, 'target': 'clean', 'dice': 0.7457200594811726, 'precision': 0.6266982824916688, 'recall': 0.9205497505412784, 'centerline_precision': 0.7535159141376758, 'centerline_recall': 0.8661471151216844, 'cldice': 0.8059153335457354, 'foreground_fraction': 0.119049072265625, 'thin_clean_centerline_recall': 0.7837777777777778}, {'experiment': 'L1_proxy_none_t0.5', 'id': '9', 'fold': 0, 'target': 'proxy', 'dice': 0.8398074342990454, 'precision': 0.8524416816200974, 'recall': 0.8275422278906275, 'centerline_precision': 0.9256846780162842, 'centerline_recall': 0.8523676880222841, 'cldice': 0.8875145905812739, 'foreground_fraction': 0.119049072265625, 'thin_clean_centerline_recall': 0.7837777777777778}, {'experiment': 'L1_proxy_none_t0.5', 'id': '10', 'fold': 0, 'target': 'clean', 'dice': 0.724745066393286, 'precision': 0.5949771139484462, 'recall': 0.9269093638581347, 'centerline_precision': 0.7078534031413612, 'centerline_recall': 0.8624441964285714, 'cldice': 0.7775393143677708, 'foreground_fraction': 0.126678466796875, 'thin_clean_centerline_recall': 0.7615571776155717}, {'experiment': 'L1_proxy_none_t0.5', 'id': '10', 'fold': 0, 'target': 'proxy', 'dice': 0.8329004064422074, 'precision': 0.8207359672368104, 'recall': 0.845430857993672, 'centerline_precision': 0.9027673896783844, 'centerline_recall': 0.842471248997058, 'cldice': 0.8715777355392382, 'foreground_fraction': 0.126678466796875, 'thin_clean_centerline_recall': 0.7615571776155717}, {'experiment': 'L1_proxy_none_t0.5', 'id': '11', 'fold': 2, 'target': 'clean', 'dice': 0.7327365224598085, 'precision': 0.5974276527331189, 'recall': 0.9472825532782706, 'centerline_precision': 0.7075457875457876, 'centerline_recall': 0.9312509340905695, 'cldice': 0.8041298433962456, 'foreground_fraction': 0.1186370849609375, 'thin_clean_centerline_recall': 0.8877928483353884}, {'experiment': 'L1_proxy_none_t0.5', 'id': '11', 'fold': 2, 'target': 'proxy', 'dice': 0.8528911424185296, 'precision': 0.8302250803858521, 'recall': 0.8768295581892892, 'centerline_precision': 0.9052014652014652, 'centerline_recall': 0.9179539492859224, 'cldice': 0.9115331071374626, 'foreground_fraction': 0.1186370849609375, 'thin_clean_centerline_recall': 0.8877928483353884}, {'experiment': 'L1_proxy_none_t0.5', 'id': '12', 'fold': 4, 'target': 'clean', 'dice': 0.654188031155804, 'precision': 0.6018542700443981, 'recall': 0.7164898371614007, 'centerline_precision': 0.7786950309572948, 'centerline_recall': 0.728629417382999, 'cldice': 0.7528307622823858, 'foreground_fraction': 0.116851806640625, 'thin_clean_centerline_recall': 0.690566037735849}, {'experiment': 'L1_proxy_none_t0.5', 'id': '12', 'fold': 4, 'target': 'proxy', 'dice': 0.7157793006505655, 'precision': 0.797891094280491, 'recall': 0.6489909718534254, 'centerline_precision': 0.8969677726623273, 'centerline_recall': 0.7130715824754609, 'cldice': 0.7945175092034662, 'foreground_fraction': 0.116851806640625, 'thin_clean_centerline_recall': 0.690566037735849}, {'experiment': 'L1_proxy_none_t0.5', 'id': '13', 'fold': 4, 'target': 'clean', 'dice': 0.7463404925185982, 'precision': 0.6428526899819214, 'recall': 0.8895406512831572, 'centerline_precision': 0.7454046639231824, 'centerline_recall': 0.8328368878827594, 'cldice': 0.7866989686144191, 'foreground_fraction': 0.12238311767578125, 'thin_clean_centerline_recall': 0.7438843587842846}, {'experiment': 'L1_proxy_none_t0.5', 'id': '13', 'fold': 4, 'target': 'proxy', 'dice': 0.8285221188030792, 'precision': 0.8753818340502463, 'recall': 0.7864243510402957, 'centerline_precision': 0.9410150891632373, 'centerline_recall': 0.8155995343422584, 'cldice': 0.8738302166686637, 'foreground_fraction': 0.12238311767578125, 'thin_clean_centerline_recall': 0.7438843587842846}, {'experiment': 'L1_proxy_none_t0.5', 'id': '14', 'fold': 1, 'target': 'clean', 'dice': 0.7446985079902098, 'precision': 0.6201076249007558, 'recall': 0.9319427258264098, 'centerline_precision': 0.8072173435784852, 'centerline_recall': 0.905984060516007, 'cldice': 0.8537537325224421, 'foreground_fraction': 0.12972640991210938, 'thin_clean_centerline_recall': 0.8431638418079096}, {'experiment': 'L1_proxy_none_t0.5', 'id': '14', 'fold': 1, 'target': 'proxy', 'dice': 0.8386868209733133, 'precision': 0.8353574264122092, 'recall': 0.8420428609538488, 'centerline_precision': 0.9102634467618003, 'centerline_recall': 0.8973363672868424, 'cldice': 0.9037536829147443, 'foreground_fraction': 0.12972640991210938, 'thin_clean_centerline_recall': 0.8431638418079096}, {'experiment': 'L1_proxy_none_t0.5', 'id': '15', 'fold': 3, 'target': 'clean', 'dice': 0.7201753505342164, 'precision': 0.579456112008616, 'recall': 0.9511623795633343, 'centerline_precision': 0.7854088376312038, 'centerline_recall': 0.9347737464329393, 'cldice': 0.8536065513458122, 'foreground_fraction': 0.1416778564453125, 'thin_clean_centerline_recall': 0.8942152569716525}, {'experiment': 'L1_proxy_none_t0.5', 'id': '15', 'fold': 3, 'target': 'proxy', 'dice': 0.8241596698049019, 'precision': 0.7849488422186321, 'recall': 0.8674938998988276, 'centerline_precision': 0.8845406245950499, 'centerline_recall': 0.9112187247780468, 'cldice': 0.8976815075352477, 'foreground_fraction': 0.1416778564453125, 'thin_clean_centerline_recall': 0.8942152569716525}, {'experiment': 'L1_proxy_none_t0.5', 'id': '16', 'fold': 3, 'target': 'clean', 'dice': 0.7318752081719184, 'precision': 0.6183091545772886, 'recall': 0.8965454710309185, 'centerline_precision': 0.7206832871652816, 'centerline_recall': 0.8068137318887874, 'cldice': 0.7613202057674426, 'foreground_fraction': 0.12200927734375, 'thin_clean_centerline_recall': 0.6936824605153782}, {'experiment': 'L1_proxy_none_t0.5', 'id': '16', 'fold': 3, 'target': 'proxy', 'dice': 0.8175919031049989, 'precision': 0.8410455227613807, 'recall': 0.7954108637156628, 'centerline_precision': 0.9253616497383811, 'centerline_recall': 0.7852668504134303, 'cldice': 0.849577600418731, 'foreground_fraction': 0.12200927734375, 'thin_clean_centerline_recall': 0.6936824605153782}, {'experiment': 'L1_proxy_none_t0.5', 'id': '17', 'fold': 1, 'target': 'clean', 'dice': 0.7483882651217885, 'precision': 0.6548871703853956, 'recall': 0.8730353219536927, 'centerline_precision': 0.756152125279642, 'centerline_recall': 0.8070809248554913, 'cldice': 0.7807869166396958, 'foreground_fraction': 0.120361328125, 'thin_clean_centerline_recall': 0.6954614797864226}, {'experiment': 'L1_proxy_none_t0.5', 'id': '17', 'fold': 1, 'target': 'proxy', 'dice': 0.8186821315504058, 'precision': 0.8769333164300203, 'recall': 0.767687697686033, 'centerline_precision': 0.9379194630872483, 'centerline_recall': 0.7935731749912679, 'cldice': 0.8597295880324749, 'foreground_fraction': 0.120361328125, 'thin_clean_centerline_recall': 0.6954614797864226}, {'experiment': 'L1_proxy_none_t0.5', 'id': '18', 'fold': 3, 'target': 'clean', 'dice': 0.733642506036278, 'precision': 0.6015438076791615, 'recall': 0.9400845760071221, 'centerline_precision': 0.7895649853215906, 'centerline_recall': 0.9265993265993266, 'cldice': 0.8526111149421298, 'foreground_fraction': 0.1339263916015625, 'thin_clean_centerline_recall': 0.878589192855528}, {'experiment': 'L1_proxy_none_t0.5', 'id': '18', 'fold': 3, 'target': 'proxy', 'dice': 0.8268861613637559, 'precision': 0.8097869431468611, 'recall': 0.8447230805799857, 'centerline_precision': 0.9124633039765145, 'centerline_recall': 0.8993147074327886, 'cldice': 0.9058412941224432, 'foreground_fraction': 0.1339263916015625, 'thin_clean_centerline_recall': 0.878589192855528}, {'experiment': 'L1_proxy_none_t0.5', 'id': '19', 'fold': 3, 'target': 'clean', 'dice': 0.7768061195584535, 'precision': 0.7384902679020326, 'recall': 0.8193155105781107, 'centerline_precision': 0.814350225790266, 'centerline_recall': 0.7622864651773982, 'cldice': 0.7874587196788883, 'foreground_fraction': 0.10622406005859375, 'thin_clean_centerline_recall': 0.564154272750189}, {'experiment': 'L1_proxy_none_t0.5', 'id': '19', 'fold': 3, 'target': 'proxy', 'dice': 0.7873134328361515, 'precision': 0.9092867916397328, 'recall': 0.6941931238690574, 'centerline_precision': 0.9456430841277805, 'centerline_recall': 0.7433175597902545, 'cldice': 0.8323617394609989, 'foreground_fraction': 0.10622406005859375, 'thin_clean_centerline_recall': 0.564154272750189}, {'experiment': 'L1_proxy_none_t0.5', 'id': '20', 'fold': 1, 'target': 'clean', 'dice': 0.7750527246755305, 'precision': 0.6888264103029204, 'recall': 0.8859553620032662, 'centerline_precision': 0.7911531087373322, 'centerline_recall': 0.842281078501551, 'cldice': 0.8159169177345993, 'foreground_fraction': 0.12618255615234375, 'thin_clean_centerline_recall': 0.7391824526420738}, {'experiment': 'L1_proxy_none_t0.5', 'id': '20', 'fold': 1, 'target': 'proxy', 'dice': 0.8352743089661354, 'precision': 0.8998427958159502, 'recall': 0.7793516966904064, 'centerline_precision': 0.951383182689674, 'centerline_recall': 0.8251895043731778, 'cldice': 0.8838044429137328, 'foreground_fraction': 0.12618255615234375, 'thin_clean_centerline_recall': 0.7391824526420738}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '1', 'fold': 2, 'target': 'clean', 'dice': 0.6452301303500328, 'precision': 0.48132690818346097, 'recall': 0.9783977610634949, 'centerline_precision': 0.6756300630063007, 'centerline_recall': 0.9596350566768039, 'cldice': 0.7929702480674794, 'foreground_fraction': 0.1773223876953125, 'thin_clean_centerline_recall': 0.929126213592233}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '1', 'fold': 2, 'target': 'proxy', 'dice': 0.7941337582792265, 'precision': 0.6861285603648567, 'recall': 0.9424940898345153, 'centerline_precision': 0.7802655265526552, 'centerline_recall': 0.9553461696026219, 'cldice': 0.8589751771278251, 'foreground_fraction': 0.1773223876953125, 'thin_clean_centerline_recall': 0.929126213592233}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '2', 'fold': 0, 'target': 'clean', 'dice': 0.6688792936784351, 'precision': 0.5158933320078535, 'recall': 0.95084970912922, 'centerline_precision': 0.6088490770901195, 'centerline_recall': 0.9073223039215687, 'cldice': 0.7287069974864133, 'foreground_fraction': 0.15349197387695312, 'thin_clean_centerline_recall': 0.8266550522648084}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '2', 'fold': 0, 'target': 'proxy', 'dice': 0.7997834897021598, 'precision': 0.7160822128886348, 'recall': 0.9056419927707057, 'centerline_precision': 0.7881378935939196, 'centerline_recall': 0.9059500959692899, 'cldice': 0.8429474793957321, 'foreground_fraction': 0.15349197387695312, 'thin_clean_centerline_recall': 0.8266550522648084}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '3', 'fold': 1, 'target': 'clean', 'dice': 0.7285979858504921, 'precision': 0.5886713347164592, 'recall': 0.9557879223832415, 'centerline_precision': 0.8156151969327292, 'centerline_recall': 0.9314066852367688, 'cldice': 0.8696736483466024, 'foreground_fraction': 0.176513671875, 'thin_clean_centerline_recall': 0.877572448551029}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '3', 'fold': 1, 'target': 'proxy', 'dice': 0.8513468725358561, 'precision': 0.804936030428769, 'recall': 0.9034370679409125, 'centerline_precision': 0.9130939932612989, 'centerline_recall': 0.9213122275751319, 'cldice': 0.9171847013617943, 'foreground_fraction': 0.176513671875, 'thin_clean_centerline_recall': 0.877572448551029}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '4', 'fold': 2, 'target': 'clean', 'dice': 0.6070804741116546, 'precision': 0.43898863226272106, 'recall': 0.9837747674888799, 'centerline_precision': 0.6734136346703092, 'centerline_recall': 0.9732898056091408, 'cldice': 0.7960469499858246, 'foreground_fraction': 0.16912841796875, 'thin_clean_centerline_recall': 0.9573301549463648}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '4', 'fold': 2, 'target': 'proxy', 'dice': 0.7692121538050966, 'precision': 0.645231865752436, 'recall': 0.9521701504460125, 'centerline_precision': 0.7881534831739724, 'centerline_recall': 0.963022041763341, 'cldice': 0.866856766532593, 'foreground_fraction': 0.16912841796875, 'thin_clean_centerline_recall': 0.9573301549463648}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '5', 'fold': 2, 'target': 'clean', 'dice': 0.5512472534581374, 'precision': 0.38332771598696774, 'recall': 0.9809671669254212, 'centerline_precision': 0.4644511347846225, 'centerline_recall': 0.9683388157894737, 'cldice': 0.627790642542143, 'foreground_fraction': 0.16977310180664062, 'thin_clean_centerline_recall': 0.929551692589204}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '5', 'fold': 2, 'target': 'proxy', 'dice': 0.6726388607489759, 'precision': 0.5221660487585664, 'recall': 0.9449436831618754, 'centerline_precision': 0.5424965261695229, 'centerline_recall': 0.9595731582187564, 'cldice': 0.6931304324288982, 'foreground_fraction': 0.16977310180664062, 'thin_clean_centerline_recall': 0.929551692589204}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '6', 'fold': 0, 'target': 'clean', 'dice': 0.7132164664175286, 'precision': 0.5717105133316229, 'recall': 0.9478127147204136, 'centerline_precision': 0.7280434045439131, 'centerline_recall': 0.8779806659505908, 'cldice': 0.7960130173710093, 'foreground_fraction': 0.19328689575195312, 'thin_clean_centerline_recall': 0.7680833673886391}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '6', 'fold': 0, 'target': 'proxy', 'dice': 0.829202030200462, 'precision': 0.7770628984191518, 'recall': 0.888841230783123, 'centerline_precision': 0.8780377529105912, 'centerline_recall': 0.856741865058738, 'cldice': 0.8672590964622493, 'foreground_fraction': 0.19328689575195312, 'thin_clean_centerline_recall': 0.7680833673886391}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '7', 'fold': 4, 'target': 'clean', 'dice': 0.6100698568646749, 'precision': 0.4481381798822868, 'recall': 0.9552389358214601, 'centerline_precision': 0.666199649737303, 'centerline_recall': 0.8993408159629431, 'cldice': 0.7654104760824373, 'foreground_fraction': 0.12897872924804688, 'thin_clean_centerline_recall': 0.8389395667046751}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '7', 'fold': 4, 'target': 'proxy', 'dice': 0.7659881721912977, 'precision': 0.6588979917778238, 'recall': 0.9146446606724966, 'centerline_precision': 0.8288966725043783, 'centerline_recall': 0.8852201257861635, 'cldice': 0.8561330446441222, 'foreground_fraction': 0.12897872924804688, 'thin_clean_centerline_recall': 0.8389395667046751}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '8', 'fold': 4, 'target': 'clean', 'dice': 0.6252604456608953, 'precision': 0.46441735185710237, 'recall': 0.9565428974645969, 'centerline_precision': 0.6182522715125601, 'centerline_recall': 0.9153519001234738, 'cldice': 0.7380240637725438, 'foreground_fraction': 0.1614532470703125, 'thin_clean_centerline_recall': 0.8644551845342706}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '8', 'fold': 4, 'target': 'proxy', 'dice': 0.7788748564870954, 'precision': 0.6812210566109064, 'recall': 0.9092113146857558, 'centerline_precision': 0.8212185996793159, 'centerline_recall': 0.8991518685396236, 'cldice': 0.8584200345470926, 'foreground_fraction': 0.1614532470703125, 'thin_clean_centerline_recall': 0.8644551845342706}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '9', 'fold': 0, 'target': 'clean', 'dice': 0.6606484833861764, 'precision': 0.5019193623315974, 'recall': 0.9662054033700461, 'centerline_precision': 0.6597858340859244, 'centerline_recall': 0.9297238173366147, 'cldice': 0.771833758595937, 'foreground_fraction': 0.15601730346679688, 'thin_clean_centerline_recall': 0.886}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '9', 'fold': 0, 'target': 'proxy', 'dice': 0.8142266516991837, 'precision': 0.7271082422553119, 'recall': 0.9250629918810465, 'centerline_precision': 0.852922203586634, 'centerline_recall': 0.9217402838572755, 'cldice': 0.8859969257303241, 'foreground_fraction': 0.15601730346679688, 'thin_clean_centerline_recall': 0.886}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '10', 'fold': 0, 'target': 'clean', 'dice': 0.636358022786075, 'precision': 0.4742303621554461, 'recall': 0.9669262525802215, 'centerline_precision': 0.6208256639495136, 'centerline_recall': 0.9236886160714286, 'cldice': 0.7425630255064738, 'foreground_fraction': 0.16579437255859375, 'thin_clean_centerline_recall': 0.8669099756690998}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '10', 'fold': 0, 'target': 'proxy', 'dice': 0.7905416116251116, 'precision': 0.6884634853435184, 'recall': 0.9281593150939884, 'centerline_precision': 0.8213252695240599, 'centerline_recall': 0.9109387536774539, 'cldice': 0.8638140691755403, 'foreground_fraction': 0.16579437255859375, 'thin_clean_centerline_recall': 0.8669099756690998}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '11', 'fold': 2, 'target': 'clean', 'dice': 0.6228011975084994, 'precision': 0.45620065906450774, 'recall': 0.9810849393290507, 'centerline_precision': 0.5950965824665676, 'centerline_recall': 0.9641309221342101, 'cldice': 0.7359426576550946, 'foreground_fraction': 0.16090774536132812, 'thin_clean_centerline_recall': 0.9410604192355118}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '11', 'fold': 2, 'target': 'proxy', 'dice': 0.7866197576369763, 'precision': 0.6678836442948247, 'recall': 0.9567018711583523, 'centerline_precision': 0.7860326894502229, 'centerline_recall': 0.9610900612066453, 'cldice': 0.8647912178238326, 'foreground_fraction': 0.16090774536132812, 'thin_clean_centerline_recall': 0.9410604192355118}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '12', 'fold': 4, 'target': 'clean', 'dice': 0.6152852714963619, 'precision': 0.48772664194553766, 'recall': 0.8331973106369749, 'centerline_precision': 0.6727748691099477, 'centerline_recall': 0.8453915950334289, 'cldice': 0.7492699030421358, 'foreground_fraction': 0.16768264770507812, 'thin_clean_centerline_recall': 0.8216981132075472}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '12', 'fold': 4, 'target': 'proxy', 'dice': 0.732298418222022, 'precision': 0.6798462133448597, 'recall': 0.7935209771640999, 'centerline_precision': 0.7989528795811518, 'centerline_recall': 0.835647593966962, 'cldice': 0.8168883617973741, 'foreground_fraction': 0.16768264770507812, 'thin_clean_centerline_recall': 0.8216981132075472}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '13', 'fold': 4, 'target': 'clean', 'dice': 0.6676312775069819, 'precision': 0.5131246669601279, 'recall': 0.9552728056933362, 'centerline_precision': 0.6657885283199617, 'centerline_recall': 0.9154056952222089, 'cldice': 0.7708940515509686, 'foreground_fraction': 0.16465377807617188, 'thin_clean_centerline_recall': 0.868421052631579}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '13', 'fold': 4, 'target': 'proxy', 'dice': 0.8218931460306034, 'precision': 0.7509440956374672, 'recall': 0.9076475035703284, 'centerline_precision': 0.8769009699437194, 'centerline_recall': 0.9121071012805588, 'cldice': 0.894157622506563, 'foreground_fraction': 0.16465377807617188, 'thin_clean_centerline_recall': 0.868421052631579}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '14', 'fold': 1, 'target': 'clean', 'dice': 0.6506325022410845, 'precision': 0.487333553427974, 'recall': 0.9785221849036592, 'centerline_precision': 0.6952161383285302, 'centerline_recall': 0.9648791030663245, 'cldice': 0.8081458307464854, 'foreground_fraction': 0.17332077026367188, 'thin_clean_centerline_recall': 0.9412429378531073}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '14', 'fold': 1, 'target': 'proxy', 'dice': 0.8052846966101586, 'precision': 0.7016176956091119, 'recall': 0.944897293772416, 'centerline_precision': 0.8129106628242075, 'centerline_recall': 0.957836969615848, 'cldice': 0.8794430912495468, 'foreground_fraction': 0.17332077026367188, 'thin_clean_centerline_recall': 0.9412429378531073}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '15', 'fold': 3, 'target': 'clean', 'dice': 0.6231611454244481, 'precision': 0.45659686510058833, 'recall': 0.981039512065765, 'centerline_precision': 0.6971615967431867, 'centerline_recall': 0.9732300584318522, 'cldice': 0.8123826761619156, 'foreground_fraction': 0.18544769287109375, 'thin_clean_centerline_recall': 0.9582853191979719}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '15', 'fold': 3, 'target': 'proxy', 'dice': 0.7750182437365908, 'precision': 0.6553873369811165, 'recall': 0.9480747485568053, 'centerline_precision': 0.792943571186249, 'centerline_recall': 0.966908797417272, 'cldice': 0.8713277642076859, 'foreground_fraction': 0.18544769287109375, 'thin_clean_centerline_recall': 0.9582853191979719}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '16', 'fold': 3, 'target': 'clean', 'dice': 0.6541914752864345, 'precision': 0.5012551897267549, 'recall': 0.9414271466134736, 'centerline_precision': 0.6446091644204852, 'centerline_recall': 0.8660749249445242, 'cldice': 0.7391086430634054, 'foreground_fraction': 0.1580352783203125, 'thin_clean_centerline_recall': 0.786783042394015}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '16', 'fold': 3, 'target': 'proxy', 'dice': 0.7950615971403576, 'precision': 0.7220478903157285, 'recall': 0.884502794287235, 'centerline_precision': 0.8575471698113207, 'centerline_recall': 0.8520420947131044, 'cldice': 0.8547857687730499, 'foreground_fraction': 0.1580352783203125, 'thin_clean_centerline_recall': 0.786783042394015}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '17', 'fold': 1, 'target': 'clean', 'dice': 0.6730915392337684, 'precision': 0.5256659860392231, 'recall': 0.935440256886936, 'centerline_precision': 0.6958333333333333, 'centerline_recall': 0.8786127167630058, 'cldice': 0.7766134830429394, 'foreground_fraction': 0.16066741943359375, 'thin_clean_centerline_recall': 0.8077803203661327}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '17', 'fold': 1, 'target': 'proxy', 'dice': 0.8098259979531605, 'precision': 0.7514126976589581, 'recall': 0.8780866766550136, 'centerline_precision': 0.8801470588235294, 'centerline_recall': 0.8693677960181627, 'cldice': 0.8747242203559544, 'foreground_fraction': 0.16066741943359375, 'thin_clean_centerline_recall': 0.8077803203661327}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '18', 'fold': 3, 'target': 'clean', 'dice': 0.6336310078413577, 'precision': 0.4672924859402089, 'recall': 0.9838415312708658, 'centerline_precision': 0.6844665817213019, 'centerline_recall': 0.9764309764309764, 'cldice': 0.8047869893529312, 'foreground_fraction': 0.18042755126953125, 'thin_clean_centerline_recall': 0.9604340945059914}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '18', 'fold': 3, 'target': 'proxy', 'dice': 0.7888430466687393, 'precision': 0.6750813987906465, 'recall': 0.9487164250059424, 'centerline_precision': 0.821383064983204, 'centerline_recall': 0.9682393252503954, 'cldice': 0.8887856890387131, 'foreground_fraction': 0.18042755126953125, 'thin_clean_centerline_recall': 0.9604340945059914}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '19', 'fold': 3, 'target': 'clean', 'dice': 0.7275147549401342, 'precision': 0.6088368721982123, 'recall': 0.9036615004581856, 'centerline_precision': 0.7293189125986553, 'centerline_recall': 0.826149802890933, 'cldice': 0.7747204040659367, 'foreground_fraction': 0.14210891723632812, 'thin_clean_centerline_recall': 0.6735568439626922}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '19', 'fold': 3, 'target': 'proxy', 'dice': 0.8120498596174915, 'precision': 0.8035594448769227, 'recall': 0.8207216099139113, 'centerline_precision': 0.867874890382929, 'centerline_recall': 0.8136590356823123, 'cldice': 0.8398929518529008, 'foreground_fraction': 0.14210891723632812, 'thin_clean_centerline_recall': 0.6735568439626922}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '20', 'fold': 1, 'target': 'clean', 'dice': 0.7067575406036735, 'precision': 0.5635493270431525, 'recall': 0.9475464655105373, 'centerline_precision': 0.716734492539154, 'centerline_recall': 0.9016941064185159, 'cldice': 0.798645387514342, 'foreground_fraction': 0.16495513916015625, 'thin_clean_centerline_recall': 0.8360917248255234}, {'experiment': 'L1_proxy_d4_t0.1', 'id': '20', 'fold': 1, 'target': 'proxy', 'dice': 0.8363337181032047, 'precision': 0.7874982655751352, 'recall': 0.8916265186426476, 'centerline_precision': 0.8916019237883833, 'centerline_recall': 0.8906122448979592, 'cldice': 0.8911068095545619, 'foreground_fraction': 0.16495513916015625, 'thin_clean_centerline_recall': 0.8360917248255234}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '1', 'fold': 2, 'target': 'clean', 'dice': 0.74343689685254, 'precision': 0.6099666470472828, 'recall': 0.95167920237887, 'centerline_precision': 0.7964434864609996, 'centerline_recall': 0.9319878352225601, 'cldice': 0.8589009369500314, 'foreground_fraction': 0.13610458374023438, 'thin_clean_centerline_recall': 0.8815533980582524}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '1', 'fold': 2, 'target': 'proxy', 'dice': 0.8475093140007084, 'precision': 0.8256677597466296, 'recall': 0.8705378250591016, 'centerline_precision': 0.8966725043782837, 'centerline_recall': 0.9161545814556875, 'cldice': 0.9063088579941334, 'foreground_fraction': 0.13610458374023438, 'thin_clean_centerline_recall': 0.8815533980582524}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '2', 'fold': 0, 'target': 'clean', 'dice': 0.7490851172761162, 'precision': 0.6390902031836417, 'recall': 0.9048142549585452, 'centerline_precision': 0.7297955209347614, 'centerline_recall': 0.8566176470588235, 'cldice': 0.7881373334386389, 'foreground_fraction': 0.1179046630859375, 'thin_clean_centerline_recall': 0.735191637630662}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '2', 'fold': 0, 'target': 'proxy', 'dice': 0.8364714697960294, 'precision': 0.8487446615762909, 'recall': 0.8245481691026245, 'centerline_precision': 0.9118792599805258, 'centerline_recall': 0.8501402628082091, 'cldice': 0.8799281321267726, 'foreground_fraction': 0.1179046630859375, 'thin_clean_centerline_recall': 0.735191637630662}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '3', 'fold': 1, 'target': 'clean', 'dice': 0.7921393779301092, 'precision': 0.7012056660899654, 'recall': 0.9101722867469034, 'centerline_precision': 0.8885761801202507, 'centerline_recall': 0.8862581244196843, 'cldice': 0.8874156384998604, 'foreground_fraction': 0.14111328125, 'thin_clean_centerline_recall': 0.8017639647207055}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '3', 'fold': 1, 'target': 'proxy', 'dice': 0.8530663905191155, 'precision': 0.9018977076124568, 'recall': 0.8092512188614258, 'centerline_precision': 0.9646923372137649, 'centerline_recall': 0.8683184216563432, 'cldice': 0.9139718614087894, 'foreground_fraction': 0.14111328125, 'thin_clean_centerline_recall': 0.8017639647207055}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '4', 'fold': 2, 'target': 'clean', 'dice': 0.6979620879026145, 'precision': 0.5460930139606589, 'recall': 0.966841892438334, 'centerline_precision': 0.7775565353434789, 'centerline_recall': 0.9496958005638819, 'cldice': 0.8550484037166287, 'foreground_fraction': 0.13361740112304688, 'thin_clean_centerline_recall': 0.9194278903456495}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '4', 'fold': 2, 'target': 'proxy', 'dice': 0.8264818429103188, 'precision': 0.7676934935906586, 'recall': 0.8950206363999468, 'centerline_precision': 0.8938984497226568, 'centerline_recall': 0.9253190255220418, 'cldice': 0.9093373977201663, 'foreground_fraction': 0.13361740112304688, 'thin_clean_centerline_recall': 0.9194278903456495}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '5', 'fold': 2, 'target': 'clean', 'dice': 0.6684999592608398, 'precision': 0.5176177407652756, 'recall': 0.9435340118452072, 'centerline_precision': 0.5784871010056843, 'centerline_recall': 0.9389391447368421, 'cldice': 0.7159019232513464, 'foreground_fraction': 0.12092971801757812, 'thin_clean_centerline_recall': 0.8659652333028363}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '5', 'fold': 2, 'target': 'proxy', 'dice': 0.7544320886777376, 'precision': 0.6698526860351408, 'recall': 0.863457081283292, 'centerline_precision': 0.654860807462469, 'centerline_recall': 0.9250974758875437, 'cldice': 0.7668684501678097, 'foreground_fraction': 0.12092971801757812, 'thin_clean_centerline_recall': 0.8659652333028363}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '6', 'fold': 0, 'target': 'clean', 'dice': 0.7847310025432446, 'precision': 0.6934568645174249, 'recall': 0.9036743775152962, 'centerline_precision': 0.8205859476276899, 'centerline_recall': 0.8193340494092374, 'cldice': 0.8199595206752908, 'foreground_fraction': 0.1519317626953125, 'thin_clean_centerline_recall': 0.6591744993870045}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '6', 'fold': 0, 'target': 'proxy', 'dice': 0.8463833580982509, 'precision': 0.8938686351310636, 'recall': 0.803688737386279, 'centerline_precision': 0.9551464869069225, 'centerline_recall': 0.7943653186401913, 'cldice': 0.8673679606095626, 'foreground_fraction': 0.1519317626953125, 'thin_clean_centerline_recall': 0.6591744993870045}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '7', 'fold': 4, 'target': 'clean', 'dice': 0.6991454623580963, 'precision': 0.5625768285187461, 'recall': 0.9232757533728407, 'centerline_precision': 0.7768645983151838, 'centerline_recall': 0.8499910921076074, 'cldice': 0.8117843423101077, 'foreground_fraction': 0.09930419921875, 'thin_clean_centerline_recall': 0.7599771949828963}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '7', 'fold': 4, 'target': 'proxy', 'dice': 0.8201393161209388, 'precision': 0.7937538414259373, 'recall': 0.8483392864474278, 'centerline_precision': 0.9412369015820834, 'centerline_recall': 0.8301886792452831, 'cldice': 0.8822320605942561, 'foreground_fraction': 0.09930419921875, 'thin_clean_centerline_recall': 0.7599771949828963}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '8', 'fold': 4, 'target': 'clean', 'dice': 0.7074148673837934, 'precision': 0.5768404248787377, 'recall': 0.914399727480656, 'centerline_precision': 0.7054275318374773, 'centerline_recall': 0.862258197283578, 'cldice': 0.7759982254318495, 'foreground_fraction': 0.12425994873046875, 'thin_clean_centerline_recall': 0.7807557117750439}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '8', 'fold': 4, 'target': 'proxy', 'dice': 0.8162712919035292, 'precision': 0.805458341008166, 'recall': 0.8273785121881997, 'centerline_precision': 0.9085809581564585, 'centerline_recall': 0.836072091174132, 'cldice': 0.8708197678367448, 'foreground_fraction': 0.12425994873046875, 'thin_clean_centerline_recall': 0.7807557117750439}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '9', 'fold': 0, 'target': 'clean', 'dice': 0.7390921523450438, 'precision': 0.6138860361622008, 'recall': 0.9284571213404876, 'centerline_precision': 0.7550655542312277, 'centerline_recall': 0.8732567678424938, 'cldice': 0.8098717268181141, 'foreground_fraction': 0.12257766723632812, 'thin_clean_centerline_recall': 0.7951111111111111}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '9', 'fold': 0, 'target': 'proxy', 'dice': 0.8457996266337184, 'precision': 0.8459838795008247, 'recall': 0.8456154540081501, 'centerline_precision': 0.9372765196662693, 'centerline_recall': 0.8604589468099217, 'cldice': 0.8972265186073615, 'foreground_fraction': 0.12257766723632812, 'thin_clean_centerline_recall': 0.7951111111111111}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '10', 'fold': 0, 'target': 'clean', 'dice': 0.7178549009062544, 'precision': 0.5810853973692616, 'recall': 0.9388252955526365, 'centerline_precision': 0.7186095295175308, 'centerline_recall': 0.8807198660714286, 'cldice': 0.7914488289152798, 'foreground_fraction': 0.13137435913085938, 'thin_clean_centerline_recall': 0.7931873479318735}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '10', 'fold': 0, 'target': 'proxy', 'dice': 0.8392999085144213, 'precision': 0.8124800371671651, 'recall': 0.8679508654383026, 'centerline_precision': 0.9229847168115073, 'centerline_recall': 0.8629312650441294, 'cldice': 0.8919483081919515, 'foreground_fraction': 0.13137435913085938, 'thin_clean_centerline_recall': 0.7931873479318735}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '11', 'fold': 2, 'target': 'clean', 'dice': 0.7218186709717479, 'precision': 0.5792314335060449, 'recall': 0.9575303354746609, 'centerline_precision': 0.6989825581395349, 'centerline_recall': 0.9351367508593633, 'cldice': 0.7999957833267503, 'foreground_fraction': 0.123687744140625, 'thin_clean_centerline_recall': 0.8934648581997534}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '11', 'fold': 2, 'target': 'proxy', 'dice': 0.8545845387987028, 'precision': 0.815352825067851, 'recall': 0.8977824566169729, 'centerline_precision': 0.9039244186046511, 'centerline_recall': 0.9258233751092976, 'cldice': 0.914742850300274, 'foreground_fraction': 0.123687744140625, 'thin_clean_centerline_recall': 0.8934648581997534}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '12', 'fold': 4, 'target': 'clean', 'dice': 0.6514024041219556, 'precision': 0.5882518796992481, 'recall': 0.7297423341494695, 'centerline_precision': 0.7712324393358876, 'centerline_recall': 0.7470152817574021, 'cldice': 0.7589307198907975, 'foreground_fraction': 0.12176513671875, 'thin_clean_centerline_recall': 0.7173584905660377}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '12', 'fold': 4, 'target': 'proxy', 'dice': 0.7274216728949017, 'precision': 0.7928258145363408, 'recall': 0.6719861922464153, 'centerline_precision': 0.9007024265644955, 'centerline_recall': 0.732942303088341, 'cldice': 0.8082086624351047, 'foreground_fraction': 0.12176513671875, 'thin_clean_centerline_recall': 0.7173584905660377}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '13', 'fold': 4, 'target': 'clean', 'dice': 0.740590925079598, 'precision': 0.6252412769116555, 'recall': 0.9081302566314428, 'centerline_precision': 0.7395286195286195, 'centerline_recall': 0.8529727153580364, 'cldice': 0.7922099917476088, 'foreground_fraction': 0.12845993041992188, 'thin_clean_centerline_recall': 0.7720533728687917}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '13', 'fold': 4, 'target': 'proxy', 'dice': 0.8393335831438052, 'precision': 0.8647067557535264, 'recall': 0.815407017445605, 'centerline_precision': 0.941010101010101, 'centerline_recall': 0.839813736903376, 'cldice': 0.8875366474418368, 'foreground_fraction': 0.12845993041992188, 'thin_clean_centerline_recall': 0.7720533728687917}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '14', 'fold': 1, 'target': 'clean', 'dice': 0.7410372312899123, 'precision': 0.6077998528329654, 'recall': 0.9490896234753403, 'centerline_precision': 0.8127705627705628, 'centerline_recall': 0.9269215183033905, 'cldice': 0.8661009982990958, 'foreground_fraction': 0.13478851318359375, 'thin_clean_centerline_recall': 0.8777401129943503}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '14', 'fold': 1, 'target': 'proxy', 'dice': 0.8511531612400991, 'precision': 0.8319182656931001, 'recall': 0.871298574265643, 'centerline_precision': 0.917478354978355, 'centerline_recall': 0.9190202114844064, 'cldice': 0.9182486359884345, 'foreground_fraction': 0.13478851318359375, 'thin_clean_centerline_recall': 0.8777401129943503}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '15', 'fold': 3, 'target': 'clean', 'dice': 0.712356637734897, 'precision': 0.5664779102841784, 'recall': 0.9594272076372315, 'centerline_precision': 0.7964475560741605, 'centerline_recall': 0.9474113330615572, 'cldice': 0.8653950678174568, 'foreground_fraction': 0.14618301391601562, 'thin_clean_centerline_recall': 0.915187831297534}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '15', 'fold': 3, 'target': 'proxy', 'dice': 0.8264490386088306, 'precision': 0.7756060645599019, 'recall': 0.8844254002261501, 'centerline_precision': 0.8923894723194606, 'centerline_recall': 0.9252085014796879, 'cldice': 0.9085026923695004, 'foreground_fraction': 0.14618301391601562, 'thin_clean_centerline_recall': 0.915187831297534}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '16', 'fold': 3, 'target': 'clean', 'dice': 0.7275498539316515, 'precision': 0.6065410098932018, 'recall': 0.9088765980596609, 'centerline_precision': 0.7219690063810392, 'centerline_recall': 0.8189531392768568, 'cldice': 0.7674090295897558, 'foreground_fraction': 0.12608718872070312, 'thin_clean_centerline_recall': 0.7121778886118039}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '16', 'fold': 3, 'target': 'proxy', 'dice': 0.8247996171793504, 'precision': 0.8343569418812211, 'recall': 0.8154587657825483, 'centerline_precision': 0.9302643573381951, 'centerline_recall': 0.7980456026058632, 'cldice': 0.8590974962138686, 'foreground_fraction': 0.12608718872070312, 'thin_clean_centerline_recall': 0.7121778886118039}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '17', 'fold': 1, 'target': 'clean', 'dice': 0.7435915492962261, 'precision': 0.63738983460099, 'recall': 0.8922595910089572, 'centerline_precision': 0.7649661274713121, 'centerline_recall': 0.8273121387283237, 'cldice': 0.7949185470998165, 'foreground_fraction': 0.1263885498046875, 'thin_clean_centerline_recall': 0.7269260106788711}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '17', 'fold': 1, 'target': 'proxy', 'dice': 0.8286639488827758, 'precision': 0.8650549317879995, 'recall': 0.795211142555907, 'centerline_precision': 0.9424858288400387, 'centerline_recall': 0.8112702293631389, 'cldice': 0.8719692695664336, 'foreground_fraction': 0.1263885498046875, 'thin_clean_centerline_recall': 0.7269260106788711}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '18', 'fold': 3, 'target': 'clean', 'dice': 0.7186920726111476, 'precision': 0.5751015608295916, 'recall': 0.9578455375027821, 'centerline_precision': 0.780078125, 'centerline_recall': 0.9488215488215488, 'cldice': 0.8562150204219496, 'foreground_fraction': 0.142730712890625, 'thin_clean_centerline_recall': 0.9154420076870902}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '18', 'fold': 3, 'target': 'proxy', 'dice': 0.8335209365153176, 'precision': 0.7916399401325636, 'recall': 0.8800808176848111, 'centerline_precision': 0.9102864583333333, 'centerline_recall': 0.9268581971534001, 'cldice': 0.9184975860710928, 'foreground_fraction': 0.142730712890625, 'thin_clean_centerline_recall': 0.9154420076870902}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '19', 'fold': 3, 'target': 'clean', 'dice': 0.7796979432078216, 'precision': 0.72652513505144, 'recall': 0.8412685764373082, 'centerline_precision': 0.8193708609271523, 'centerline_recall': 0.7754270696452037, 'cldice': 0.7967935416286684, 'foreground_fraction': 0.11086654663085938, 'thin_clean_centerline_recall': 0.5863372825812957}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '19', 'fold': 3, 'target': 'proxy', 'dice': 0.802722126432698, 'precision': 0.9050682999002168, 'recall': 0.7211712452706037, 'centerline_precision': 0.9529801324503311, 'centerline_recall': 0.7581532165238521, 'cldice': 0.8444753334200209, 'foreground_fraction': 0.11086654663085938, 'thin_clean_centerline_recall': 0.5863372825812957}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '20', 'fold': 1, 'target': 'clean', 'dice': 0.7712537516378343, 'precision': 0.672352482002949, 'recall': 0.9042693833112995, 'centerline_precision': 0.7956871843865156, 'centerline_recall': 0.854688618468146, 'cldice': 0.824133241871105, 'foreground_fraction': 0.13194656372070312, 'thin_clean_centerline_recall': 0.7583250249252244}, {'experiment': 'L1_proxy_d4_t0.4', 'id': '20', 'fold': 1, 'target': 'proxy', 'dice': 0.8463747406605483, 'precision': 0.8904565035126775, 'recall': 0.8064516129032258, 'centerline_precision': 0.9546881397570629, 'centerline_recall': 0.8377842565597667, 'cldice': 0.8924240005662263, 'foreground_fraction': 0.13194656372070312, 'thin_clean_centerline_recall': 0.7583250249252244}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '1', 'fold': 2, 'target': 'clean', 'dice': 0.7624473619028231, 'precision': 0.640297176820208, 'recall': 0.9421899597691097, 'centerline_precision': 0.818973862536302, 'centerline_recall': 0.9213436549626762, 'cldice': 0.8671479362139327, 'foreground_fraction': 0.12836456298828125, 'thin_clean_centerline_recall': 0.8638349514563107}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '1', 'fold': 2, 'target': 'proxy', 'dice': 0.8498444213959848, 'precision': 0.8522436849925705, 'recall': 0.8474586288416076, 'centerline_precision': 0.916194164016042, 'centerline_recall': 0.9019527516045337, 'cldice': 0.9090176818370339, 'foreground_fraction': 0.12836456298828125, 'thin_clean_centerline_recall': 0.8638349514563107}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '2', 'fold': 0, 'target': 'clean', 'dice': 0.7632850241550543, 'precision': 0.6680416623698051, 'recall': 0.8902020063212862, 'centerline_precision': 0.7521497218007082, 'centerline_recall': 0.842984068627451, 'cldice': 0.7949806298446944, 'foreground_fraction': 0.11097335815429688, 'thin_clean_centerline_recall': 0.7102206736353078}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '2', 'fold': 0, 'target': 'proxy', 'dice': 0.8355827012119732, 'precision': 0.8747035165515108, 'recall': 0.7998114097123998, 'centerline_precision': 0.9345810150059012, 'centerline_recall': 0.8369998523549387, 'cldice': 0.8831029799266157, 'foreground_fraction': 0.11097335815429688, 'thin_clean_centerline_recall': 0.7102206736353078}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '3', 'fold': 1, 'target': 'clean', 'dice': 0.8015667028458597, 'precision': 0.7264355362946913, 'recall': 0.8940313695217376, 'centerline_precision': 0.9004720692368214, 'centerline_recall': 0.8722144846796658, 'cldice': 0.8861180563507876, 'foreground_fraction': 0.13379669189453125, 'thin_clean_centerline_recall': 0.7786644267114657}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '3', 'fold': 1, 'target': 'proxy', 'dice': 0.8442615430992461, 'precision': 0.9183155613845013, 'recall': 0.7812598539791884, 'centerline_precision': 0.9718069761342775, 'centerline_recall': 0.8526038082128928, 'cldice': 0.9083111498887868, 'foreground_fraction': 0.13379669189453125, 'thin_clean_centerline_recall': 0.7786644267114657}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '4', 'fold': 2, 'target': 'clean', 'dice': 0.7151364390175728, 'precision': 0.5701322115384615, 'recall': 0.9590578245046503, 'centerline_precision': 0.7917880023296447, 'centerline_recall': 0.9399020626205669, 'cldice': 0.8595108231093861, 'foreground_fraction': 0.126953125, 'thin_clean_centerline_recall': 0.9036948748510131}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '4', 'fold': 2, 'target': 'proxy', 'dice': 0.8308698123936724, 'precision': 0.7904747596153846, 'recall': 0.875615763546798, 'centerline_precision': 0.9075422248107163, 'centerline_recall': 0.9108178654292344, 'cldice': 0.9091770947083602, 'foreground_fraction': 0.126953125, 'thin_clean_centerline_recall': 0.9036948748510131}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '5', 'fold': 2, 'target': 'clean', 'dice': 0.6940782410793703, 'precision': 0.5538498440873111, 'recall': 0.9293887643033754, 'centerline_precision': 0.6050691244239631, 'centerline_recall': 0.9278371710526315, 'cldice': 0.7324722017953178, 'foreground_fraction': 0.11132431030273438, 'thin_clean_centerline_recall': 0.843092406221409}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '5', 'fold': 2, 'target': 'proxy', 'dice': 0.7644674204110301, 'precision': 0.7043484220265223, 'recall': 0.8358069369332737, 'centerline_precision': 0.6820276497695853, 'centerline_recall': 0.9127847321978247, 'cldice': 0.7807118037025178, 'foreground_fraction': 0.11132431030273438, 'thin_clean_centerline_recall': 0.843092406221409}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '6', 'fold': 0, 'target': 'clean', 'dice': 0.7951140208272257, 'precision': 0.7197327252479185, 'recall': 0.8881327094853254, 'centerline_precision': 0.8304232804232804, 'centerline_recall': 0.8024704618689581, 'cldice': 0.8162076148966483, 'foreground_fraction': 0.14386749267578125, 'thin_clean_centerline_recall': 0.6301593788312219}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '6', 'fold': 0, 'target': 'proxy', 'dice': 0.8395459145726311, 'precision': 0.9128175213448587, 'recall': 0.7771632390455335, 'centerline_precision': 0.9641534391534392, 'centerline_recall': 0.7771078074643933, 'cldice': 0.8605844374186858, 'foreground_fraction': 0.14386749267578125, 'thin_clean_centerline_recall': 0.6301593788312219}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '7', 'fold': 4, 'target': 'clean', 'dice': 0.7153248552632111, 'precision': 0.5886952272357061, 'recall': 0.9113604841760181, 'centerline_precision': 0.7889713679745494, 'centerline_recall': 0.8311063602351684, 'cldice': 0.8094909405262894, 'foreground_fraction': 0.0936737060546875, 'thin_clean_centerline_recall': 0.7297605473204105}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '7', 'fold': 4, 'target': 'proxy', 'dice': 0.8217038415148903, 'precision': 0.8183743280664604, 'recall': 0.8250605575399269, 'centerline_precision': 0.9522799575821845, 'centerline_recall': 0.8141160027952481, 'cldice': 0.8777945262546709, 'foreground_fraction': 0.0936737060546875, 'thin_clean_centerline_recall': 0.7297605473204105}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '8', 'fold': 4, 'target': 'clean', 'dice': 0.7219609297247869, 'precision': 0.6026651896259612, 'recall': 0.9001411260888608, 'centerline_precision': 0.7268790596968759, 'centerline_recall': 0.8467553848264509, 'cldice': 0.7822512528980085, 'foreground_fraction': 0.1170806884765625, 'thin_clean_centerline_recall': 0.7563708260105448}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '8', 'fold': 4, 'target': 'proxy', 'dice': 0.8167235549575874, 'precision': 0.8302815065815197, 'recall': 0.8036012740058655, 'centerline_precision': 0.9245283018867925, 'centerline_recall': 0.8193745030479724, 'cldice': 0.8687811221688057, 'foreground_fraction': 0.1170806884765625, 'thin_clean_centerline_recall': 0.7563708260105448}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '9', 'fold': 0, 'target': 'clean', 'dice': 0.7542425416509991, 'precision': 0.6409099888062159, 'recall': 0.9162665913583733, 'centerline_precision': 0.7698182373606232, 'centerline_recall': 0.8564397046759639, 'cldice': 0.810822055735695, 'foreground_fraction': 0.11586761474609375, 'thin_clean_centerline_recall': 0.7682222222222223}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '9', 'fold': 0, 'target': 'proxy', 'dice': 0.8454759200910966, 'precision': 0.8701521037729637, 'recall': 0.8221606992876473, 'centerline_precision': 0.9471513670383381, 'centerline_recall': 0.8433479241278684, 'cldice': 0.8922406651235106, 'foreground_fraction': 0.11586761474609375, 'thin_clean_centerline_recall': 0.7682222222222223}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '10', 'fold': 0, 'target': 'clean', 'dice': 0.7343570342351279, 'precision': 0.6072195301478256, 'recall': 0.9288328016513417, 'centerline_precision': 0.7317223333846391, 'centerline_recall': 0.8657924107142857, 'cldice': 0.7931315129763069, 'foreground_fraction': 0.12438201904296875, 'thin_clean_centerline_recall': 0.7671532846715329}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '10', 'fold': 0, 'target': 'proxy', 'dice': 0.8420825365495002, 'precision': 0.8373305526590198, 'recall': 0.8468887648117129, 'centerline_precision': 0.9330460212405726, 'centerline_recall': 0.845145760898636, 'cldice': 0.8869233313249841, 'foreground_fraction': 0.12438201904296875, 'thin_clean_centerline_recall': 0.7671532846715329}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '11', 'fold': 2, 'target': 'clean', 'dice': 0.7399924283186085, 'precision': 0.6073659765152258, 'recall': 0.9467217293769756, 'centerline_precision': 0.713346286055896, 'centerline_recall': 0.9221342101330144, 'cldice': 0.804413156349242, 'foreground_fraction': 0.11662673950195312, 'thin_clean_centerline_recall': 0.8725030826140567}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '11', 'fold': 2, 'target': 'proxy', 'dice': 0.8590469843387887, 'precision': 0.8432276845582704, 'recall': 0.8754711855197473, 'centerline_precision': 0.9197429382752952, 'centerline_recall': 0.9122704750801516, 'cldice': 0.9159914672406659, 'foreground_fraction': 0.11662673950195312, 'thin_clean_centerline_recall': 0.8725030826140567}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '12', 'fold': 4, 'target': 'clean', 'dice': 0.6555314533628787, 'precision': 0.6127952955490216, 'recall': 0.7046752943919785, 'centerline_precision': 0.7900543388769965, 'centerline_recall': 0.7245702005730659, 'cldice': 0.7558966804955826, 'foreground_fraction': 0.11287307739257812, 'thin_clean_centerline_recall': 0.6939622641509434}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '12', 'fold': 4, 'target': 'proxy', 'dice': 0.7192374607805034, 'precision': 0.8173307648112474, 'recall': 0.6421667551779076, 'centerline_precision': 0.9189856742960646, 'centerline_recall': 0.7075652382092411, 'cldice': 0.7995351544731362, 'foreground_fraction': 0.11287307739257812, 'thin_clean_centerline_recall': 0.6939622641509434}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '13', 'fold': 4, 'target': 'clean', 'dice': 0.7528984190446067, 'precision': 0.6505259852410111, 'recall': 0.8935087340953203, 'centerline_precision': 0.7501037775010377, 'centerline_recall': 0.8353389729536519, 'cldice': 0.7904302049716665, 'foreground_fraction': 0.12147903442382812, 'thin_clean_centerline_recall': 0.7457375833951075}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '13', 'fold': 4, 'target': 'proxy', 'dice': 0.834004381550367, 'precision': 0.8846286701208981, 'recall': 0.7888605751729159, 'centerline_precision': 0.9489414694894147, 'centerline_recall': 0.8194412107101281, 'cldice': 0.879449629718959, 'foreground_fraction': 0.12147903442382812, 'thin_clean_centerline_recall': 0.7457375833951075}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '14', 'fold': 1, 'target': 'clean', 'dice': 0.7582921476622151, 'precision': 0.6358658248301864, 'recall': 0.9391019975251901, 'centerline_precision': 0.8295438694378574, 'centerline_recall': 0.9151695258678914, 'cldice': 0.8702555637191289, 'foreground_fraction': 0.12748336791992188, 'thin_clean_centerline_recall': 0.8583050847457627}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '14', 'fold': 1, 'target': 'proxy', 'dice': 0.8520459824887545, 'precision': 0.8560998234537239, 'recall': 0.8480303524320479, 'centerline_precision': 0.935276886595062, 'centerline_recall': 0.906438227814215, 'cldice': 0.920631770861851, 'foreground_fraction': 0.12748336791992188, 'thin_clean_centerline_recall': 0.8583050847457627}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '15', 'fold': 3, 'target': 'clean', 'dice': 0.730539328895988, 'precision': 0.5931829463350063, 'recall': 0.9506762132060461, 'centerline_precision': 0.8098020985522646, 'centerline_recall': 0.9374915069982335, 'cldice': 0.8689811344017461, 'foreground_fraction': 0.13832855224609375, 'thin_clean_centerline_recall': 0.8997464853652916}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '15', 'fold': 3, 'target': 'proxy', 'dice': 0.8293925688443502, 'precision': 0.7990182560255915, 'recall': 0.8621674700946259, 'centerline_precision': 0.9070261654934254, 'centerline_recall': 0.9094700026903417, 'cldice': 0.9082464401742277, 'foreground_fraction': 0.13832855224609375, 'thin_clean_centerline_recall': 0.8997464853652916}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '16', 'fold': 3, 'target': 'clean', 'dice': 0.7410123078078991, 'precision': 0.630747675455356, 'recall': 0.8979961918578293, 'centerline_precision': 0.7330119732545483, 'centerline_recall': 0.8073358569377366, 'cldice': 0.7683808007172094, 'foreground_fraction': 0.1197967529296875, 'thin_clean_centerline_recall': 0.6940980881130507}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '16', 'fold': 3, 'target': 'proxy', 'dice': 0.8233905217487337, 'precision': 0.8550503120621576, 'recall': 0.7939915432153524, 'centerline_precision': 0.9406002176955373, 'centerline_recall': 0.781257830117765, 'cldice': 0.853556175572484, 'foreground_fraction': 0.1197967529296875, 'thin_clean_centerline_recall': 0.6940980881130507}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '17', 'fold': 1, 'target': 'clean', 'dice': 0.7562419187416798, 'precision': 0.6645648625292065, 'recall': 0.8772604360317728, 'centerline_precision': 0.7708156929156083, 'centerline_recall': 0.8125, 'cldice': 0.7911091304105621, 'foreground_fraction': 0.11918258666992188, 'thin_clean_centerline_recall': 0.704042715484363}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '17', 'fold': 1, 'target': 'proxy', 'dice': 0.8245522776252886, 'precision': 0.8878788848702109, 'recall': 0.7696576216636146, 'centerline_precision': 0.952300310471352, 'centerline_recall': 0.7953195948305972, 'cldice': 0.8667595222317738, 'foreground_fraction': 0.11918258666992188, 'thin_clean_centerline_recall': 0.704042715484363}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '18', 'fold': 3, 'target': 'clean', 'dice': 0.7351017379153045, 'precision': 0.6002875021139862, 'recall': 0.9480080124638326, 'centerline_precision': 0.793980556665335, 'centerline_recall': 0.9381818181818182, 'cldice': 0.8600788622013872, 'foreground_fraction': 0.13533782958984375, 'thin_clean_centerline_recall': 0.8991634637124124}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '18', 'fold': 3, 'target': 'proxy', 'dice': 0.8350449851016359, 'precision': 0.8136027960989909, 'recall': 0.8576479676729261, 'centerline_precision': 0.9223598348648289, 'centerline_recall': 0.911702688455456, 'cldice': 0.9170002990380545, 'foreground_fraction': 0.13533782958984375, 'thin_clean_centerline_recall': 0.8991634637124124}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '19', 'fold': 3, 'target': 'clean', 'dice': 0.7835372805354877, 'precision': 0.7489556467724945, 'recall': 0.8214669907167617, 'centerline_precision': 0.827953354740578, 'centerline_recall': 0.7613666228646517, 'cldice': 0.7932651177494625, 'foreground_fraction': 0.10501480102539062, 'thin_clean_centerline_recall': 0.5621376354928157}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '19', 'fold': 3, 'target': 'proxy', 'dice': 0.7922753620927264, 'precision': 0.9209924080061026, 'recall': 0.695125294730493, 'centerline_precision': 0.9587628865979382, 'centerline_recall': 0.7437012405678476, 'cldice': 0.8376483672056393, 'foreground_fraction': 0.10501480102539062, 'thin_clean_centerline_recall': 0.5621376354928157}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '20', 'fold': 1, 'target': 'clean', 'dice': 0.781275635767396, 'precision': 0.6969872537659328, 'recall': 0.8887549576172331, 'centerline_precision': 0.8059178235871547, 'centerline_recall': 0.8410880458124552, 'cldice': 0.8231274216083343, 'foreground_fraction': 0.12509918212890625, 'thin_clean_centerline_recall': 0.735593220338983}, {'experiment': 'L1_proxy_d4_t0.5', 'id': '20', 'fold': 1, 'target': 'proxy', 'dice': 0.8402501901785701, 'precision': 0.9094041592974325, 'recall': 0.7808703393380813, 'centerline_precision': 0.964801570607208, 'centerline_recall': 0.8218075801749272, 'cldice': 0.8875822042471619, 'foreground_fraction': 0.12509918212890625, 'thin_clean_centerline_recall': 0.735593220338983}]
REFERENCE_GRID=[{'mode': 'none', 'target': 'clean', 'threshold': 0.05, 'mean_dice': 0.6321729825797686, 'worst_fold_dice': 0.5981584934456088, 'fold0': 0.6474833654285944, 'fold1': 0.6666502908210941, 'fold2': 0.5981584934456088, 'fold3': 0.638027456189582, 'fold4': 0.6105453070139635}, {'mode': 'none', 'target': 'clean', 'threshold': 0.075, 'mean_dice': 0.6508736774191982, 'worst_fold_dice': 0.6184783612738034, 'fold0': 0.6665145514909777, 'fold1': 0.6842944146038106, 'fold2': 0.6184783612738034, 'fold3': 0.6569578276083854, 'fold4': 0.6281232321190143}, {'mode': 'none', 'target': 'clean', 'threshold': 0.1, 'mean_dice': 0.6635837349383282, 'worst_fold_dice': 0.6329858629625551, 'fold0': 0.6789618501399513, 'fold1': 0.6964077284125235, 'fold2': 0.6329858629625551, 'fold3': 0.6695476648979181, 'fold4': 0.6400155682786929}, {'mode': 'none', 'target': 'clean', 'threshold': 0.125, 'mean_dice': 0.6732326052820808, 'worst_fold_dice': 0.644092268257052, 'fold0': 0.6887959291386355, 'fold1': 0.7054492538810551, 'fold2': 0.644092268257052, 'fold3': 0.678982587852783, 'fold4': 0.6488429872808791}, {'mode': 'none', 'target': 'clean', 'threshold': 0.15, 'mean_dice': 0.6808924330449825, 'worst_fold_dice': 0.6529999231175557, 'fold0': 0.6961918935517687, 'fold1': 0.7129567707252851, 'fold2': 0.6529999231175557, 'fold3': 0.6865685029810875, 'fold4': 0.6557450748492153}, {'mode': 'none', 'target': 'clean', 'threshold': 0.2, 'mean_dice': 0.6933189442140415, 'worst_fold_dice': 0.6667990995375738, 'fold0': 0.7084599667436332, 'fold1': 0.7249735222002271, 'fold2': 0.6673966015058915, 'fold3': 0.6989655310828821, 'fold4': 0.6667990995375738}, {'mode': 'none', 'target': 'clean', 'threshold': 0.3, 'mean_dice': 0.7114513857243174, 'worst_fold_dice': 0.6831623432963321, 'fold0': 0.7267512843301054, 'fold1': 0.7415231554175471, 'fold2': 0.6891941089160242, 'fold3': 0.7166260366615782, 'fold4': 0.6831623432963321}, {'mode': 'none', 'target': 'clean', 'threshold': 0.4, 'mean_dice': 0.7252237698074049, 'worst_fold_dice': 0.6957756825138413, 'fold0': 0.7403252579661473, 'fold1': 0.7543377983809767, 'fold2': 0.7060170692390425, 'fold3': 0.7296630409370171, 'fold4': 0.6957756825138413}, {'mode': 'none', 'target': 'clean', 'threshold': 0.5, 'mean_dice': 0.7369179521114223, 'worst_fold_dice': 0.7063342658827177, 'fold0': 0.7518700259604048, 'fold1': 0.7645251011866045, 'fold2': 0.7212355714521678, 'fold3': 0.7406247960752166, 'fold4': 0.7063342658827177}, {'mode': 'none', 'target': 'clean', 'threshold': 0.6, 'mean_dice': 0.7478253148691242, 'worst_fold_dice': 0.716173638379042, 'fold0': 0.7621512027715206, 'fold1': 0.7748046555112006, 'fold2': 0.7353167276651518, 'fold3': 0.7506803500187058, 'fold4': 0.716173638379042}, {'mode': 'none', 'target': 'clean', 'threshold': 0.7, 'mean_dice': 0.7584441105281211, 'worst_fold_dice': 0.7253796668464401, 'fold0': 0.7723807589561158, 'fold1': 0.7842353274001805, 'fold2': 0.7494412147826524, 'fold3': 0.7607835846552172, 'fold4': 0.7253796668464401}, {'mode': 'none', 'target': 'clean', 'threshold': 0.8, 'mean_dice': 0.7689323803208229, 'worst_fold_dice': 0.7338531739087638, 'fold0': 0.7822118596137362, 'fold1': 0.7927986512284931, 'fold2': 0.7648757827395964, 'fold3': 0.7709224341135252, 'fold4': 0.7338531739087638}, {'mode': 'none', 'target': 'clean', 'threshold': 0.9, 'mean_dice': 0.7775369651047085, 'worst_fold_dice': 0.7404772175003376, 'fold0': 0.7897058666435836, 'fold1': 0.7971417249204759, 'fold2': 0.7825901286685042, 'fold3': 0.7777698877906409, 'fold4': 0.7404772175003376}, {'mode': 'none', 'target': 'proxy', 'threshold': 0.05, 'mean_dice': 0.7747348878973148, 'worst_fold_dice': 0.7479214731891263, 'fold0': 0.7866507429611949, 'fold1': 0.8070922665032895, 'fold2': 0.7479214731891263, 'fold3': 0.7752492774570835, 'fold4': 0.7567606793758799}, {'mode': 'none', 'target': 'proxy', 'threshold': 0.075, 'mean_dice': 0.7879465324950752, 'worst_fold_dice': 0.7643249188552924, 'fold0': 0.8003416363802265, 'fold1': 0.8177552086875077, 'fold2': 0.7643249188552924, 'fold3': 0.7883662386693833, 'fold4': 0.7689446598829665}, {'mode': 'none', 'target': 'proxy', 'threshold': 0.1, 'mean_dice': 0.7959830032582017, 'worst_fold_dice': 0.7755273806587948, 'fold0': 0.8084618148198601, 'fold1': 0.8238749039477193, 'fold2': 0.7755273806587948, 'fold3': 0.7957025077610014, 'fold4': 0.7763484091036333}, {'mode': 'none', 'target': 'proxy', 'threshold': 0.125, 'mean_dice': 0.8014643376494284, 'worst_fold_dice': 0.7810821054343329, 'fold0': 0.8141029327806626, 'fold1': 0.827834976360113, 'fold2': 0.7832986248583963, 'fold3': 0.8010030488136368, 'fold4': 0.7810821054343329}, {'mode': 'none', 'target': 'proxy', 'threshold': 0.15, 'mean_dice': 0.8054103130170803, 'worst_fold_dice': 0.784359716082912, 'fold0': 0.8182584749274666, 'fold1': 0.8305578102545668, 'fold2': 0.7892525995186697, 'fold3': 0.8046229643017865, 'fold4': 0.784359716082912}, {'mode': 'none', 'target': 'proxy', 'threshold': 0.2, 'mean_dice': 0.8109067508949449, 'worst_fold_dice': 0.7888778662219853, 'fold0': 0.8238606250351933, 'fold1': 0.8343279963811232, 'fold2': 0.7977679396328445, 'fold3': 0.8096993272035784, 'fold4': 0.7888778662219853}, {'mode': 'none', 'target': 'proxy', 'threshold': 0.3, 'mean_dice': 0.8164916333267896, 'worst_fold_dice': 0.7934965411143514, 'fold0': 0.8299631285774683, 'fold1': 0.8369150917372761, 'fold2': 0.8080362053943893, 'fold3': 0.8140471998104627, 'fold4': 0.7934965411143514}, {'mode': 'none', 'target': 'proxy', 'threshold': 0.4, 'mean_dice': 0.8183856916864297, 'worst_fold_dice': 0.79457310195324, 'fold0': 0.831864265128692, 'fold1': 0.8362251654410114, 'fold2': 0.8138992327648159, 'fold3': 0.8153666931443887, 'fold4': 0.79457310195324}, {'mode': 'none', 'target': 'proxy', 'threshold': 0.5, 'mean_dice': 0.8177736727676009, 'worst_fold_dice': 0.7934176486236101, 'fold0': 0.831818304911305, 'fold1': 0.8326407183433817, 'fold2': 0.8170039001822559, 'fold3': 0.813987791777452, 'fold4': 0.7934176486236101}, {'mode': 'none', 'target': 'proxy', 'threshold': 0.6, 'mean_dice': 0.8150984343594407, 'worst_fold_dice': 0.7901764829685911, 'fold0': 0.828965921436552, 'fold1': 0.8274077634168701, 'fold2': 0.8183419848996152, 'fold3': 0.8106000190755754, 'fold4': 0.7901764829685911}, {'mode': 'none', 'target': 'proxy', 'threshold': 0.7, 'mean_dice': 0.8092989408906168, 'worst_fold_dice': 0.7833056408911971, 'fold0': 0.823369162494235, 'fold1': 0.8183705937662769, 'fold2': 0.816348121529794, 'fold3': 0.8051011857715813, 'fold4': 0.7833056408911971}, {'mode': 'none', 'target': 'proxy', 'threshold': 0.8, 'mean_dice': 0.7978895255853329, 'worst_fold_dice': 0.7705354322659288, 'fold0': 0.8113034761677612, 'fold1': 0.80366154557399, 'fold2': 0.81041080068344, 'fold3': 0.7935363732355445, 'fold4': 0.7705354322659288}, {'mode': 'none', 'target': 'proxy', 'threshold': 0.9, 'mean_dice': 0.769764230999747, 'worst_fold_dice': 0.7408992432767322, 'fold0': 0.7828952704099819, 'fold1': 0.7695617667597415, 'fold2': 0.7899646740151065, 'fold3': 0.7655002005371727, 'fold4': 0.7408992432767322}, {'mode': 'd4', 'target': 'clean', 'threshold': 0.05, 'mean_dice': 0.6122300612845737, 'worst_fold_dice': 0.5628202439334451, 'fold0': 0.6308804652483652, 'fold1': 0.6526473410354495, 'fold2': 0.5628202439334451, 'fold3': 0.6212018206484262, 'fold4': 0.5936004355571822}, {'mode': 'd4', 'target': 'clean', 'threshold': 0.075, 'mean_dice': 0.6349154096433565, 'worst_fold_dice': 0.5878228102425762, 'fold0': 0.653692953432499, 'fold1': 0.6742595868895314, 'fold2': 0.5878228102425762, 'fold3': 0.6439176567918524, 'fold4': 0.6148840408603232}, {'mode': 'd4', 'target': 'clean', 'threshold': 0.1, 'mean_dice': 0.6510643062323422, 'worst_fold_dice': 0.6065897638570811, 'fold0': 0.6697755665670537, 'fold1': 0.6897698919822546, 'fold2': 0.6065897638570811, 'fold3': 0.6596245958730936, 'fold4': 0.6295617128822285}, {'mode': 'd4', 'target': 'clean', 'threshold': 0.125, 'mean_dice': 0.6638232191307777, 'worst_fold_dice': 0.6218072287998485, 'fold0': 0.6829948941567432, 'fold1': 0.7016335941644264, 'fold2': 0.6218072287998485, 'fold3': 0.6717174181430738, 'fold4': 0.6409629603897973}, {'mode': 'd4', 'target': 'clean', 'threshold': 0.15, 'mean_dice': 0.6740992990902946, 'worst_fold_dice': 0.6341911458136615, 'fold0': 0.6934658044126099, 'fold1': 0.7110574632247239, 'fold2': 0.6341911458136615, 'fold3': 0.6816148045299517, 'fold4': 0.6501672774705263}, {'mode': 'd4', 'target': 'clean', 'threshold': 0.2, 'mean_dice': 0.6901107370070773, 'worst_fold_dice': 0.654121751603935, 'fold0': 0.709455865844417, 'fold1': 0.7259411773814465, 'fold2': 0.654121751603935, 'fold3': 0.6968841965679785, 'fold4': 0.6641506936376094}, {'mode': 'd4', 'target': 'clean', 'threshold': 0.3, 'mean_dice': 0.7133752093367931, 'worst_fold_dice': 0.6844576403260177, 'fold0': 0.7314291555809889, 'fold1': 0.7469647855570551, 'fold2': 0.6844576403260177, 'fold3': 0.7189446570540889, 'fold4': 0.6850798081658149}, {'mode': 'd4', 'target': 'clean', 'threshold': 0.4, 'mean_dice': 0.7303676432320723, 'worst_fold_dice': 0.6996384147358607, 'fold0': 0.7476907932676647, 'fold1': 0.7620054775385205, 'fold2': 0.7079294037469356, 'fold3': 0.7345741268713795, 'fold4': 0.6996384147358607}, {'mode': 'd4', 'target': 'clean', 'threshold': 0.5, 'mean_dice': 0.7445967904377048, 'worst_fold_dice': 0.7114289143488708, 'fold0': 0.7617496552171017, 'fold1': 0.7743441012542877, 'fold2': 0.7279136175795937, 'fold3': 0.7475476637886699, 'fold4': 0.7114289143488708}, {'mode': 'd4', 'target': 'clean', 'threshold': 0.6, 'mean_dice': 0.7565429769813432, 'worst_fold_dice': 0.7210706151566422, 'fold0': 0.7724851499721941, 'fold1': 0.7842841289300563, 'fold2': 0.745353715548841, 'fold3': 0.7595212752989831, 'fold4': 0.7210706151566422}, {'mode': 'd4', 'target': 'clean', 'threshold': 0.7, 'mean_dice': 0.7674106898418437, 'worst_fold_dice': 0.7302690545523955, 'fold0': 0.7822547742233972, 'fold1': 0.7930233597966874, 'fold2': 0.762137822874374, 'fold3': 0.7693684377623637, 'fold4': 0.7302690545523955}, {'mode': 'd4', 'target': 'clean', 'threshold': 0.8, 'mean_dice': 0.7769336538795931, 'worst_fold_dice': 0.7365561563281506, 'fold0': 0.7914170084771084, 'fold1': 0.7995271921137699, 'fold2': 0.7795528803389712, 'fold3': 0.7776150321399657, 'fold4': 0.7365561563281506}, {'mode': 'd4', 'target': 'clean', 'threshold': 0.9, 'mean_dice': 0.7814021859642829, 'worst_fold_dice': 0.7368265749851042, 'fold0': 0.7955670328110467, 'fold1': 0.7985102833145283, 'fold2': 0.7942818647859196, 'fold3': 0.7818251739248159, 'fold4': 0.7368265749851042}, {'mode': 'd4', 'target': 'proxy', 'threshold': 0.05, 'mean_dice': 0.761334791622301, 'worst_fold_dice': 0.716243355284791, 'fold0': 0.7781947403241258, 'fold1': 0.8002763614503716, 'fold2': 0.716243355284791, 'fold3': 0.7644722380204474, 'fold4': 0.7474872630317694}, {'mode': 'd4', 'target': 'proxy', 'threshold': 0.075, 'mean_dice': 0.7794668858303269, 'worst_fold_dice': 0.7393781814606665, 'fold0': 0.7965317019582051, 'fold1': 0.8156910639925842, 'fold2': 0.7393781814606665, 'fold3': 0.7814431649203494, 'fold4': 0.7642903168198292}, {'mode': 'd4', 'target': 'proxy', 'threshold': 0.1, 'mean_dice': 0.7914588469496885, 'worst_fold_dice': 0.7556511326175688, 'fold0': 0.8084384458067293, 'fold1': 0.825697821300595, 'fold2': 0.7556511326175688, 'fold3': 0.7927431867907947, 'fold4': 0.7747636482327546}, {'mode': 'd4', 'target': 'proxy', 'threshold': 0.125, 'mean_dice': 0.800092309331165, 'worst_fold_dice': 0.7681650338491697, 'fold0': 0.8174626461413446, 'fold1': 0.832486341052043, 'fold2': 0.7681650338491697, 'fold3': 0.8004875383309787, 'fold4': 0.7818599872822889}, {'mode': 'd4', 'target': 'proxy', 'threshold': 0.15, 'mean_dice': 0.8063137050738085, 'worst_fold_dice': 0.7777706630893344, 'fold0': 0.823685569736019, 'fold1': 0.8370732901773063, 'fold2': 0.7777706630893344, 'fold3': 0.8058682191374305, 'fold4': 0.7871707832289521}, {'mode': 'd4', 'target': 'proxy', 'threshold': 0.2, 'mean_dice': 0.8146688538501143, 'worst_fold_dice': 0.7920908313204729, 'fold0': 0.8319415406537403, 'fold1': 0.842884860805025, 'fold2': 0.7920908313204729, 'fold3': 0.8129308465384206, 'fold4': 0.7934961899329134}, {'mode': 'd4', 'target': 'proxy', 'threshold': 0.3, 'mean_dice': 0.8233929627308159, 'worst_fold_dice': 0.8002504748010679, 'fold0': 0.8398994088289714, 'fold1': 0.8463627566745673, 'fold2': 0.810210404201997, 'fold3': 0.8202417691474764, 'fold4': 0.8002504748010679}, {'mode': 'd4', 'target': 'proxy', 'threshold': 0.4, 'mean_dice': 0.82604389857659, 'worst_fold_dice': 0.8007914660157938, 'fold0': 0.8419885907606051, 'fold1': 0.8448145603256346, 'fold2': 0.8207519460968669, 'fold3': 0.8218729296840492, 'fold4': 0.8007914660157938}, {'mode': 'd4', 'target': 'proxy', 'threshold': 0.5, 'mean_dice': 0.8249899190473666, 'worst_fold_dice': 0.7979173097008371, 'fold0': 0.8406717681063003, 'fold1': 0.8402774983479648, 'fold2': 0.8260571596348689, 'fold3': 0.8200258594468617, 'fold4': 0.7979173097008371}, {'mode': 'd4', 'target': 'proxy', 'threshold': 0.6, 'mean_dice': 0.8201657222074245, 'worst_fold_dice': 0.7908604882252694, 'fold0': 0.8351573978774657, 'fold1': 0.8318626211848263, 'fold2': 0.8270960439491374, 'fold3': 0.8158520598004231, 'fold4': 0.7908604882252694}, {'mode': 'd4', 'target': 'proxy', 'threshold': 0.7, 'mean_dice': 0.8108474512689477, 'worst_fold_dice': 0.7797093701464145, 'fold0': 0.8251691896130662, 'fold1': 0.8189052309614484, 'fold2': 0.8240688286136932, 'fold3': 0.8063846370101166, 'fold4': 0.7797093701464145}, {'mode': 'd4', 'target': 'proxy', 'threshold': 0.8, 'mean_dice': 0.7946491281192465, 'worst_fold_dice': 0.7603436241143491, 'fold0': 0.8091203063870614, 'fold1': 0.7979953188033166, 'fold2': 0.8149940588202221, 'fold3': 0.7907923324712831, 'fold4': 0.7603436241143491}, {'mode': 'd4', 'target': 'proxy', 'threshold': 0.9, 'mean_dice': 0.7597093974489484, 'worst_fold_dice': 0.7208454585331843, 'fold0': 0.7751446355197285, 'fold1': 0.7572395412961692, 'fold2': 0.7884053408267907, 'fold3': 0.7569120110688696, 'fold4': 0.7208454585331843}]
REFERENCE_INFO={'config': {'seed': 42, 'folds': 5, 'fold': 0, 'epochs': 80, 'patch': 256, 'patches_per_epoch': 256, 'batch_size': 8, 'encoder_lr': 0.0001, 'decoder_lr': 0.0007, 'weight_decay': 0.0001, 'warmup_epochs': 3, 'ema_decay': 0.99, 'monitor_every': 10, 'threads': 4, 'amp': True, 'pretrained': True, 'tta_gain_required': 0.001, 'soft_boundary': 0.5}, 'data_fingerprint': '703ca1abcfc30fa5d5a5b7dbb8bdba6751aca8a873b6652d841d45abb47ec9a4', 'environment': {'python': '3.13.15', 'torch': '2.11.0+cu128', 'torchvision': '0.26.0+cu128', 'numpy': '2.1.3', 'opencv': '4.14.0', 'pillow': '13.0.0', 'skimage': '0.25.2', 'gpu': 'Tesla T4'}, 'origin': 'verified recovery L1 OOF; no public score input', 'reference_rows_sha256': 'e6a662d096d1173b60f10aac982b977571878f58cdf056645017e49d97eb8f22', 'reference_grid_sha256': '1993310fdaddfdc3e742713d04ff1dd224deee4d77585e89bca81bd567c30532'}
for name,source in SOURCES.items():
    (RUNTIME_DIR/name).write_text(source,encoding='utf-8',newline='\n')
def load_module(name):
    spec=importlib.util.spec_from_file_location(name,RUNTIME_DIR/(name+'.py'))
    module=importlib.util.module_from_spec(spec);sys.modules[name]=module
    spec.loader.exec_module(module);return module
BASE=load_module('task4_pipeline')
V2=load_module('task4_kaggle_v2')
ABL=load_module('task4_controlled_ablation')
COREBAND=load_module('task4_core_band')


## 3. 数据与GPU检查


In [ ]:
import torch
CFG=ABL.AblationConfig(**REFERENCE_INFO['config'])
ROOT=BASE.find_data(DATA_DIR)
TRAIN,TEST,LABELS,SPLIT,IDS,AUDIT=V2.audit_v2(ROOT,RUNTIME_DIR/'preflight_audit',CFG)
if AUDIT['data_fingerprint']!=REFERENCE_INFO['data_fingerprint']:raise ValueError('Dataset differs from frozen L1 reference')
for i,path in LABELS.items():
    c,p=BASE.strict_label(path),BASE.converter_label(path)
    if (c&~p).any():raise ValueError('Clean core outside proxy label: '+i)
print('One experiment, 5 folds x 80 epochs, fixed final EMA; no backbone/patch change')
print('Loss: 0.5 CE(3 classes) + 0.25 Dice(core,C) + 0.25 Dice(core+band,P)')
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Output:',OUTPUT_DIR)
print('Includes OOF analysis, all test predictions and ONE candidate CSV. No later inference cell needed.')
if RUN_EXPERIMENT and not torch.cuda.is_available():raise RuntimeError('Enable GPU before running the experiment')


## 4. 新方法五折训练、OOF与测试推理


In [ ]:
def show_small_package(output,paths):
    from IPython.display import display,FileLink
    display(FileLink(paths['analysis']))
RESULT_DIR=None
if RUN_EXPERIMENT:
    RESULT_DIR=COREBAND.run(ROOT,OUTPUT_DIR,CFG,REFERENCE_RECORDS,REFERENCE_ROWS,REFERENCE_GRID,REFERENCE_INFO,
        weights_path=WEIGHTS_PATH,folds_to_run=tuple(FOLDS_TO_RUN),resume_from=RESUME_FROM,on_package=show_small_package)
else:
    print('Preflight only: no training or inference started.')


## 5. 查看并下载小结果


In [ ]:
from IPython.display import display,FileLink
import json,pandas as pd
if RESULT_DIR is not None:
    STATUS=json.loads((RESULT_DIR/'core_band_status.json').read_text())
    print('Complete 20-image OOF:',STATUS['complete_20_image_oof'])
    if STATUS['complete_20_image_oof']:
        print('主对照：同一proxy输出分别评价C/P。core输出是另一个mask，不能合并成绩。')
        frame=pd.read_csv(RESULT_DIR/'core_band_summary.csv')
        display(frame[frame.experiment=='proxy_d4_t0.5'])
        display(pd.read_csv(RESULT_DIR/'candidate_oof_summary.csv'))
        print('OOF选择阈值与开发比较:',STATUS['selected_proxy_threshold'],STATUS['calibrated_proxy_comparison'])
        display(FileLink(str(RESULT_DIR/'submission_core_band_proxy.csv')))
    display(FileLink(str(RESULT_DIR/'core_band_analysis.zip')))
    print('只需下载CSV和小分析ZIP。权重留Kaggle；不要为了保存结果再Run All。')
